In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# MTG Deck Generator with Trained Model and RAG Integration (Complete, Corrected)
# Run this in Google Colab

!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 69.2 MB/s eta 0:00:00


In [ ]:
import os
import json
import logging
import requests
import torch
import faiss
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
import re

# Initialize logging
logging.basicConfig(level=logging.INFO)

# Globals
card_dict = {}
faiss_index = None
embed_model = None
card_texts = []

# Load MTG card data from Scryfall

def load_card_data():
    global card_dict
    logging.info("Loading card data from Scryfall...")
    url = "https://api.scryfall.com/bulk-data"
    bulk_data = requests.get(url).json()
    oracle_url = next(item for item in bulk_data["data"] if item["name"] == "Oracle Cards")["download_uri"]
    cards = requests.get(oracle_url).json()
    for card in cards:
        card_dict[card['name']] = {
            'color_identity': card.get('color_identity', []),
            'type_line': card.get('type_line', ''),
            'oracle_text': card.get('oracle_text', '')
        }
    logging.info(f"Loaded {len(card_dict)} cards")
    return cards

# Setup retriever

def setup_retriever(cards):
    global faiss_index, embed_model, card_texts
    embed_model = SentenceTransformer('all-MiniLM-L6-v2')
    card_texts = [f"{c['name']}. Types: {c['type_line']}. Text: {c.get('oracle_text', '')}" for c in cards]
    embeddings = embed_model.encode(card_texts, show_progress_bar=True)
    embeddings = np.array(embeddings).astype('float32')
    faiss_index = faiss.IndexFlatL2(embeddings.shape[1])
    faiss_index.add(embeddings)
    logging.info(f"Indexed {len(card_texts)} cards.")

# Verify card existence via local dict

def verify_card_exists(card_name):
    return card_name in card_dict

# Check banned cards (Commander)
COMMANDER_BANNED_CARDS = ["Black Lotus", "Time Walk", "Balance", "Channel", "Griselbrand"]

def is_card_banned(card_name):
    return card_name in COMMANDER_BANNED_CARDS

# Load fine-tuned model

def load_language_model(model_dir):
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForCausalLM.from_pretrained(model_dir)
    return tokenizer, model

# Prompt creation

def construct_prompt(commander, theme=None):
    context = retrieve_context(commander + ' ' + (theme or ''))
    prompt = f"""Generate a 100-card Commander deck.
Commander: {commander}
Theme: {theme if theme else 'Any'}
Relevant Cards:
{context}
Deck List:
"""
    return prompt

# Retrieve context

def retrieve_context(query, top_k=10):
    if not faiss_index or not embed_model:
        return ""
    query_vec = embed_model.encode([query])
    distances, indices = faiss_index.search(np.array(query_vec).astype('float32'), top_k)
    return '\n'.join([card_texts[i] for i in indices[0]])

# Generate deck

def generate_deck(commander_name, model, tokenizer):
    prompt = construct_prompt(commander_name)
    inputs = tokenizer.encode(prompt, return_tensors='pt')
    output = model.generate(inputs, max_length=1200, temperature=0.7, do_sample=True)
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Extract deck

def extract_deck(output_text):
    deck = {}
    matches = re.findall(r'(\d+)x ([^\n]+)', output_text)
    for count, name in matches:
        if verify_card_exists(name) and not is_card_banned(name):
            deck[name] = int(count)
    return deck

# Main execution

def main():
    cards = load_card_data()
    setup_retriever(cards)

    model_dir = '/content/drive/MyDrive/MTGModel/real_deck_training/final-model'
    tokenizer, model = load_language_model(model_dir)

    commander = input("Enter commander name: ")
    deck_text = generate_deck(commander_name=commander, tokenizer=tokenizer, model=model)
    deck = extract_deck(deck_text)

    print("Generated Deck:")
    for card, qty in deck.items():
        print(f"{qty}x {card}")

if __name__ == '__main__':
    main()

Batches:   0%|          | 0/1068 [00:00<?, ?it/s]

Enter commander name: Birgi, God of Storytelling


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Generated Deck:
1x Discontinuity
1x Tragic Arrogance
1x Doomed Necromancer
1x Dread Return
1x Evolving Wilds
1x Exotic Orchard
1x Fabled Passage
1x Fellwar Stone
1x Flaming Tyrannosaurus
1x Foreboding Landscape
1x Grave Endeavor
1x Gilded Lotus
1x Gloomlake Verge
1x Hallowed Fountain
1x Homunculus Horde
1x Imprisoned in the Moon
1x Inti, Seneschal of the Sun
3x Island
1x Jungle Hollow
1x Kogla, the Titan Ape
1x Kolaghan, the Storm's Fury
1x Lotus Bloom
1x Mox Amber
1x Nezahal, Primal Tide
1x Octopus
1x Ominous Seas
1x Ophiomancer
1x Otawara, Soaring City
1x Overlord of the Balemurk
1x Phyrexian Arena
1x Plains
1x Plaza of Heroes
1x Prairie Stream
1x Profane Tutor
1x Sol Ring
1x Sublime Epiphany
1x Talisman of Creativity
1x Teferi's Tutelage
1x Teferi, Mage of Zhalfir
1x Teferi, Time Raveler
1x Temple of Enlightenment
1x Thassa's Intervention
1x The Grey Havens
1x Thirst for Knowledge
1x Time Warp
1x Turnabout
1x Wrath of God


In [ ]:
import os
import json
import logging
import requests
import torch
import faiss
import numpy as np
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer

logging.basicConfig(level=logging.INFO)

card_dict = {}
faiss_index, embed_model, card_texts = None, None, []

COMMANDER_BANNED_CARDS = ["Ancestral Recall", "Balance", "Biorhythm", "Black Lotus", "Braids, Cabal Minion", "Channel", "Chaos Orb",
"Coalition Victory", "Dockside Extortionist", "Emrakul, the Aeons Torn", "Erayo, Soratami Ascendant", "Falling Star",
"Fastbond", "Flash", "Gifts Ungiven", "Golos, Tireless Pilgrim", "Griselbrand", "Hullbreacher", "Iona, Shield of Emeria",
"Jeweled Lotus", "Karakas", "Leovold, Emissary of Trest", "Library of Alexandria", "Limited Resources", "Lutri, the Spellchaser",
"Mana Crypt", "Mox Emerald", "Mox Jet", "Mox Pearl", "Mox Ruby", "Mox Sapphire", "Nadu, Winged Wisdom", "Panoptic Mirror",
"Paradox Engine", "Primeval Titan", "Prophet of Kruphix", "Recurring Nightmare", "Rofellos, Llanowar Emissary", "Shahrazad",
"Sundering Titan", "Sway of the Stars", "Sylvan Primordial", "Time Vault", "Time Walk", "Tinker", "Tolarian Academy",
"Trade Secrets", "Upheaval", "Yawgmoth’s Bargain"]

BASIC_LANDS = {
    'W': 'Plains',
    'U': 'Island',
    'B': 'Swamp',
    'R': 'Mountain',
    'G': 'Forest'
}

# Load cards including dual-faced

def load_card_data():
    global card_dict
    bulk_data = requests.get("https://api.scryfall.com/bulk-data").json()
    oracle_url = next(item for item in bulk_data["data"] if item["name"] == "Oracle Cards")["download_uri"]
    cards = requests.get(oracle_url).json()
    for card in cards:
        card_dict[card['name']] = {
            'color_identity': card.get('color_identity', []),
            'type_line': card.get('type_line', ''),
            'oracle_text': card.get('oracle_text', '')
        }
        if 'card_faces' in card:
            for face in card['card_faces']:
                card_dict[face['name']] = {
                    'color_identity': card.get('color_identity', []),
                    'type_line': face.get('type_line', ''),
                    'oracle_text': face.get('oracle_text', '')
                }
    return cards


def setup_retriever(cards):
    global faiss_index, embed_model, card_texts
    embed_model = SentenceTransformer('all-MiniLM-L6-v2')
    card_texts = [f"{c['name']}. Types: {c['type_line']}. Text: {c.get('oracle_text', '')}" for c in cards]
    embeddings = embed_model.encode(card_texts, show_progress_bar=True)
    embeddings = np.array(embeddings).astype('float32')
    faiss_index = faiss.IndexFlatL2(embeddings.shape[1])
    faiss_index.add(embeddings)


def load_language_model(model_dir):
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForCausalLM.from_pretrained(model_dir)
    return tokenizer, model


def get_commander_identity(commander_name):
    commander_info = card_dict.get(commander_name)
    return commander_info.get('color_identity', []) if commander_info else []


def validate_deck_identity(deck, commander_identity):
    valid_deck, invalid_cards = {}, []
    for card, qty in deck.items():
        card_info = card_dict.get(card)
        if card in BASIC_LANDS.values():
            if card in [BASIC_LANDS[color] for color in commander_identity]:
                valid_deck[card] = qty
            else:
                invalid_cards.append(card)
        elif card_info and all(c in commander_identity for c in card_info.get('color_identity', [])) and card not in COMMANDER_BANNED_CARDS:
            valid_deck[card] = qty
        else:
            invalid_cards.append(card)
    logging.info(f"Removed {len(invalid_cards)} invalid cards.")
    return valid_deck, invalid_cards


# Deck generation logic using fine-tuned model
def generate_deck(commander_name, tokenizer, model):
    prompt = f"Generate a synergistic Commander decklist for {commander_name}. Include card names and quantities."
    inputs = tokenizer(prompt, return_tensors='pt')
    outputs = model.generate(**inputs, max_length=1500, temperature=0.7, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def extract_deck(deck_text):
    deck = {}
    pattern = r"(\d+)x ([^\n]+)"
    matches = re.findall(pattern, deck_text)
    for qty, card in matches:
        card_name = card.strip()
        deck[card_name] = int(qty)
    return deck


def main():
    cards = load_card_data()
    setup_retriever(cards)

    model_dir = '/content/drive/MyDrive/MTGModel/real_deck_training/final-model'
    tokenizer, model = load_language_model(model_dir)

    commander = input("Enter commander name: ")
    deck_text = generate_deck(commander, tokenizer, model)
    deck = extract_deck(deck_text)

    commander_identity = get_commander_identity(commander)
    valid_deck, invalid_cards = validate_deck_identity(deck, commander_identity)

    print(f"\nValidated Deck for {commander} (Color Identity: {', '.join(commander_identity)}):")
    for card, qty in valid_deck.items():
        print(f"{qty}x {card}")


if __name__ == '__main__':
    main()


KeyboardInterrupt: 

In [ ]:
# MTG Deck Generator with Comprehensive Rules, Dual-Faced Card Support, and Validation
# Enhanced for Commander Deck Building Guidelines
# Run this in Google Colab

import os
import json
import logging
import requests
import torch
import faiss
import numpy as np
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
import random

logging.basicConfig(level=logging.INFO)

card_dict = {}
faiss_index, embed_model, card_texts, card_objects = None, None, [], []

COMMANDER_BANNED_CARDS = ["Ancestral Recall", "Balance", "Biorhythm", "Black Lotus", "Braids, Cabal Minion", "Channel", "Chaos Orb",
"Coalition Victory", "Dockside Extortionist", "Emrakul, the Aeons Torn", "Erayo, Soratami Ascendant", "Falling Star",
"Fastbond", "Flash", "Gifts Ungiven", "Golos, Tireless Pilgrim", "Griselbrand", "Hullbreacher", "Iona, Shield of Emeria",
"Jeweled Lotus", "Karakas", "Leovold, Emissary of Trest", "Library of Alexandria", "Limited Resources", "Lutri, the Spellchaser",
"Mana Crypt", "Mox Emerald", "Mox Jet", "Mox Pearl", "Mox Ruby", "Mox Sapphire", "Nadu, Winged Wisdom", "Panoptic Mirror",
"Paradox Engine", "Primeval Titan", "Prophet of Kruphix", "Recurring Nightmare", "Rofellos, Llanowar Emissary", "Shahrazad",
"Sundering Titan", "Sway of the Stars", "Sylvan Primordial", "Time Vault", "Time Walk", "Tinker", "Tolarian Academy",
"Trade Secrets", "Upheaval", "Yawgmoth's Bargain"]

BASIC_LANDS = {
    'W': 'Plains',
    'U': 'Island',
    'B': 'Swamp',
    'R': 'Mountain',
    'G': 'Forest'
}

CARD_CATEGORIES = {
    'ramp': ['type:artifact text:add mana', 'type:land text:add mana', 'type:creature text:"add mana"'],
    'draw': ['text:"draw a card"', 'text:"draw cards"'],
    'removal': ['text:destroy', 'text:exile', 'text:"damage to"'],
    'wrath': ['text:"destroy all"', 'text:"exile all"', 'text:"damage to all"'],
    'synergy': []  # Will be populated based on commander
}

def load_card_data():
    global card_dict, card_objects
    bulk_data = requests.get("https://api.scryfall.com/bulk-data").json()
    oracle_url = next(item for item in bulk_data["data"] if item["name"] == "Oracle Cards")["download_uri"]
    cards = requests.get(oracle_url).json()
    card_objects = []
    for card in cards:
        if 'name' not in card or 'legalities' not in card:
            continue
        if card.get('legalities', {}).get('commander') in ['legal', 'restricted']:
            card_objects.append(card)
            card_dict[card['name']] = {
                'color_identity': card.get('color_identity', []),
                'type_line': card.get('type_line', ''),
                'oracle_text': card.get('oracle_text', ''),
                'mana_value': card.get('cmc', 0),
                'types': extract_types(card.get('type_line', ''))
            }
            if 'card_faces' in card:
                for face in card['card_faces']:
                    if 'name' in face and face['name'] != card['name']:
                        card_dict[face['name']] = {
                            'color_identity': card.get('color_identity', []),
                            'type_line': face.get('type_line', ''),
                            'oracle_text': face.get('oracle_text', ''),
                            'mana_value': card.get('cmc', 0),
                            'types': extract_types(face.get('type_line', ''))
                        }
    logging.info(f"Loaded {len(card_objects)} legal Commander cards")
    return card_objects

def extract_types(type_line):
    types = []
    if "Land" in type_line:
        types.append("Land")
    if "Creature" in type_line:
        types.append("Creature")
    if "Artifact" in type_line:
        types.append("Artifact")
    if "Enchantment" in type_line:
        types.append("Enchantment")
    if "Planeswalker" in type_line:
        types.append("Planeswalker")
    if "Instant" in type_line:
        types.append("Instant")
    if "Sorcery" in type_line:
        types.append("Sorcery")
    return types

def setup_retriever(cards):
    global faiss_index, embed_model, card_texts
    embed_model = SentenceTransformer('all-MiniLM-L6-v2')

    # Create more detailed card texts for better semantic search
    card_texts = []
    for c in cards:
        if 'name' not in c or 'type_line' not in c:
            continue
        text = f"{c['name']}. Types: {c['type_line']}. "
        if 'oracle_text' in c and c['oracle_text']:
            text += f"Text: {c['oracle_text']}"
        if 'card_faces' in c:
            face_texts = []
            for face in c['card_faces']:
                if 'name' in face and 'type_line' in face:
                    face_text = f"{face['name']}. Types: {face['type_line']}. "
                    if 'oracle_text' in face and face['oracle_text']:
                        face_text += f"Text: {face['oracle_text']}"
                    face_texts.append(face_text)
            if face_texts:
                text += f" Faces: {' | '.join(face_texts)}"
        card_texts.append(text)

    logging.info(f"Generating embeddings for {len(card_texts)} cards...")
    embeddings = embed_model.encode(card_texts, show_progress_bar=True)
    embeddings = np.array(embeddings).astype('float32')
    faiss_index = faiss.IndexFlatL2(embeddings.shape[1])
    faiss_index.add(embeddings)
    logging.info("Embeddings completed and indexed")

def load_language_model(model_dir):
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForCausalLM.from_pretrained(model_dir)
    return tokenizer, model

def get_commander_identity(commander_name):
    commander_info = card_dict.get(commander_name)
    if not commander_info:
        logging.warning(f"Commander '{commander_name}' not found in database")
        # Try to find similar commander names
        similar_names = search_similar_cards(commander_name, n=5)
        logging.info(f"Did you mean one of these? {', '.join(similar_names)}")
        return []
    return commander_info.get('color_identity', [])

def validate_deck_identity(deck, commander_identity):
    valid_deck, invalid_cards = {}, []
    for card, qty in deck.items():
        # Skip validation for basic lands that match commander color identity
        if card in BASIC_LANDS.values():
            if not commander_identity or any(color in commander_identity for color, land in BASIC_LANDS.items() if land == card):
                valid_deck[card] = qty
                continue

        card_info = card_dict.get(card)
        if not card_info:
            invalid_cards.append(f"{card} (not found)")
            continue

        card_identity = card_info.get('color_identity', [])

        # Check if card's color identity is a subset of commander's identity
        if all(c in commander_identity for c in card_identity) and card not in COMMANDER_BANNED_CARDS:
            valid_deck[card] = qty
        else:
            reason = "banned" if card in COMMANDER_BANNED_CARDS else "color identity mismatch"
            invalid_cards.append(f"{card} ({reason})")

    logging.info(f"Removed {len(invalid_cards)} invalid cards: {', '.join(invalid_cards)}")
    return valid_deck, invalid_cards

def search_similar_cards(query, n=10):
    if not faiss_index or not card_objects:
        return []

    query_embedding = embed_model.encode([query])
    distances, indices = faiss_index.search(query_embedding, n)
    return [card_objects[i]['name'] for i in indices[0] if i < len(card_objects)]

def search_cards_by_criteria(query, commander_identity, exclude_cards=None, n=30):
    """Search for cards matching query with specified color identity"""
    if not exclude_cards:
        exclude_cards = set()
    else:
        exclude_cards = set(exclude_cards)

    query_embedding = embed_model.encode([query])
    distances, indices = faiss_index.search(query_embedding, n*3)  # Get more results to filter

    results = []
    for i in indices[0]:
        if i < len(card_objects):
            card = card_objects[i]
            if 'name' not in card or card['name'] in exclude_cards:
                continue

            # Check color identity
            card_identity = card.get('color_identity', [])
            if not all(c in commander_identity for c in card_identity):
                continue

            # Check legality
            if card.get('legalities', {}).get('commander') not in ['legal', 'restricted'] or card['name'] in COMMANDER_BANNED_CARDS:
                continue

            results.append(card['name'])
            if len(results) >= n:
                break

    return results

def get_commander_themes(commander_name):
    """Identify potential themes for the commander"""
    if not commander_name in card_dict:
        return []

    commander_info = card_dict[commander_name]
    oracle_text = commander_info.get('oracle_text', '')

    themes = []
    if 'draw' in oracle_text.lower() or 'card' in oracle_text.lower():
        themes.append("card draw")
    if 'damage' in oracle_text.lower():
        themes.append("damage")
    if 'counter' in oracle_text.lower():
        themes.append("counters")
    if 'token' in oracle_text.lower():
        themes.append("tokens")
    if 'graveyard' in oracle_text.lower() or 'cemetery' in oracle_text.lower():
        themes.append("graveyard")
    if 'sacrifice' in oracle_text.lower():
        themes.append("sacrifice")
    if 'discard' in oracle_text.lower():
        themes.append("discard")

    return themes

def generate_synergy_queries(commander_name):
    """Generate search queries based on commander themes"""
    themes = get_commander_themes(commander_name)
    synergy_queries = []

    # Add commander name for direct synergy
    synergy_queries.append(f"synergy with {commander_name}")

    # Add theme-based queries
    for theme in themes:
        synergy_queries.append(f"cards that work with {theme}")

    # Add general synergy queries based on commander text
    if commander_name in card_dict:
        commander_text = card_dict[commander_name].get('oracle_text', '')
        key_terms = re.findall(r'\b\w+\b', commander_text.lower())

        # Filter out common words
        stop_words = {'a', 'an', 'the', 'in', 'on', 'at', 'to', 'for', 'and', 'or', 'of', 'with', 'by'}
        key_terms = [term for term in key_terms if term not in stop_words and len(term) > 3]

        # Add key terms as synergy queries
        for term in key_terms[:3]:  # Use top 3 terms
            synergy_queries.append(f"cards with {term}")

    return synergy_queries

def generate_deck(commander_name, tokenizer, model):
    """Generate initial deck using language model"""
    prompt = f"Generate a synergistic Commander decklist for {commander_name}. Include card names and quantities, with 99 cards plus the commander."
    inputs = tokenizer(prompt, return_tensors='pt')
    outputs = model.generate(
        **inputs,
        max_length=2000,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def extract_deck(deck_text):
    """Extract card names and quantities from generated text"""
    deck = {}

    # Pattern for "NxCardName" format
    pattern1 = r"(\d+)x ([^\n,]+)"
    # Pattern for "N CardName" format
    pattern2 = r"(\d+) ([^\n,]+)"
    # Pattern for lines with card names only
    pattern3 = r"^([^0-9\n][^\n]+)$"

    # Find all matches for each pattern
    matches1 = re.findall(pattern1, deck_text)
    matches2 = re.findall(pattern2, deck_text)

    # Process matches from patterns with quantities
    for qty_str, card in matches1 + matches2:
        try:
            qty = int(qty_str)
            card_name = card.strip()
            if card_name and 0 < qty <= 99:  # Sanity check
                deck[card_name] = qty
        except ValueError:
            continue

    # Only use pattern3 if we didn't get enough cards
    if len(deck) < 20:
        lines = deck_text.split('\n')
        for line in lines:
            line = line.strip()
            if line and not any(c.isdigit() for c in line[:2]):  # Avoid lines starting with numbers
                deck[line] = 1

    return deck

def complete_deck(partial_deck, commander_name, commander_identity):
    """Complete deck with appropriate cards to reach 100 cards total"""
    logging.info(f"Completing deck with {len(partial_deck)} initial cards")

    # Create a copy of the partial deck
    completed_deck = dict(partial_deck)

    # Count cards by type
    card_count = sum(completed_deck.values())

    # Generate synergy search queries
    synergy_queries = generate_synergy_queries(commander_name)
    CARD_CATEGORIES['synergy'] = synergy_queries

    # Calculate how many cards to add
    cards_needed = 99 - card_count  # 99 cards plus commander = 100
    if cards_needed <= 0:
        return completed_deck

    logging.info(f"Need to add {cards_needed} more cards to reach 99")

    # Track all cards we're excluding
    exclude_list = list(completed_deck.keys())

    # Count card types in current deck
    type_counts = count_card_types(completed_deck)
    logging.info(f"Current type distribution: {type_counts}")

    # Add necessary basic lands if missing (most decks need them)
    added_lands = 0
    if type_counts.get('Land', 0) < 36:  # Aim for 36 lands in total
        lands_needed = min(36 - type_counts.get('Land', 0), cards_needed)
        logging.info(f"Adding {lands_needed} basic lands")

        # Determine which basic lands to add based on color identity
        land_types = []
        for color in commander_identity:
            if color in BASIC_LANDS:
                land_types.append(BASIC_LANDS[color])

        # Add equal numbers of each basic land type
        if land_types:
            per_land = lands_needed // len(land_types)
            remainder = lands_needed % len(land_types)

            for i, land in enumerate(land_types):
                qty = per_land + (1 if i < remainder else 0)
                if land in completed_deck:
                    completed_deck[land] += qty
                else:
                    completed_deck[land] = qty
                added_lands += qty

    # Recalculate cards needed after adding lands
    cards_needed -= added_lands

    # If we still need cards, add by category
    if cards_needed > 0:
        # Define target card type distribution (approximate)
        target_distribution = {
            'ramp': 0.10,      # 10% ramp spells
            'draw': 0.10,      # 10% card draw
            'removal': 0.10,   # 10% removal
            'wrath': 0.05,     # 5% board wipes
            'synergy': 0.65    # 65% synergy cards
        }

        # Add cards by category
        for category, percentage in target_distribution.items():
            category_count = int(cards_needed * percentage)
            if category_count > 0:
                logging.info(f"Adding {category_count} {category} cards")
                added = 0

                # Use different queries for the category
                queries = CARD_CATEGORIES[category]
                if not queries:
                    continue

                for query in queries:
                    if added >= category_count:
                        break

                    # Search for matching cards
                    results = search_cards_by_criteria(
                        query,
                        commander_identity,
                        exclude_cards=exclude_list
                    )

                    # Add a portion of results
                    to_add = min(len(results), category_count - added)
                    for i in range(to_add):
                        card_name = results[i]
                        completed_deck[card_name] = 1
                        exclude_list.append(card_name)
                        added += 1

    # Check final count
    total_cards = sum(completed_deck.values())
    if total_cards < 99:
        # Still need more cards - add random cards that match color identity
        logging.info(f"Still need {99 - total_cards} more cards")

        # Get cards that match color identity
        valid_cards = []
        for card_name, info in card_dict.items():
            if (card_name not in exclude_list and
                all(c in commander_identity for c in info.get('color_identity', [])) and
                card_name not in COMMANDER_BANNED_CARDS and
                not any(land == card_name for land in BASIC_LANDS.values())):
                valid_cards.append(card_name)

        # Randomly select remaining cards
        random.shuffle(valid_cards)
        for card_name in valid_cards:
            if sum(completed_deck.values()) >= 99:
                break
            completed_deck[card_name] = 1

    # If we have too many cards, trim some
    while sum(completed_deck.values()) > 99:
        # Find a card with qty > 1 to reduce
        for card, qty in list(completed_deck.items()):
            if qty > 1:
                completed_deck[card] -= 1
                break
        else:
            # If all cards have qty=1, remove a random card
            card_to_remove = random.choice(list(completed_deck.keys()))
            del completed_deck[card_to_remove]

    return completed_deck

def count_card_types(deck):
    """Count cards by type in the deck"""
    type_counts = {'Land': 0, 'Creature': 0, 'Artifact': 0,
                  'Enchantment': 0, 'Planeswalker': 0,
                  'Instant': 0, 'Sorcery': 0, 'Other': 0}

    for card, qty in deck.items():
        if card in BASIC_LANDS.values():
            type_counts['Land'] += qty
            continue

        card_info = card_dict.get(card)
        if not card_info:
            type_counts['Other'] += qty
            continue

        types = card_info.get('types', [])

        # Count by primary type (use first match in hierarchy)
        if 'Land' in types:
            type_counts['Land'] += qty
        elif 'Creature' in types:
            type_counts['Creature'] += qty
        elif 'Artifact' in types:
            type_counts['Artifact'] += qty
        elif 'Enchantment' in types:
            type_counts['Enchantment'] += qty
        elif 'Planeswalker' in types:
            type_counts['Planeswalker'] += qty
        elif 'Instant' in types:
            type_counts['Instant'] += qty
        elif 'Sorcery' in types:
            type_counts['Sorcery'] += qty
        else:
            type_counts['Other'] += qty

    return type_counts

def main():
    cards = load_card_data()
    setup_retriever(cards)

    model_dir = '/content/drive/MyDrive/MTGModel/real_deck_training/final-model'
    tokenizer, model = load_language_model(model_dir)

    commander = input("Enter commander name: ")
    commander_identity = get_commander_identity(commander)

    if not commander_identity:
        logging.error(f"Could not determine color identity for {commander}")
        similar_commanders = search_similar_cards(commander, n=5)
        print(f"Did you mean one of these? {', '.join(similar_commanders)}")
        commander = input("Try entering commander name again: ")
        commander_identity = get_commander_identity(commander)
        if not commander_identity:
            print("Still couldn't find commander. Using default color identity (colorless).")
            commander_identity = []

    print(f"\nGenerating deck for {commander} (Color Identity: {', '.join(commander_identity)})")

    # Generate initial deck with LLM
    deck_text = generate_deck(commander, tokenizer, model)
    initial_deck = extract_deck(deck_text)

    # Validate and filter cards
    valid_deck, invalid_cards = validate_deck_identity(initial_deck, commander_identity)

    # Complete deck to 100 cards
    final_deck = complete_deck(valid_deck, commander, commander_identity)

    # Add commander as 1x if not already in deck
    if commander not in final_deck:
        final_deck[commander] = 1

    # Print final deck with type categorization
    type_counts = count_card_types(final_deck)
    print(f"\nFinal Deck for {commander} (Color Identity: {', '.join(commander_identity)}):")
    print(f"Total cards: {sum(final_deck.values())}")
    print(f"Card type distribution: {type_counts}")
    print("\n--- Commander ---")
    print(f"1x {commander}")
    print("\n--- Lands ---")
    for card, qty in sorted(final_deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if 'Land' in types or card in BASIC_LANDS.values():
            print(f"{qty}x {card}")

    print("\n--- Creatures ---")
    for card, qty in sorted(final_deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if 'Creature' in types and 'Land' not in types:
            print(f"{qty}x {card}")

    print("\n--- Spells ---")
    for card, qty in sorted(final_deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if not ('Land' in types or 'Creature' in types or card in BASIC_LANDS.values()):
            print(f"{qty}x {card}")

if __name__ == '__main__':
    main()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/890 [00:00<?, ?it/s]

Enter commander name: Birgi, God of Storytelling

Generating deck for Birgi, God of Storytelling (Color Identity: R)

Final Deck for Birgi, God of Storytelling (Color Identity: R):
Total cards: 100
Card type distribution: {'Land': 39, 'Creature': 17, 'Artifact': 18, 'Enchantment': 2, 'Planeswalker': 0, 'Instant': 11, 'Sorcery': 13, 'Other': 0}

--- Commander ---
1x Birgi, God of Storytelling

--- Lands ---
1x Boseiju, Who Shelters All
1x Great Hall of the Citadel
34x Mountain
1x Myriad Landscape
1x Nykthos, Shrine to Nyx
1x Reliquary Tower

--- Creatures ---
1x Birgi, God of Storytelling // Harnfel, Horn of Bounty
1x Bonehoard Dracosaur
1x Clamor Shaman
1x Conspiracy Theorist
1x Dualcaster Mage
1x Fable of the Mirror-Breaker // Reflection of Kiki-Jiki
1x Goblin Elite Infantry
1x Guttersnipe
1x Harmonic Prodigy
1x Mannichi, the Fevered Dream
1x Neheb, the Eternal
1x Subira, Tulzidi Caravanner
1x Thermo-Alchemist
1x Three Tree Mascot
1x Twinscroll Shaman
1x Young Pyromancer

--- Spells -

In [ ]:
# MTG Deck Generator with Comprehensive Rules, Dual-Faced Card Support, and Validation
# Enhanced for Commander Deck Building Guidelines
# Run this in Google Colab

import os
import json
import logging
import requests
import torch
import faiss
import numpy as np
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
import random

logging.basicConfig(level=logging.INFO)

card_dict = {}
faiss_index, embed_model, card_texts, card_objects = None, None, [], []

COMMANDER_BANNED_CARDS = ["Ancestral Recall", "Balance", "Biorhythm", "Black Lotus", "Braids, Cabal Minion", "Channel", "Chaos Orb",
"Coalition Victory", "Dockside Extortionist", "Emrakul, the Aeons Torn", "Erayo, Soratami Ascendant", "Falling Star",
"Fastbond", "Flash", "Gifts Ungiven", "Golos, Tireless Pilgrim", "Griselbrand", "Hullbreacher", "Iona, Shield of Emeria",
"Jeweled Lotus", "Karakas", "Leovold, Emissary of Trest", "Library of Alexandria", "Limited Resources", "Lutri, the Spellchaser",
"Mana Crypt", "Mox Emerald", "Mox Jet", "Mox Pearl", "Mox Ruby", "Mox Sapphire", "Nadu, Winged Wisdom", "Panoptic Mirror",
"Paradox Engine", "Primeval Titan", "Prophet of Kruphix", "Recurring Nightmare", "Rofellos, Llanowar Emissary", "Shahrazad",
"Sundering Titan", "Sway of the Stars", "Sylvan Primordial", "Time Vault", "Time Walk", "Tinker", "Tolarian Academy",
"Trade Secrets", "Upheaval", "Yawgmoth's Bargain"]

BASIC_LANDS = {
    'W': 'Plains',
    'U': 'Island',
    'B': 'Swamp',
    'R': 'Mountain',
    'G': 'Forest'
}

CARD_CATEGORIES = {
    'ramp': ['type:artifact text:add mana', 'type:land text:add mana', 'type:creature text:"add mana"'],
    'draw': ['text:"draw a card"', 'text:"draw cards"'],
    'removal': ['text:destroy', 'text:exile', 'text:"damage to"'],
    'wrath': ['text:"destroy all"', 'text:"exile all"', 'text:"damage to all"'],
    'synergy': []  # Will be populated based on commander
}

def load_card_data():
    global card_dict, card_objects
    bulk_data = requests.get("https://api.scryfall.com/bulk-data").json()
    oracle_url = next(item for item in bulk_data["data"] if item["name"] == "Oracle Cards")["download_uri"]
    cards = requests.get(oracle_url).json()
    card_objects = []
    for card in cards:
        if 'name' not in card or 'legalities' not in card:
            continue
        if card.get('legalities', {}).get('commander') in ['legal', 'restricted']:
            card_objects.append(card)
            card_dict[card['name']] = {
                'color_identity': card.get('color_identity', []),
                'type_line': card.get('type_line', ''),
                'oracle_text': card.get('oracle_text', ''),
                'mana_value': card.get('cmc', 0),
                'types': extract_types(card.get('type_line', ''))
            }
            if 'card_faces' in card:
                for face in card['card_faces']:
                    if 'name' in face and face['name'] != card['name']:
                        card_dict[face['name']] = {
                            'color_identity': card.get('color_identity', []),
                            'type_line': face.get('type_line', ''),
                            'oracle_text': face.get('oracle_text', ''),
                            'mana_value': card.get('cmc', 0),
                            'types': extract_types(face.get('type_line', ''))
                        }
    logging.info(f"Loaded {len(card_objects)} legal Commander cards")
    return card_objects

def extract_types(type_line):
    types = []
    if "Land" in type_line:
        types.append("Land")
    if "Creature" in type_line:
        types.append("Creature")
    if "Artifact" in type_line:
        types.append("Artifact")
    if "Enchantment" in type_line:
        types.append("Enchantment")
    if "Planeswalker" in type_line:
        types.append("Planeswalker")
    if "Instant" in type_line:
        types.append("Instant")
    if "Sorcery" in type_line:
        types.append("Sorcery")
    return types

def setup_retriever(cards):
    global faiss_index, embed_model, card_texts
    embed_model = SentenceTransformer('all-MiniLM-L6-v2')

    # Create more detailed card texts for better semantic search
    card_texts = []
    for c in cards:
        if 'name' not in c or 'type_line' not in c:
            continue

        # Extract key card properties for embedding context
        card_name = c['name']
        type_line = c['type_line']
        oracle_text = c.get('oracle_text', '')
        keywords = c.get('keywords', [])
        mana_cost = c.get('mana_cost', '')

        # Create a rich text representation
        text = f"{card_name}. Cost: {mana_cost}. Types: {type_line}. "

        if keywords:
            text += f"Keywords: {', '.join(keywords)}. "

        if oracle_text:
            text += f"Text: {oracle_text}"

        # Add card faces for dual-faced cards
        if 'card_faces' in c:
            face_texts = []
            for face in c['card_faces']:
                if 'name' in face and 'type_line' in face:
                    face_name = face['name']
                    face_type = face['type_line']
                    face_text = face.get('oracle_text', '')
                    face_cost = face.get('mana_cost', '')

                    face_desc = f"{face_name}. Cost: {face_cost}. Types: {face_type}. "
                    if face_text:
                        face_desc += f"Text: {face_text}"
                    face_texts.append(face_desc)

            if face_texts:
                text += f" Faces: {' | '.join(face_texts)}"

        card_texts.append(text)

    logging.info(f"Generating embeddings for {len(card_texts)} cards...")

    # Create embeddings with batched processing for memory efficiency
    batch_size = 256
    all_embeddings = []

    for i in range(0, len(card_texts), batch_size):
        batch = card_texts[i:i + batch_size]
        batch_embeddings = embed_model.encode(batch, show_progress_bar=True)
        all_embeddings.append(batch_embeddings)

    embeddings = np.vstack(all_embeddings).astype('float32')

    # Create and populate the FAISS index
    faiss_index = faiss.IndexFlatL2(embeddings.shape[1])
    faiss_index.add(embeddings)
    logging.info("Embeddings completed and indexed")

def load_language_model(model_dir):
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForCausalLM.from_pretrained(model_dir)
    return tokenizer, model

def get_commander_identity(commander_name):
    commander_info = card_dict.get(commander_name)
    if not commander_info:
        logging.warning(f"Commander '{commander_name}' not found in database")
        # Try to find similar commander names
        similar_names = search_similar_cards(commander_name, n=5)
        logging.info(f"Did you mean one of these? {', '.join(similar_names)}")
        return []
    return commander_info.get('color_identity', [])

def validate_deck_identity(deck, commander_identity):
    valid_deck, invalid_cards = {}, []
    for card, qty in deck.items():
        # Skip validation for basic lands that match commander color identity
        if card in BASIC_LANDS.values():
            if not commander_identity or any(color in commander_identity for color, land in BASIC_LANDS.items() if land == card):
                valid_deck[card] = qty
                continue

        card_info = card_dict.get(card)
        if not card_info:
            invalid_cards.append(f"{card} (not found)")
            continue

        card_identity = card_info.get('color_identity', [])

        # Check if card's color identity is a subset of commander's identity
        if all(c in commander_identity for c in card_identity) and card not in COMMANDER_BANNED_CARDS:
            valid_deck[card] = qty
        else:
            reason = "banned" if card in COMMANDER_BANNED_CARDS else "color identity mismatch"
            invalid_cards.append(f"{card} ({reason})")

    logging.info(f"Removed {len(invalid_cards)} invalid cards: {', '.join(invalid_cards)}")
    return valid_deck, invalid_cards

def search_similar_cards(query, n=10):
    if not faiss_index or not card_objects:
        return []

    query_embedding = embed_model.encode([query])
    distances, indices = faiss_index.search(query_embedding, n)
    return [card_objects[i]['name'] for i in indices[0] if i < len(card_objects)]

def search_cards_by_criteria(query, commander_identity, exclude_cards=None, n=30):
    """Search for cards matching query with specified color identity"""
    if not exclude_cards:
        exclude_cards = set()
    else:
        exclude_cards = set(exclude_cards)

    query_embedding = embed_model.encode([query])
    distances, indices = faiss_index.search(query_embedding, n*3)  # Get more results to filter

    results = []
    for i in indices[0]:
        if i < len(card_objects):
            card = card_objects[i]
            if 'name' not in card or card['name'] in exclude_cards:
                continue

            # Check color identity
            card_identity = card.get('color_identity', [])
            if not all(c in commander_identity for c in card_identity):
                continue

            # Check legality
            if card.get('legalities', {}).get('commander') not in ['legal', 'restricted'] or card['name'] in COMMANDER_BANNED_CARDS:
                continue

            results.append(card['name'])
            if len(results) >= n:
                break

    return results

def get_commander_themes(commander_name):
    """Identify potential themes for the commander"""
    if not commander_name in card_dict:
        return []

    commander_info = card_dict[commander_name]
    oracle_text = commander_info.get('oracle_text', '')

    themes = []
    if 'draw' in oracle_text.lower() or 'card' in oracle_text.lower():
        themes.append("card draw")
    if 'damage' in oracle_text.lower():
        themes.append("damage")
    if 'counter' in oracle_text.lower():
        themes.append("counters")
    if 'token' in oracle_text.lower():
        themes.append("tokens")
    if 'graveyard' in oracle_text.lower() or 'cemetery' in oracle_text.lower():
        themes.append("graveyard")
    if 'sacrifice' in oracle_text.lower():
        themes.append("sacrifice")
    if 'discard' in oracle_text.lower():
        themes.append("discard")

    return themes

def generate_synergy_queries(commander_name):
    """Generate search queries based on commander themes"""
    themes = get_commander_themes(commander_name)
    synergy_queries = []

    # Add commander name for direct synergy
    synergy_queries.append(f"synergy with {commander_name}")

    # Add theme-based queries
    for theme in themes:
        synergy_queries.append(f"cards that work with {theme}")

    # Add general synergy queries based on commander text
    if commander_name in card_dict:
        commander_text = card_dict[commander_name].get('oracle_text', '')
        key_terms = re.findall(r'\b\w+\b', commander_text.lower())

        # Filter out common words
        stop_words = {'a', 'an', 'the', 'in', 'on', 'at', 'to', 'for', 'and', 'or', 'of', 'with', 'by'}
        key_terms = [term for term in key_terms if term not in stop_words and len(term) > 3]

        # Add key terms as synergy queries
        for term in key_terms[:3]:  # Use top 3 terms
            synergy_queries.append(f"cards with {term}")

    return synergy_queries

def analyze_mana_curve(curve):
    """Analyze the mana curve for potential issues"""
    issues = []

    # Calculate total spells
    total_spells = sum(curve.values())

    if total_spells < 10:
        return []  # Not enough spells to analyze

    # Check for imbalances in the curve
    if curve[1] + curve[2] < total_spells * 0.2:
        issues.append("low_early_drops")

    if curve['7+'] > total_spells * 0.15:
        issues.append("top_heavy")

    # Check for gaps in the curve
    for i in range(2, 5):
        if curve[i] == 0:
            issues.append(f"gap_at_{i}")

    return issues
def generate_deck(commander_name, tokenizer, model):
    """Generate initial deck using language model"""
    prompt = f"Generate a synergistic Commander decklist for {commander_name}. Include card names and quantities, with 99 cards plus the commander."
    inputs = tokenizer(prompt, return_tensors='pt')
    outputs = model.generate(
        **inputs,
        max_length=2000,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def extract_deck(deck_text):
    """Extract card names and quantities from generated text"""
    deck = {}

    # Pattern for "NxCardName" format
    pattern1 = r"(\d+)x ([^\n,]+)"
    # Pattern for "N CardName" format
    pattern2 = r"(\d+) ([^\n,]+)"
    # Pattern for lines with card names only
    pattern3 = r"^([^0-9\n][^\n]+)$"

    # Find all matches for each pattern
    matches1 = re.findall(pattern1, deck_text)
    matches2 = re.findall(pattern2, deck_text)

    # Process matches from patterns with quantities
    for qty_str, card in matches1 + matches2:
        try:
            qty = int(qty_str)
            card_name = card.strip()
            if card_name and 0 < qty <= 99:  # Sanity check
                deck[card_name] = qty
        except ValueError:
            continue

    # Only use pattern3 if we didn't get enough cards
    if len(deck) < 20:
        lines = deck_text.split('\n')
        for line in lines:
            line = line.strip()
            if line and not any(c.isdigit() for c in line[:2]):  # Avoid lines starting with numbers
                deck[line] = 1

    return deck

def complete_deck(partial_deck, commander_name, commander_identity, win_condition=None):
    """Complete deck with appropriate cards to reach 100 cards total"""
    logging.info(f"Completing deck with {len(partial_deck)} initial cards")

    # Create a copy of the partial deck
    completed_deck = dict(partial_deck)

    # Count cards by type
    card_count = sum(completed_deck.values())

    # Generate synergy search queries
    synergy_queries = generate_synergy_queries(commander_name)
    CARD_CATEGORIES['synergy'] = synergy_queries

    # If win condition is provided, add win condition-specific queries
    if win_condition:
        win_condition_name = win_condition.get('name', '')

        # Create targeted queries based on win condition
        win_queries = []

        # Basic win condition categorization
        if 'combo' in win_condition_name.lower():
            win_queries.extend([
                "infinite combo pieces",
                "combo enablers",
                "tutor effects"
            ])
        elif any(x in win_condition_name.lower() for x in ['damage', 'burn']):
            win_queries.extend([
                "direct damage spells",
                "damage doubler",
                "burn spells"
            ])
        elif 'token' in win_condition_name.lower():
            win_queries.extend([
                "token generators",
                "token doublers",
                "anthem effects"
            ])
        elif any(x in win_condition_name.lower() for x in ['mill', 'deck out']):
            win_queries.extend([
                "mill effects",
                "exile library cards"
            ])
        elif 'storm' in win_condition_name.lower():
            win_queries.extend([
                "storm cards",
                "cast multiple spells",
                "copy spells"
            ])

        # Add win condition name itself as a query
        win_queries.append(f"cards for {win_condition_name} strategy")

        # Add key cards from win condition as semantic anchors
        for key_card in win_condition.get('key_cards', []):
            if key_card in card_dict:
                win_queries.append(f"cards that work well with {key_card}")

        # Add these to our synergy queries
        CARD_CATEGORIES['win_condition'] = win_queries

    # Calculate how many cards to add
    cards_needed = 99 - card_count  # 99 cards plus commander = 100
    if cards_needed <= 0:
        return completed_deck

    logging.info(f"Need to add {cards_needed} more cards to reach 99")

    # Track all cards we're excluding
    exclude_list = list(completed_deck.keys())

    # Count card types in current deck
    type_counts = count_card_types(completed_deck)
    logging.info(f"Current type distribution: {type_counts}")

    # Calculate mana curve to analyze deck balance
    mana_curve = calculate_mana_curve(completed_deck)
    logging.info(f"Current mana curve: {mana_curve}")

    # Add necessary basic lands if missing (most decks need them)
    added_lands = 0
    if type_counts.get('Land', 0) < 36:  # Aim for 36 lands in total
        lands_needed = min(36 - type_counts.get('Land', 0), cards_needed)
        logging.info(f"Adding {lands_needed} basic lands")

        # Determine which basic lands to add based on color identity
        land_types = []
        for color in commander_identity:
            if color in BASIC_LANDS:
                land_types.append(BASIC_LANDS[color])

        # Add equal numbers of each basic land type
        if land_types:
            per_land = lands_needed // len(land_types)
            remainder = lands_needed % len(land_types)

            for i, land in enumerate(land_types):
                qty = per_land + (1 if i < remainder else 0)
                if land in completed_deck:
                    completed_deck[land] += qty
                else:
                    completed_deck[land] = qty
                added_lands += qty

    # Recalculate cards needed after adding lands
    cards_needed -= added_lands

    # If we still need cards, add by category
    if cards_needed > 0:
        # Define target card type distribution based on win condition
        if win_condition and 'name' in win_condition:
            win_condition_name = win_condition['name'].lower()

            # Adjust distribution based on win condition type
            if 'combo' in win_condition_name:
                target_distribution = {
                    'ramp': 0.15,      # More ramp for combo decks
                    'draw': 0.15,      # More card draw to find combo pieces
                    'removal': 0.10,   # Standard removal
                    'wrath': 0.05,     # Standard board wipes
                    'win_condition': 0.40, # Heavy focus on win condition
                    'synergy': 0.15    # Some general synergy
                }
            elif any(x in win_condition_name for x in ['aggro', 'damage', 'combat']):
                target_distribution = {
                    'ramp': 0.10,      # Standard ramp
                    'draw': 0.10,      # Standard draw
                    'removal': 0.15,   # More removal for combat-focused decks
                    'wrath': 0.05,     # Standard board wipes
                    'win_condition': 0.40, # Heavy focus on win condition
                    'synergy': 0.20    # More general synergy
                }
            elif 'control' in win_condition_name:
                target_distribution = {
                    'ramp': 0.10,      # Standard ramp
                    'draw': 0.15,      # More card draw for control
                    'removal': 0.20,   # More removal for control
                    'wrath': 0.10,     # More board wipes for control
                    'win_condition': 0.30, # Focus on win condition
                    'synergy': 0.15    # Some general synergy
                }
            else:
                # Default balanced distribution
                target_distribution = {
                    'ramp': 0.10,      # 10% ramp spells
                    'draw': 0.10,      # 10% card draw
                    'removal': 0.10,   # 10% removal
                    'wrath': 0.05,     # 5% board wipes
                    'win_condition': 0.35, # 35% win condition focus
                    'synergy': 0.30    # 30% general synergy
                }
        else:
            # Standard distribution without win condition
            target_distribution = {
                'ramp': 0.10,      # 10% ramp spells
                'draw': 0.10,      # 10% card draw
                'removal': 0.10,   # 10% removal
                'wrath': 0.05,     # 5% board wipes
                'synergy': 0.65    # 65% synergy cards
            }

        # Add cards by category
        for category, percentage in target_distribution.items():
            if category not in CARD_CATEGORIES:
                continue

            category_count = int(cards_needed * percentage)
            if category_count > 0:
                logging.info(f"Adding {category_count} {category} cards")
                added = 0

                # Use different queries for the category
                queries = CARD_CATEGORIES[category]
                if not queries:
                    continue

                for query in queries:
                    if added >= category_count:
                        break

                    # Search for matching cards
                    results = search_cards_by_criteria(
                        query,
                        commander_identity,
                        exclude_cards=exclude_list
                    )

                    # Add a portion of results
                    to_add = min(len(results), category_count - added)
                    for i in range(to_add):
                        card_name = results[i]
                        completed_deck[card_name] = 1
                        exclude_list.append(card_name)
                        added += 1

    # Analyze and balance mana curve if needed
    final_curve = calculate_mana_curve(completed_deck)
    curve_issues = analyze_mana_curve(final_curve)

    if curve_issues:
        logging.info(f"Mana curve issues detected: {curve_issues}")
        fix_mana_curve(completed_deck, final_curve, commander_identity, exclude_list)

    # Check final count
    total_cards = sum(completed_deck.values())
    if total_cards < 99:
        # Still need more cards - add random cards that match color identity
        logging.info(f"Still need {99 - total_cards} more cards")

        # Get cards that match color identity
        valid_cards = []
        for card_name, info in card_dict.items():
            if (card_name not in exclude_list and
                all(c in commander_identity for c in info.get('color_identity', [])) and
                card_name not in COMMANDER_BANNED_CARDS and
                not any(land == card_name for land in BASIC_LANDS.values())):
                valid_cards.append(card_name)

        # Randomly select remaining cards
        random.shuffle(valid_cards)
        for card_name in valid_cards:
            if sum(completed_deck.values()) >= 99:
                break
            completed_deck[card_name] = 1

    # If we have too many cards, trim some
    while sum(completed_deck.values()) > 99:
        # Find a card with qty > 1 to reduce
        for card, qty in list(completed_deck.items()):
            if qty > 1 and not any(land == card for land in BASIC_LANDS.values()):
                completed_deck[card] -= 1
                break
        else:
            # If all cards have qty=1, remove a random card
            non_essential = [card for card in completed_deck.keys()
                           if card != commander_name
                           and not any(land == card for land in BASIC_LANDS.values())
                           and (not win_condition or card not in win_condition.get('key_cards', []))]
            if non_essential:
                card_to_remove = random.choice(non_essential)
                del completed_deck[card_to_remove]

    return completed_deck

def calculate_mana_curve(deck):
    """Calculate the mana curve of the deck"""
    curve = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, '7+': 0}

    for card, qty in deck.items():
        card_info = card_dict.get(card)
        if not card_info:
            continue

        # Skip lands
        if 'Land' in card_info.get('types', []):
            continue

        mana_value = card_info.get('mana_value', 0)

        # Categorize by mana value
        if mana_value >= 7:
            curve['7+'] += qty
        else:
            curve[int(mana_value)] += qty

    return curve

def count_card_types(deck):
    """Count cards by type in the deck"""
    type_counts = {'Land': 0, 'Creature': 0, 'Artifact': 0,
                  'Enchantment': 0, 'Planeswalker': 0,
                  'Instant': 0, 'Sorcery': 0, 'Other': 0}

    for card, qty in deck.items():
        if card in BASIC_LANDS.values():
            type_counts['Land'] += qty
            continue

        card_info = card_dict.get(card)
        if not card_info:
            type_counts['Other'] += qty
            continue

        types = card_info.get('types', [])

        # Count by primary type (use first match in hierarchy)
        if 'Land' in types:
            type_counts['Land'] += qty
        elif 'Creature' in types:
            type_counts['Creature'] += qty
        elif 'Artifact' in types:
            type_counts['Artifact'] += qty
        elif 'Enchantment' in types:
            type_counts['Enchantment'] += qty
        elif 'Planeswalker' in types:
            type_counts['Planeswalker'] += qty
        elif 'Instant' in types:
            type_counts['Instant'] += qty
        elif 'Sorcery' in types:
            type_counts['Sorcery'] += qty
        else:
            type_counts['Other'] += qty

    return type_counts

def determine_win_condition(commander, tokenizer, model, commander_identity):
    """Use the language model to determine appropriate win conditions for the commander"""
    prompt = f"""Analyze {commander} for Commander format and suggest the top 3 win conditions that synergize with this commander.
For each win condition:
1. Name the strategy (e.g., "Combat damage", "Combo", "Mill", etc.)
2. Describe how it works with {commander}
3. List 5-10 key cards that enable this win condition within the {', '.join(commander_identity) if commander_identity else 'colorless'} color identity
4. Rate its power level from 1-10

Format your response clearly with headers."""

    inputs = tokenizer(prompt, return_tensors='pt')
    outputs = model.generate(
        **inputs,
        max_length=1500,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    win_condition_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Parse the win condition text to extract key cards
    win_conditions = []
    lines = win_condition_text.split('\n')
    current_condition = None
    key_cards = []

    for line in lines:
        line = line.strip()
        # Look for strategy names typically formatted as headers or numbered items
        if re.match(r'^(#+\s*|[0-9]+\.\s*)?(Win Condition|Strategy|Approach)', line, re.IGNORECASE):
            # Save previous condition if exists
            if current_condition and key_cards:
                win_conditions.append({
                    'name': current_condition,
                    'key_cards': key_cards
                })
            # Extract new condition name
            parts = re.split(r':|—|-', line, 1)
            if len(parts) > 1:
                current_condition = parts[1].strip()
            else:
                current_condition = line.split(' ', 1)[1].strip() if ' ' in line else line
            key_cards = []

        # Look for card names in lists, typically prefixed with bullets or numbers
        # Fixed regex pattern that was causing the error
        elif (line.startswith('-') or line.startswith('•') or line.startswith('*') or
              re.match(r'^\s*\d+\.', line) or re.match(r'^\s*[A-Z]', line)) and current_condition:
            # Clean up formatting around card names
            card = re.sub(r'^\s*[-•*0-9.)\]]*\s*', '', line)
            card = re.sub(r'[\[\]]', '', card)  # Remove [[]] wiki-style brackets

            # If there's explanation after the card name, trim it
            if ' - ' in card:
                card = card.split(' - ')[0].strip()
            elif ' – ' in card:
                card = card.split(' – ')[0].strip()
            elif ' — ' in card:
                card = card.split(' — ')[0].strip()
            elif ': ' in card:
                card = card.split(': ')[0].strip()

            # Validate the card exists in our database
            if card in card_dict:
                key_cards.append(card)

    # Add the last condition
    if current_condition and key_cards:
        win_conditions.append({
            'name': current_condition,
            'key_cards': key_cards
        })

    # If no valid win conditions were found, create a default one
    if not win_conditions:
        logging.warning("Could not parse win conditions from model output. Using default.")
        win_conditions = [{
            'name': 'Generic Synergy',
            'key_cards': []
        }]

    return win_conditions, win_condition_text

def generate_conditioned_deck(commander, tokenizer, model, win_condition, commander_identity):
    """Generate a deck with a specific win condition in mind"""
    # Prepare a prompt that specifies the win condition
    key_cards_str = ", ".join(win_condition['key_cards'][:5])  # Use top 5 key cards for prompt

    prompt = f"""Generate a Commander deck for {commander} focusing on the "{win_condition['name']}" win condition.
Include the following key cards if within color identity: {key_cards_str}.
Make sure the deck is cohesive and all cards work toward the win condition.
Format as a list with quantities (e.g., "1x Card Name").
Include exactly 99 cards (excluding the commander)."""

    inputs = tokenizer(prompt, return_tensors='pt')
    outputs = model.generate(
        **inputs,
        max_length=2000,
        temperature=0.65,  # Slightly lower temperature for more focused outputs
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    conditioned_deck_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return conditioned_deck_text

def main():
    cards = load_card_data()
    setup_retriever(cards)

    model_dir = '/content/drive/MyDrive/MTGModel/real_deck_training/final-model'
    tokenizer, model = load_language_model(model_dir)

    commander = input("Enter commander name: ")
    commander_identity = get_commander_identity(commander)

    if not commander_identity:
        logging.error(f"Could not determine color identity for {commander}")
        similar_commanders = search_similar_cards(commander, n=5)
        print(f"Did you mean one of these? {', '.join(similar_commanders)}")
        commander = input("Try entering commander name again: ")
        commander_identity = get_commander_identity(commander)
        if not commander_identity:
            print("Still couldn't find commander. Using default color identity (colorless).")
            commander_identity = []

    print(f"\nAnalyzing {commander} (Color Identity: {', '.join(commander_identity)})")

    # First, determine possible win conditions
    win_conditions, win_condition_text = determine_win_condition(commander, tokenizer, model, commander_identity)

    # Display win conditions to user
    print("\n=== Possible Win Conditions ===")
    print(win_condition_text)
    print("\n===========================")

    # Let user select a win condition
    if len(win_conditions) > 1:
        print("\nSelect a win condition to build around:")
        for i, wc in enumerate(win_conditions):
            print(f"{i+1}. {wc['name']} ({len(wc['key_cards'])} key cards identified)")

        selection = input(f"Enter 1-{len(win_conditions)} [1]: ").strip()
        if not selection:
            selected_idx = 0
        else:
            try:
                selected_idx = int(selection) - 1
                if selected_idx < 0 or selected_idx >= len(win_conditions):
                    selected_idx = 0
            except ValueError:
                selected_idx = 0
    else:
        selected_idx = 0

    selected_win_condition = win_conditions[selected_idx]
    print(f"\nGenerating deck for {commander} with '{selected_win_condition['name']}' win condition")

    # Generate deck with chosen win condition
    deck_text = generate_conditioned_deck(commander, tokenizer, model, selected_win_condition, commander_identity)
    initial_deck = extract_deck(deck_text)

    # Add key cards from win condition if not already in deck
    for card in selected_win_condition['key_cards']:
        if card not in initial_deck and card != commander:
            card_info = card_dict.get(card)
            if card_info and all(c in commander_identity for c in card_info.get('color_identity', [])):
                initial_deck[card] = 1

    # Validate and filter cards
    valid_deck, invalid_cards = validate_deck_identity(initial_deck, commander_identity)

    # Complete deck to 100 cards
    final_deck = complete_deck(valid_deck, commander, commander_identity)

    # Add commander as 1x if not already in deck
    if commander not in final_deck:
        final_deck[commander] = 1

    # Print final deck with type categorization
    type_counts = count_card_types(final_deck)
    print(f"\nFinal Deck for {commander} (Color Identity: {', '.join(commander_identity)}):")
    print(f"Total cards: {sum(final_deck.values())}")
    print(f"Card type distribution: {type_counts}")
    print("\n--- Commander ---")
    print(f"1x {commander}")
    print("\n--- Lands ---")
    for card, qty in sorted(final_deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if 'Land' in types or card in BASIC_LANDS.values():
            print(f"{qty}x {card}")

    print("\n--- Creatures ---")
    for card, qty in sorted(final_deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if 'Creature' in types and 'Land' not in types:
            print(f"{qty}x {card}")

    print("\n--- Spells ---")
    for card, qty in sorted(final_deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if not ('Land' in types or 'Creature' in types or card in BASIC_LANDS.values()):
            print(f"{qty}x {card}")

if __name__ == '__main__':
    main()

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Enter commander name: Birgi, God of Storytelling

Analyzing Birgi, God of Storytelling (Color Identity: R)



=== Possible Win Conditions ===
Analyze Birgi, God of Storytelling for Commander format and suggest the top 3 win conditions that synergize with this commander.
For each win condition:
1. Name the strategy (e.g., "Combat damage", "Combo", "Mill", etc.)
2. Describe how it works with Birgi, God of Storytelling
3. List 5-10 key cards that enable this win condition within the R color identity
4. Rate its power level from 1-10

Format your response clearly with headers.
1x Reiterate
1x Reliquary Tower
1x Reverse the Polarity
1x Tyvar's Stand
1x Unexpected Windfall
1x Valgavoth's Lair
1x Vexing Bauble
1x Explore
1x Faithless Looting
1x Fall of Gil-galad
1x Ferrous Lake
1x Fiery Islet
1x Fire Diamond
1x Flood of Recollection
1x Font of Mythos
1x Forced Fruition
1x Frostboil Snarl
1x Glimpse the Impossible
1x Goblin Engineer
1x Goblin Welder
1x Grapeshot
1x Grinning Ignus
1x Greater Sandwurm
1x Halimar Depths
1x Hall of Storm Giants
1x Howling Mine
1x Ian Malcolm, Chaotician
1x Ian the Reckle

NameError: name 'analyze_mana_curve' is not defined

In [ ]:
# MTG Deck Generator with Comprehensive Rules, Dual-Faced Card Support, and Validation
# Enhanced for Commander Deck Building Guidelines
# Run this in Google Colab

import os
import json
import logging
import requests
import torch
import faiss
import numpy as np
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
import random

logging.basicConfig(level=logging.INFO)

card_dict = {}
faiss_index, embed_model, card_texts, card_objects = None, None, [], []

COMMANDER_BANNED_CARDS = ["Ancestral Recall", "Balance", "Biorhythm", "Black Lotus", "Braids, Cabal Minion", "Channel", "Chaos Orb",
"Coalition Victory", "Dockside Extortionist", "Emrakul, the Aeons Torn", "Erayo, Soratami Ascendant", "Falling Star",
"Fastbond", "Flash", "Gifts Ungiven", "Golos, Tireless Pilgrim", "Griselbrand", "Hullbreacher", "Iona, Shield of Emeria",
"Jeweled Lotus", "Karakas", "Leovold, Emissary of Trest", "Library of Alexandria", "Limited Resources", "Lutri, the Spellchaser",
"Mana Crypt", "Mox Emerald", "Mox Jet", "Mox Pearl", "Mox Ruby", "Mox Sapphire", "Nadu, Winged Wisdom", "Panoptic Mirror",
"Paradox Engine", "Primeval Titan", "Prophet of Kruphix", "Recurring Nightmare", "Rofellos, Llanowar Emissary", "Shahrazad",
"Sundering Titan", "Sway of the Stars", "Sylvan Primordial", "Time Vault", "Time Walk", "Tinker", "Tolarian Academy",
"Trade Secrets", "Upheaval", "Yawgmoth's Bargain"]

BASIC_LANDS = {
    'W': 'Plains',
    'U': 'Island',
    'B': 'Swamp',
    'R': 'Mountain',
    'G': 'Forest'
}

CARD_CATEGORIES = {
    'ramp': ['type:artifact text:add mana', 'type:land text:add mana', 'type:creature text:"add mana"'],
    'draw': ['text:"draw a card"', 'text:"draw cards"'],
    'removal': ['text:destroy', 'text:exile', 'text:"damage to"'],
    'wrath': ['text:"destroy all"', 'text:"exile all"', 'text:"damage to all"'],
    'synergy': []  # Will be populated based on commander
}

def load_card_data():
    global card_dict, card_objects
    bulk_data = requests.get("https://api.scryfall.com/bulk-data").json()
    oracle_url = next(item for item in bulk_data["data"] if item["name"] == "Oracle Cards")["download_uri"]
    cards = requests.get(oracle_url).json()
    card_objects = []
    for card in cards:
        if 'name' not in card or 'legalities' not in card:
            continue
        if card.get('legalities', {}).get('commander') in ['legal', 'restricted']:
            card_objects.append(card)
            card_dict[card['name']] = {
                'color_identity': card.get('color_identity', []),
                'type_line': card.get('type_line', ''),
                'oracle_text': card.get('oracle_text', ''),
                'mana_value': card.get('cmc', 0),
                'types': extract_types(card.get('type_line', ''))
            }
            if 'card_faces' in card:
                for face in card['card_faces']:
                    if 'name' in face and face['name'] != card['name']:
                        card_dict[face['name']] = {
                            'color_identity': card.get('color_identity', []),
                            'type_line': face.get('type_line', ''),
                            'oracle_text': face.get('oracle_text', ''),
                            'mana_value': card.get('cmc', 0),
                            'types': extract_types(face.get('type_line', ''))
                        }
    logging.info(f"Loaded {len(card_objects)} legal Commander cards")
    return card_objects

def extract_types(type_line):
    types = []
    if "Land" in type_line:
        types.append("Land")
    if "Creature" in type_line:
        types.append("Creature")
    if "Artifact" in type_line:
        types.append("Artifact")
    if "Enchantment" in type_line:
        types.append("Enchantment")
    if "Planeswalker" in type_line:
        types.append("Planeswalker")
    if "Instant" in type_line:
        types.append("Instant")
    if "Sorcery" in type_line:
        types.append("Sorcery")
    return types

def setup_retriever(cards):
    global faiss_index, embed_model, card_texts
    embed_model = SentenceTransformer('all-MiniLM-L6-v2')

    # Create more detailed card texts for better semantic search
    card_texts = []
    for c in cards:
        if 'name' not in c or 'type_line' not in c:
            continue

        # Extract key card properties for embedding context
        card_name = c['name']
        type_line = c['type_line']
        oracle_text = c.get('oracle_text', '')
        keywords = c.get('keywords', [])
        mana_cost = c.get('mana_cost', '')

        # Create a rich text representation
        text = f"{card_name}. Cost: {mana_cost}. Types: {type_line}. "

        if keywords:
            text += f"Keywords: {', '.join(keywords)}. "

        if oracle_text:
            text += f"Text: {oracle_text}"

        # Add card faces for dual-faced cards
        if 'card_faces' in c:
            face_texts = []
            for face in c['card_faces']:
                if 'name' in face and 'type_line' in face:
                    face_name = face['name']
                    face_type = face['type_line']
                    face_text = face.get('oracle_text', '')
                    face_cost = face.get('mana_cost', '')

                    face_desc = f"{face_name}. Cost: {face_cost}. Types: {face_type}. "
                    if face_text:
                        face_desc += f"Text: {face_text}"
                    face_texts.append(face_desc)

            if face_texts:
                text += f" Faces: {' | '.join(face_texts)}"

        card_texts.append(text)

    logging.info(f"Generating embeddings for {len(card_texts)} cards...")

    # Create embeddings with batched processing for memory efficiency
    batch_size = 256
    all_embeddings = []

    for i in range(0, len(card_texts), batch_size):
        batch = card_texts[i:i + batch_size]
        batch_embeddings = embed_model.encode(batch, show_progress_bar=True)
        all_embeddings.append(batch_embeddings)

    embeddings = np.vstack(all_embeddings).astype('float32')

    # Create and populate the FAISS index
    faiss_index = faiss.IndexFlatL2(embeddings.shape[1])
    faiss_index.add(embeddings)
    logging.info("Embeddings completed and indexed")

def load_language_model(model_dir):
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForCausalLM.from_pretrained(model_dir)
    return tokenizer, model

def get_commander_identity(commander_name):
    commander_info = card_dict.get(commander_name)
    if not commander_info:
        logging.warning(f"Commander '{commander_name}' not found in database")
        # Try to find similar commander names
        similar_names = search_similar_cards(commander_name, n=5)
        logging.info(f"Did you mean one of these? {', '.join(similar_names)}")
        return []
    return commander_info.get('color_identity', [])

def validate_deck_identity(deck, commander_identity):
    valid_deck, invalid_cards = {}, []
    for card, qty in deck.items():
        # Skip validation for basic lands that match commander color identity
        if card in BASIC_LANDS.values():
            if not commander_identity or any(color in commander_identity for color, land in BASIC_LANDS.items() if land == card):
                valid_deck[card] = qty
                continue

        card_info = card_dict.get(card)
        if not card_info:
            invalid_cards.append(f"{card} (not found)")
            continue

        card_identity = card_info.get('color_identity', [])

        # Check if card's color identity is a subset of commander's identity
        if all(c in commander_identity for c in card_identity) and card not in COMMANDER_BANNED_CARDS:
            valid_deck[card] = qty
        else:
            reason = "banned" if card in COMMANDER_BANNED_CARDS else "color identity mismatch"
            invalid_cards.append(f"{card} ({reason})")

    logging.info(f"Removed {len(invalid_cards)} invalid cards: {', '.join(invalid_cards)}")
    return valid_deck, invalid_cards

def search_similar_cards(query, n=10):
    if not faiss_index or not card_objects:
        return []

    query_embedding = embed_model.encode([query])
    distances, indices = faiss_index.search(query_embedding, n)
    return [card_objects[i]['name'] for i in indices[0] if i < len(card_objects)]

def search_cards_by_criteria(query, commander_identity, exclude_cards=None, n=30):
    """Search for cards matching query with specified color identity"""
    if not exclude_cards:
        exclude_cards = set()
    else:
        exclude_cards = set(exclude_cards)

    query_embedding = embed_model.encode([query])
    distances, indices = faiss_index.search(query_embedding, n*3)  # Get more results to filter

    results = []
    for i in indices[0]:
        if i < len(card_objects):
            card = card_objects[i]
            if 'name' not in card or card['name'] in exclude_cards:
                continue

            # Check color identity
            card_identity = card.get('color_identity', [])
            if not all(c in commander_identity for c in card_identity):
                continue

            # Check legality
            if card.get('legalities', {}).get('commander') not in ['legal', 'restricted'] or card['name'] in COMMANDER_BANNED_CARDS:
                continue

            results.append(card['name'])
            if len(results) >= n:
                break

    return results

def get_commander_themes(commander_name):
    """Identify potential themes for the commander"""
    if not commander_name in card_dict:
        return []

    commander_info = card_dict[commander_name]
    oracle_text = commander_info.get('oracle_text', '')

    themes = []
    if 'draw' in oracle_text.lower() or 'card' in oracle_text.lower():
        themes.append("card draw")
    if 'damage' in oracle_text.lower():
        themes.append("damage")
    if 'counter' in oracle_text.lower():
        themes.append("counters")
    if 'token' in oracle_text.lower():
        themes.append("tokens")
    if 'graveyard' in oracle_text.lower() or 'cemetery' in oracle_text.lower():
        themes.append("graveyard")
    if 'sacrifice' in oracle_text.lower():
        themes.append("sacrifice")
    if 'discard' in oracle_text.lower():
        themes.append("discard")

    return themes

def generate_synergy_queries(commander_name):
    """Generate search queries based on commander themes"""
    themes = get_commander_themes(commander_name)
    synergy_queries = []

    # Add commander name for direct synergy
    synergy_queries.append(f"synergy with {commander_name}")

    # Add theme-based queries
    for theme in themes:
        synergy_queries.append(f"cards that work with {theme}")

    # Add general synergy queries based on commander text
    if commander_name in card_dict:
        commander_text = card_dict[commander_name].get('oracle_text', '')
        key_terms = re.findall(r'\b\w+\b', commander_text.lower())

        # Filter out common words
        stop_words = {'a', 'an', 'the', 'in', 'on', 'at', 'to', 'for', 'and', 'or', 'of', 'with', 'by'}
        key_terms = [term for term in key_terms if term not in stop_words and len(term) > 3]

        # Add key terms as synergy queries
        for term in key_terms[:3]:  # Use top 3 terms
            synergy_queries.append(f"cards with {term}")

    return synergy_queries

def generate_deck(commander_name, tokenizer, model):
    """Generate initial deck using language model"""
    prompt = f"Generate a synergistic Commander decklist for {commander_name}. Include card names and quantities, with 99 cards plus the commander."
    inputs = tokenizer(prompt, return_tensors='pt')
    outputs = model.generate(
        **inputs,
        max_length=2000,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def extract_deck(deck_text):
    """Extract card names and quantities from generated text"""
    deck = {}

    # Pattern for "NxCardName" format
    pattern1 = r"(\d+)x ([^\n,]+)"
    # Pattern for "N CardName" format
    pattern2 = r"(\d+) ([^\n,]+)"
    # Pattern for lines with card names only
    pattern3 = r"^([^0-9\n][^\n]+)$"

    # Find all matches for each pattern
    matches1 = re.findall(pattern1, deck_text)
    matches2 = re.findall(pattern2, deck_text)

    # Process matches from patterns with quantities
    for qty_str, card in matches1 + matches2:
        try:
            qty = int(qty_str)
            card_name = card.strip()
            if card_name and 0 < qty <= 99:  # Sanity check
                deck[card_name] = qty
        except ValueError:
            continue

    # Only use pattern3 if we didn't get enough cards
    if len(deck) < 20:
        lines = deck_text.split('\n')
        for line in lines:
            line = line.strip()
            if line and not any(c.isdigit() for c in line[:2]):  # Avoid lines starting with numbers
                deck[line] = 1

    return deck

def complete_deck(partial_deck, commander_name, commander_identity, win_condition=None):
    """Complete deck with appropriate cards to reach 100 cards total"""
    logging.info(f"Completing deck with {len(partial_deck)} initial cards")

    # Create a copy of the partial deck
    completed_deck = dict(partial_deck)

    # Count cards by type
    card_count = sum(completed_deck.values())

    # Generate synergy search queries
    synergy_queries = generate_synergy_queries(commander_name)
    CARD_CATEGORIES['synergy'] = synergy_queries

    # If win condition is provided, add win condition-specific queries
    if win_condition:
        win_condition_name = win_condition.get('name', '')

        # Create targeted queries based on win condition
        win_queries = []

        # Basic win condition categorization
        if 'combo' in win_condition_name.lower():
            win_queries.extend([
                "infinite combo pieces",
                "combo enablers",
                "tutor effects"
            ])
        elif any(x in win_condition_name.lower() for x in ['damage', 'burn']):
            win_queries.extend([
                "direct damage spells",
                "damage doubler",
                "burn spells"
            ])
        elif 'token' in win_condition_name.lower():
            win_queries.extend([
                "token generators",
                "token doublers",
                "anthem effects"
            ])
        elif any(x in win_condition_name.lower() for x in ['mill', 'deck out']):
            win_queries.extend([
                "mill effects",
                "exile library cards"
            ])
        elif 'storm' in win_condition_name.lower():
            win_queries.extend([
                "storm cards",
                "cast multiple spells",
                "copy spells"
            ])

        # Add win condition name itself as a query
        win_queries.append(f"cards for {win_condition_name} strategy")

        # Add key cards from win condition as semantic anchors
        for key_card in win_condition.get('key_cards', []):
            if key_card in card_dict:
                win_queries.append(f"cards that work well with {key_card}")

        # Add these to our synergy queries
        CARD_CATEGORIES['win_condition'] = win_queries

    # Calculate how many cards to add
    cards_needed = 99 - card_count  # 99 cards plus commander = 100
    if cards_needed <= 0:
        return completed_deck

    logging.info(f"Need to add {cards_needed} more cards to reach 99")

    # Track all cards we're excluding
    exclude_list = list(completed_deck.keys())

    # Count card types in current deck
    type_counts = count_card_types(completed_deck)
    logging.info(f"Current type distribution: {type_counts}")

    # Calculate mana curve to analyze deck balance
    mana_curve = calculate_mana_curve(completed_deck)
    logging.info(f"Current mana curve: {mana_curve}")

    # Add necessary basic lands if missing (most decks need them)
    added_lands = 0
    if type_counts.get('Land', 0) < 36:  # Aim for 36 lands in total
        lands_needed = min(36 - type_counts.get('Land', 0), cards_needed)
        logging.info(f"Adding {lands_needed} basic lands")

        # Determine which basic lands to add based on color identity
        land_types = []
        for color in commander_identity:
            if color in BASIC_LANDS:
                land_types.append(BASIC_LANDS[color])

        # Add equal numbers of each basic land type
        if land_types:
            per_land = lands_needed // len(land_types)
            remainder = lands_needed % len(land_types)

            for i, land in enumerate(land_types):
                qty = per_land + (1 if i < remainder else 0)
                if land in completed_deck:
                    completed_deck[land] += qty
                else:
                    completed_deck[land] = qty
                added_lands += qty

    # Recalculate cards needed after adding lands
    cards_needed -= added_lands

    # If we still need cards, add by category
    if cards_needed > 0:
        # Define target card type distribution based on win condition
        if win_condition and 'name' in win_condition:
            win_condition_name = win_condition['name'].lower()

            # Adjust distribution based on win condition type
            if 'combo' in win_condition_name:
                target_distribution = {
                    'ramp': 0.15,      # More ramp for combo decks
                    'draw': 0.15,      # More card draw to find combo pieces
                    'removal': 0.10,   # Standard removal
                    'wrath': 0.05,     # Standard board wipes
                    'win_condition': 0.40, # Heavy focus on win condition
                    'synergy': 0.15    # Some general synergy
                }
            elif any(x in win_condition_name for x in ['aggro', 'damage', 'combat']):
                target_distribution = {
                    'ramp': 0.10,      # Standard ramp
                    'draw': 0.10,      # Standard draw
                    'removal': 0.15,   # More removal for combat-focused decks
                    'wrath': 0.05,     # Standard board wipes
                    'win_condition': 0.40, # Heavy focus on win condition
                    'synergy': 0.20    # More general synergy
                }
            elif 'control' in win_condition_name:
                target_distribution = {
                    'ramp': 0.10,      # Standard ramp
                    'draw': 0.15,      # More card draw for control
                    'removal': 0.20,   # More removal for control
                    'wrath': 0.10,     # More board wipes for control
                    'win_condition': 0.30, # Focus on win condition
                    'synergy': 0.15    # Some general synergy
                }
            else:
                # Default balanced distribution
                target_distribution = {
                    'ramp': 0.10,      # 10% ramp spells
                    'draw': 0.10,      # 10% card draw
                    'removal': 0.10,   # 10% removal
                    'wrath': 0.05,     # 5% board wipes
                    'win_condition': 0.35, # 35% win condition focus
                    'synergy': 0.30    # 30% general synergy
                }
        else:
            # Standard distribution without win condition
            target_distribution = {
                'ramp': 0.10,      # 10% ramp spells
                'draw': 0.10,      # 10% card draw
                'removal': 0.10,   # 10% removal
                'wrath': 0.05,     # 5% board wipes
                'synergy': 0.65    # 65% synergy cards
            }

        # Add cards by category
        for category, percentage in target_distribution.items():
            if category not in CARD_CATEGORIES:
                continue

            category_count = int(cards_needed * percentage)
            if category_count > 0:
                logging.info(f"Adding {category_count} {category} cards")
                added = 0

                # Use different queries for the category
                queries = CARD_CATEGORIES[category]
                if not queries:
                    continue

                for query in queries:
                    if added >= category_count:
                        break

                    # Search for matching cards
                    results = search_cards_by_criteria(
                        query,
                        commander_identity,
                        exclude_cards=exclude_list
                    )

                    # Add a portion of results
                    to_add = min(len(results), category_count - added)
                    for i in range(to_add):
                        card_name = results[i]
                        completed_deck[card_name] = 1
                        exclude_list.append(card_name)
                        added += 1

    # Analyze and balance mana curve if needed
    final_curve = calculate_mana_curve(completed_deck)
    curve_issues = analyze_mana_curve(final_curve)

    if curve_issues:
        logging.info(f"Mana curve issues detected: {curve_issues}")
        fix_mana_curve(completed_deck, final_curve, commander_identity, exclude_list)

    # Check final count
    total_cards = sum(completed_deck.values())
    if total_cards < 99:
        # Still need more cards - add random cards that match color identity
        logging.info(f"Still need {99 - total_cards} more cards")

        # Get cards that match color identity
        valid_cards = []
        for card_name, info in card_dict.items():
            if (card_name not in exclude_list and
                all(c in commander_identity for c in info.get('color_identity', [])) and
                card_name not in COMMANDER_BANNED_CARDS and
                not any(land == card_name for land in BASIC_LANDS.values())):
                valid_cards.append(card_name)

        # Randomly select remaining cards
        random.shuffle(valid_cards)
        for card_name in valid_cards:
            if sum(completed_deck.values()) >= 99:
                break
            completed_deck[card_name] = 1

    # If we have too many cards, trim some
    while sum(completed_deck.values()) > 99:
        # Find a card with qty > 1 to reduce
        for card, qty in list(completed_deck.items()):
            if qty > 1 and not any(land == card for land in BASIC_LANDS.values()):
                completed_deck[card] -= 1
                break
        else:
            # If all cards have qty=1, remove a random card
            non_essential = [card for card in completed_deck.keys()
                           if card != commander_name
                           and not any(land == card for land in BASIC_LANDS.values())
                           and (not win_condition or card not in win_condition.get('key_cards', []))]
            if non_essential:
                card_to_remove = random.choice(non_essential)
                del completed_deck[card_to_remove]

    return completed_deck

def calculate_mana_curve(deck):
    """Calculate the mana curve of the deck"""
    curve = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, '7+': 0}

    for card, qty in deck.items():
        card_info = card_dict.get(card)
        if not card_info:
            continue

        # Skip lands
        if 'Land' in card_info.get('types', []):
            continue

        mana_value = card_info.get('mana_value', 0)

        # Categorize by mana value
        if mana_value >= 7:
            curve['7+'] += qty
        else:
            curve[int(mana_value)] += qty

    return curve

def analyze_mana_curve(curve):
    """Analyze the mana curve for potential issues"""
    issues = []

    # Calculate total spells
    total_spells = sum(curve.values())

    if total_spells < 10:
        return []  # Not enough spells to analyze

    # Check for imbalances in the curve
    if curve[1] + curve[2] < total_spells * 0.2:
        issues.append("low_early_drops")

    if curve['7+'] > total_spells * 0.15:
        issues.append("top_heavy")

    # Check for gaps in the curve
    for i in range(2, 5):
        if curve[i] == 0:
            issues.append(f"gap_at_{i}")

    return issues

def fix_mana_curve(deck, curve, commander_identity, exclude_list):
    """Attempt to fix issues with the mana curve"""
    issues = analyze_mana_curve(curve)

    if not issues:
        return deck

    # Fix common issues
    if "low_early_drops" in issues:
        # Add more low-cost cards
        logging.info("Fixing mana curve: Adding more low-drop cards")

        # Replace some high-cost cards with low-cost ones
        high_cost_cards = []
        for card, qty in deck.items():
            card_info = card_dict.get(card)
            if not card_info:
                continue

            if 'Land' in card_info.get('types', []):
                continue

            mana_value = card_info.get('mana_value', 0)
            if mana_value >= 5:
                high_cost_cards.extend([card] * qty)

        # Shuffle to randomize selections
        random.shuffle(high_cost_cards)

        # Replace up to 5 high-cost cards
        replacements = min(5, len(high_cost_cards))

        for i in range(replacements):
            if i < len(high_cost_cards):
                card_to_replace = high_cost_cards[i]

                # Find low-cost replacements
                low_cost_cards = search_cards_by_criteria(
                    "mana value 1 or mana value 2",
                    commander_identity,
                    exclude_cards=exclude_list
                )

                if low_cost_cards:
                    # Remove high-cost card
                    if deck[card_to_replace] > 1:
                        deck[card_to_replace] -= 1
                    else:
                        del deck[card_to_replace]

                    # Add low-cost card
                    replacement = low_cost_cards[0]
                    if replacement in deck:
                        deck[replacement] += 1
                    else:
                        deck[replacement] = 1
                    exclude_list.append(replacement)

    if "top_heavy" in issues:
        # Replace some high-cost cards with mid-range ones
        logging.info("Fixing mana curve: Reducing top-heavy cards")

        high_cost_cards = []
        for card, qty in deck.items():
            card_info = card_dict.get(card)
            if not card_info:
                continue

            if 'Land' in card_info.get('types', []):
                continue

            mana_value = card_info.get('mana_value', 0)
            if mana_value >= 7:
                high_cost_cards.extend([card] * qty)

        # Replace up to half of the high-cost cards
        replacements = min(len(high_cost_cards) // 2 + 1, len(high_cost_cards))

        for i in range(replacements):
            if i < len(high_cost_cards):
                card_to_replace = high_cost_cards[i]

                # Find mid-range replacements
                mid_cost_cards = search_cards_by_criteria(
                    "mana value 3 or mana value 4 or mana value 5",
                    commander_identity,
                    exclude_cards=exclude_list
                )

                if mid_cost_cards:
                    # Remove high-cost card
                    if deck[card_to_replace] > 1:
                        deck[card_to_replace] -= 1
                    else:
                        del deck[card_to_replace]

                    # Add mid-cost card
                    replacement = mid_cost_cards[0]
                    if replacement in deck:
                        deck[replacement] += 1
                    else:
                        deck[replacement] = 1
                    exclude_list.append(replacement)

    # Fix gaps in the curve
    for issue in issues:
        if issue.startswith("gap_at_"):
            mana_value = int(issue.split("_")[-1])
            logging.info(f"Fixing mana curve: Adding cards at mana value {mana_value}")

            # Find cards with the missing mana value
            gap_cards = search_cards_by_criteria(
                f"mana value {mana_value}",
                commander_identity,
                exclude_cards=exclude_list
            )

            if gap_cards:
                # Find a card to replace
                replaced = False
                for card, qty in list(deck.items()):
                    card_info = card_dict.get(card)
                    if not card_info:
                        continue

                    if 'Land' in card_info.get('types', []):
                        continue

                    existing_mv = card_info.get('mana_value', 0)
                    if (existing_mv >= 6 or existing_mv <= 1) and card not in exclude_list:
                        # Replace this card
                        if qty > 1:
                            deck[card] -= 1
                        else:
                            del deck[card]

                        # Add gap-filling card
                        replacement = gap_cards[0]
                        if replacement in deck:
                            deck[replacement] += 1
                        else:
                            deck[replacement] = 1
                        exclude_list.append(replacement)
                        replaced = True
                        break

                # If we couldn't find a card to replace, just add the gap card
                if not replaced and sum(deck.values()) < 99:
                    replacement = gap_cards[0]
                    if replacement in deck:
                        deck[replacement] += 1
                    else:
                        deck[replacement] = 1
                    exclude_list.append(replacement)

    return deck

def count_card_types(deck):
    """Count cards by type in the deck"""
    type_counts = {'Land': 0, 'Creature': 0, 'Artifact': 0,
                  'Enchantment': 0, 'Planeswalker': 0,
                  'Instant': 0, 'Sorcery': 0, 'Other': 0}

    for card, qty in deck.items():
        if card in BASIC_LANDS.values():
            type_counts['Land'] += qty
            continue

        card_info = card_dict.get(card)
        if not card_info:
            type_counts['Other'] += qty
            continue

        types = card_info.get('types', [])

        # Count by primary type (use first match in hierarchy)
        if 'Land' in types:
            type_counts['Land'] += qty
        elif 'Creature' in types:
            type_counts['Creature'] += qty
        elif 'Artifact' in types:
            type_counts['Artifact'] += qty
        elif 'Enchantment' in types:
            type_counts['Enchantment'] += qty
        elif 'Planeswalker' in types:
            type_counts['Planeswalker'] += qty
        elif 'Instant' in types:
            type_counts['Instant'] += qty
        elif 'Sorcery' in types:
            type_counts['Sorcery'] += qty
        else:
            type_counts['Other'] += qty

    return type_counts

def determine_win_condition(commander, tokenizer, model, commander_identity):
    """Use the language model to determine appropriate win conditions for the commander"""
    prompt = f"""Analyze {commander} for Commander format and suggest the top 3 win conditions that synergize with this commander.
For each win condition:
1. Name the strategy (e.g., "Storm", "Combo", "Aggro", etc.)
2. Describe how it works with {commander}
3. List 5-10 key cards that enable this win condition within the {', '.join(commander_identity) if commander_identity else 'colorless'} color identity
4. Rate its power level from 1-10

Format your response with clear section headers for each win condition and bullet points for key cards.
"""

    inputs = tokenizer(prompt, return_tensors='pt')
    outputs = model.generate(
        **inputs,
        max_length=1500,
        temperature=0.8,  # Slightly higher temperature for more diverse responses
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    win_condition_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Define some pre-set win conditions if we can't parse the model output
    preset_win_conditions = [
        {
            'name': 'Storm Combo',
            'description': 'Cast multiple cheap spells in a turn to generate mana with Birgi, then finish with a big payoff spell.',
            'key_cards': ['Grapeshot', 'Mana Geyser', 'Seething Song', 'Jeska\'s Will', 'Runaway Steam-Kin', 'Grinning Ignus'],
            'power_level': 8
        },
        {
            'name': 'Wheel Effects',
            'description': 'Use Harnfel\'s "discard to draw" ability with wheel effects to cycle through your deck quickly.',
            'key_cards': ['Wheel of Fortune', 'Reforge the Soul', 'Magus of the Wheel', 'Past in Flames', 'Underworld Breach'],
            'power_level': 7
        },
        {
            'name': 'Artifact Combo',
            'description': 'Use cost reducers and Birgi\'s mana generation to chain artifact casts together.',
            'key_cards': ['Grinning Ignus', 'Sensei\'s Divining Top', 'Aetherflux Reservoir', 'Helm of Awakening', 'Skullclamp'],
            'power_level': 7
        },
        {
            'name': 'Red Aggro',
            'description': 'Cast numerous cheap red creatures and use Birgi\'s mana to overwhelm opponents.',
            'key_cards': ['Runaway Steam-Kin', 'Monastery Swiftspear', 'Lightning Bolt', 'Light Up the Stage', 'Reckless Impulse'],
            'power_level': 6
        }
    ]

    # Attempt to parse the model's output
    try:
        # Parse the win condition text to extract structured data
        win_conditions = []
        sections = re.split(r'(?:\n\n|\n#+\s*|\n\d+\.?\s*)(Win Condition|Strategy|Approach)[:\s-]+', win_condition_text)

        # If we have sections after splitting
        if len(sections) > 1:
            for i in range(1, len(sections), 2):
                if i+1 < len(sections):
                    section_header = sections[i].strip()
                    section_content = sections[i+1].strip()

                    # Extract strategy name from header or first line
                    strategy_name = section_header
                    if not strategy_name:
                        first_line = section_content.split('\n')[0].strip()
                        strategy_name = first_line

                    # Extract key cards - look for bullet points, numbers, or capital letters at line start
                    key_cards = []
                    description = ""
                    power_level = 0

                    for line in section_content.split('\n'):
                        line = line.strip()

                        # Look for power level rating
                        if re.search(r'(?:power|rating|score).*?(\d+)\s*(/|out of)\s*10', line.lower()):
                            match = re.search(r'(?:power|rating|score).*?(\d+)\s*(/|out of)\s*10', line.lower())
                            if match:
                                power_level = int(match.group(1))

                        # If line starts with a bullet, number, or capital letter, it might be a card
                        elif line.startswith('-') or line.startswith('*') or re.match(r'^\d+\.', line) or re.match(r'^[A-Z]', line):
                            # Clean up formatting
                            card = re.sub(r'^[-*•\d.)\]]+\s*', '', line)

                            # Remove explanations after the card name
                            if ' - ' in card:
                                card = card.split(' - ')[0].strip()
                            elif ' – ' in card:
                                card = card.split(' – ')[0].strip()
                            elif ' — ' in card:
                                card = card.split(' — ')[0].strip()
                            elif ': ' in card:
                                card = card.split(': ')[0].strip()

                            # Remove brackets if present
                            card = re.sub(r'[\[\]]', '', card)

                            # Validate card exists in database
                            if card in card_dict:
                                key_cards.append(card)
                        else:
                            # If not a card or power level, it's part of the description
                            description += line + " "

                    # Only add if we found some key cards
                    if key_cards:
                        win_conditions.append({
                            'name': strategy_name,
                            'description': description.strip(),
                            'key_cards': key_cards,
                            'power_level': power_level
                        })

        # If we failed to extract win conditions, use preset ones
        if not win_conditions:
            # Filter preset conditions to ensure cards exist in commander's color identity
            valid_presets = []
            for condition in preset_win_conditions:
                valid_cards = [card for card in condition['key_cards']
                              if card in card_dict and
                              all(c in commander_identity for c in card_dict[card].get('color_identity', []))]

                if valid_cards:
                    valid_condition = condition.copy()
                    valid_condition['key_cards'] = valid_cards
                    valid_presets.append(valid_condition)

            win_conditions = valid_presets

        # If we still don't have any, create a generic one
        if not win_conditions:
            logging.warning("Using default generic win condition")
            win_conditions = [{
                'name': 'Generic Synergy',
                'description': f'A balanced approach utilizing {commander}\'s abilities.',
                'key_cards': [],
                'power_level': 5
            }]

        return win_conditions, win_condition_text

    except Exception as e:
        logging.warning(f"Error parsing win conditions: {str(e)}")

        # Fall back to preset win conditions
        valid_presets = []
        for condition in preset_win_conditions:
            valid_cards = [card for card in condition['key_cards']
                          if card in card_dict and
                          all(c in commander_identity for c in card_dict[card].get('color_identity', []))]

            if valid_cards:
                valid_condition = condition.copy()
                valid_condition['key_cards'] = valid_cards
                valid_presets.append(valid_condition)

        if valid_presets:
            return valid_presets, win_condition_text
        else:
            return [{
                'name': 'Generic Synergy',
                'description': f'A balanced approach utilizing {commander}\'s abilities.',
                'key_cards': [],
                'power_level': 5
            }], win_condition_text

def generate_random_deck(commander, commander_identity):
    """Generate a random but coherent deck for the given commander"""
    logging.info(f"Generating random deck for {commander}")

    # Start with an empty deck
    deck = {}

    # Add commander if not already in deck
    if commander not in deck:
        deck[commander] = 1

    # Define card categories and their target percentages
    categories = {
        'Ramp': 0.10,     # 10% ramp effects
        'Draw': 0.10,     # 10% card draw
        'Removal': 0.10,  # 10% removal
        'Wrath': 0.05,    # 5% board wipes
        'Creatures': 0.25, # 25% creatures
        'Synergy': 0.15,  # 15% synergy pieces
        'Lands': 0.38     # 38% lands (including basic lands)
    }

    # Calculate target counts
    total_cards = 99  # 99 cards plus commander = 100
    card_counts = {category: int(percentage * total_cards) for category, percentage in categories.items()}

    # Adjust to ensure we hit exactly 99 cards
    remaining = total_cards - sum(card_counts.values())
    if remaining > 0:
        card_counts['Creatures'] += remaining

    # Add cards by category
    for category, count in card_counts.items():
        if category == 'Lands':
            # Handle lands separately to ensure proper basic land distribution
            continue

        logging.info(f"Adding {count} {category} cards")

        # Generate query based on category
        if category == 'Ramp':
            queries = ['type:artifact text:add mana', 'text:"add mana"', 'text:ritual']
        elif category == 'Draw':
            queries = ['text:"draw a card"', 'text:"draw cards"']
        elif category == 'Removal':
            queries = ['text:destroy target', 'text:exile target', 'text:damage to target']
        elif category == 'Wrath':
            queries = ['text:"destroy all"', 'text:"exile all"', 'text:"damage to all"']
        elif category == 'Creatures':
            queries = ['type:creature power>2', 'type:creature toughness>2', 'type:creature text:when']
        elif category == 'Synergy':
            # Generate synergy queries based on commander text
            queries = generate_synergy_queries(commander)
            if not queries:
                queries = ['type:instant', 'type:sorcery']

        # Add cards for each query
        added = 0
        exclude_list = list(deck.keys())

        for query in queries:
            if added >= count:
                break

            results = search_cards_by_criteria(query, commander_identity, exclude_cards=exclude_list)
            random.shuffle(results)  # Randomize the results

            # Add a portion of results
            to_add = min(len(results), count - added)
            for i in range(to_add):
                if i < len(results):
                    card_name = results[i]
                    deck[card_name] = 1
                    exclude_list.append(card_name)
                    added += 1

    # Add lands to reach target
    lands_to_add = card_counts['Lands']

    # First add some non-basic lands if available
    nonbasic_query = 'type:land -type:basic'
    nonbasic_lands = search_cards_by_criteria(nonbasic_query, commander_identity, exclude_cards=list(deck.keys()), n=10)

    for land in nonbasic_lands:
        if lands_to_add <= 0:
            break
        deck[land] = 1
        lands_to_add -= 1

    # Fill the rest with basic lands
    if lands_to_add > 0:
        # Determine which basic lands to add based on color identity
        land_types = []
        for color in commander_identity:
            if color in BASIC_LANDS:
                land_types.append(BASIC_LANDS[color])

        # If no colors or colorless, add Wastes
        if not land_types:
            land_types = ['Wastes']

        # Add equal numbers of each basic land type
        per_land = lands_to_add // len(land_types)
        remainder = lands_to_add % len(land_types)

        for i, land in enumerate(land_types):
            qty = per_land + (1 if i < remainder else 0)
            if land in deck:
                deck[land] += qty
            else:
                deck[land] = qty

    return deck

def main():
    cards = load_card_data()
    setup_retriever(cards)

    model_dir = '/content/drive/MyDrive/MTGModel/real_deck_training/final-model'
    tokenizer, model = load_language_model(model_dir)

    commander = input("Enter commander name: ")
    commander_identity = get_commander_identity(commander)

    if not commander_identity:
        logging.error(f"Could not determine color identity for {commander}")
        similar_commanders = search_similar_cards(commander, n=5)
        print(f"Did you mean one of these? {', '.join(similar_commanders)}")
        commander = input("Try entering commander name again: ")
        commander_identity = get_commander_identity(commander)
        if not commander_identity:
            print("Still couldn't find commander. Using default color identity (colorless).")
            commander_identity = []

    print(f"\nAnalyzing {commander} (Color Identity: {', '.join(commander_identity)})")

    # First, determine possible win conditions
    win_conditions, win_condition_text = determine_win_condition(commander, tokenizer, model, commander_identity)

    # Display win conditions to user
    print("\n=== Possible Win Conditions ===")
    print(win_condition_text)
    print("\n===========================")

    # Let user select a win condition or choose random
    if len(win_conditions) > 1:
        print("\nSelect a win condition to build around:")
        for i, wc in enumerate(win_conditions):
            key_cards_str = ", ".join(wc.get('key_cards', [])[:3])
            power_level = wc.get('power_level', 0)
            power_str = f" (Power: {power_level}/10)" if power_level > 0 else ""
            print(f"{i+1}. {wc['name']}{power_str} - Key cards: {key_cards_str}...")

        print(f"{len(win_conditions)+1}. Random - Let me pick a random win condition")
        print(f"{len(win_conditions)+2}. Fully Random Deck - Generate a completely random deck")

        selection = input(f"Enter 1-{len(win_conditions)+2} [1]: ").strip()
        if not selection:
            selected_idx = 0
        else:
            try:
                selected_idx = int(selection) - 1
                if selected_idx < 0 or selected_idx > len(win_conditions) + 1:
                    selected_idx = 0
            except ValueError:
                selected_idx = 0

        # Handle random selection
        if selected_idx == len(win_conditions):
            selected_idx = random.randint(0, len(win_conditions) - 1)
            print(f"Randomly selected: {win_conditions[selected_idx]['name']}")

        # Handle fully random deck
        elif selected_idx == len(win_conditions) + 1:
            print(f"\nGenerating random deck for {commander}")
            random_deck = generate_random_deck(commander, commander_identity)

            # Print final deck
            type_counts = count_card_types(random_deck)
            print(f"\nRandom Deck for {commander} (Color Identity: {', '.join(commander_identity)}):")
            print(f"Total cards: {sum(random_deck.values())}")
            print(f"Card type distribution: {type_counts}")

            print_formatted_deck(random_deck, commander)
            return
    else:
        selected_idx = 0

    # Use selected win condition
    selected_win_condition = win_conditions[selected_idx]
    print(f"\nGenerating deck for {commander} with '{selected_win_condition['name']}' win condition")

    # Check if we should use a temperature boost for more variation
    use_variation = input("Would you like more variety in the generated deck? (y/n) [n]: ").strip().lower()
    temp_boost = 0.2 if use_variation == 'y' else 0.0

    # Generate deck with chosen win condition
    deck_text = generate_conditioned_deck(commander, tokenizer, model, selected_win_condition, commander_identity, temp_boost)
    initial_deck = extract_deck(deck_text)

    # Add key cards from win condition if not already in deck
    for card in selected_win_condition.get('key_cards', []):
        if card not in initial_deck and card != commander:
            card_info = card_dict.get(card)
            if card_info and all(c in commander_identity for c in card_info.get('color_identity', [])):
                initial_deck[card] = 1

    # Validate and filter cards
    valid_deck, invalid_cards = validate_deck_identity(initial_deck, commander_identity)

    # Complete deck to 100 cards
    final_deck = complete_deck(valid_deck, commander, commander_identity, selected_win_condition)

    # Add commander as 1x if not already in deck
    if commander not in final_deck:
        final_deck[commander] = 1

    # Print final deck
    type_counts = count_card_types(final_deck)
    print(f"\nFinal Deck for {commander} (Color Identity: {', '.join(commander_identity)}):")
    print(f"Total cards: {sum(final_deck.values())}")
    print(f"Card type distribution: {type_counts}")

    print_formatted_deck(final_deck, commander)

def print_formatted_deck(deck, commander):
    """Print the deck in a nicely formatted way"""
    print("\n--- Commander ---")
    print(f"1x {commander}")

    print("\n--- Lands ---")
    for card, qty in sorted(deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if 'Land' in types or card in BASIC_LANDS.values():
            print(f"{qty}x {card}")

    print("\n--- Creatures ---")
    for card, qty in sorted(deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if 'Creature' in types and 'Land' not in types:
            print(f"{qty}x {card}")

    print("\n--- Artifacts ---")
    for card, qty in sorted(deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if 'Artifact' in types and 'Creature' not in types and 'Land' not in types:
            print(f"{qty}x {card}")

    print("\n--- Enchantments ---")
    for card, qty in sorted(deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if 'Enchantment' in types and 'Creature' not in types and 'Land' not in types:
            print(f"{qty}x {card}")

    print("\n--- Planeswalkers ---")
    for card, qty in sorted(deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if 'Planeswalker' in types:
            print(f"{qty}x {card}")

    print("\n--- Instants and Sorceries ---")
    for card, qty in sorted(deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if ('Instant' in types or 'Sorcery' in types) and 'Creature' not in types and 'Land' not in types:
            print(f"{qty}x {card}")

def generate_conditioned_deck(commander, tokenizer, model, win_condition, commander_identity, temp_boost=0):
    """Generate a deck with a specific win condition in mind"""
    # Prepare a prompt that specifies the win condition
    key_cards_str = ", ".join(win_condition.get('key_cards', [])[:5])  # Use top 5 key cards for prompt
    description = win_condition.get('description', f"A {win_condition['name']} strategy")

    prompt = f"""Generate a Commander deck for {commander} focusing on the "{win_condition['name']}" win condition.
Description: {description}
Include the following key cards if within color identity: {key_cards_str}.
Make sure the deck is cohesive and all cards work toward the win condition.
Format as a list with quantities (e.g., "1x Card Name").
Include exactly 99 cards (excluding the commander)."""

    inputs = tokenizer(prompt, return_tensors='pt')
    outputs = model.generate(
        **inputs,
        max_length=2000,
        temperature=0.65 + temp_boost,  # Adjust temperature based on desired variation
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    conditioned_deck_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return conditioned_deck_text

    # Print final deck with type categorization
    type_counts = count_card_types(final_deck)
    print(f"\nFinal Deck for {commander} (Color Identity: {', '.join(commander_identity)}):")
    print(f"Total cards: {sum(final_deck.values())}")
    print(f"Card type distribution: {type_counts}")
    print("\n--- Commander ---")
    print(f"1x {commander}")
    print("\n--- Lands ---")
    for card, qty in sorted(final_deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if 'Land' in types or card in BASIC_LANDS.values():
            print(f"{qty}x {card}")

    print("\n--- Creatures ---")
    for card, qty in sorted(final_deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if 'Creature' in types and 'Land' not in types:
            print(f"{qty}x {card}")

    print("\n--- Spells ---")
    for card, qty in sorted(final_deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if not ('Land' in types or 'Creature' in types or card in BASIC_LANDS.values()):
            print(f"{qty}x {card}")

if __name__ == '__main__':
    main()

In [ ]:
import os
import json
import logging
import requests
import torch
import faiss
import numpy as np
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
import random

logging.basicConfig(level=logging.INFO)

card_dict = {}
faiss_index, embed_model, card_texts, card_objects = None, None, [], []

COMMANDER_BANNED_CARDS = ["Ancestral Recall", "Balance", "Biorhythm", "Black Lotus", "Braids, Cabal Minion", "Channel", "Chaos Orb",
"Coalition Victory", "Dockside Extortionist", "Emrakul, the Aeons Torn", "Erayo, Soratami Ascendant", "Falling Star",
"Fastbond", "Flash", "Gifts Ungiven", "Golos, Tireless Pilgrim", "Griselbrand", "Hullbreacher", "Iona, Shield of Emeria",
"Jeweled Lotus", "Karakas", "Leovold, Emissary of Trest", "Library of Alexandria", "Limited Resources", "Lutri, the Spellchaser",
"Mana Crypt", "Mox Emerald", "Mox Jet", "Mox Pearl", "Mox Ruby", "Mox Sapphire", "Nadu, Winged Wisdom", "Panoptic Mirror",
"Paradox Engine", "Primeval Titan", "Prophet of Kruphix", "Recurring Nightmare", "Rofellos, Llanowar Emissary", "Shahrazad",
"Sundering Titan", "Sway of the Stars", "Sylvan Primordial", "Time Vault", "Time Walk", "Tinker", "Tolarian Academy",
"Trade Secrets", "Upheaval", "Yawgmoth's Bargain"]

BASIC_LANDS = {
    'W': 'Plains',
    'U': 'Island',
    'B': 'Swamp',
    'R': 'Mountain',
    'G': 'Forest'
}

CARD_CATEGORIES = {
    'ramp': ['type:artifact text:add mana', 'type:land text:add mana', 'type:creature text:"add mana"'],
    'draw': ['text:"draw a card"', 'text:"draw cards"'],
    'removal': ['text:destroy', 'text:exile', 'text:"damage to"'],
    'wrath': ['text:"destroy all"', 'text:"exile all"', 'text:"damage to all"'],
    'synergy': []  # Will be populated based on commander
}
# Define Commander brackets for power level targeting
COMMANDER_BRACKETS = {
    1: {
        'name': 'Exhibition',
        'description': 'Heavily themed decks where winning is not the primary goal. Focus on showing off a concept or project.',
        'restrictions': [
            'No Mass Land Denial',
            'No Game Changer Cards',
            'No 2-card Combos',
            'Extra Turns spells should be thematic and not intended to be chained or looped',
            'Tutors should be sparse and specific'
        ],
        'power_level': (1, 3)  # Power range 1-3
    },
    2: {
        'name': 'Core',
        'description': 'The bulk of casual decks and modern precons. Draw power from synergy rather than card quality.',
        'restrictions': [
            'No Mass Land Denial',
            'No Game Changer Cards',
            'No 2-card Combos',
            'Extra Turns spells should be minimal and not intended to be chained or looped',
            'Tutors should be sparse and specific'
        ],
        'power_level': (3, 5)  # Power range 3-5
    },
    3: {
        'name': 'Upgraded',
        'description': 'Intentionally tuned and upgraded decks with improved power level. Theme and flavor take a back seat.',
        'restrictions': [
            'No Mass Land Denial',
            'Up to 3 Game Changer Cards',
            'No 2-card Combos before turn 6',
            'Some extra turns and tutors expected (but not looped)'
        ],
        'power_level': (5, 7)  # Power range 5-7
    },
    4: {
        'name': 'Optimized',
        'description': 'High-powered decks using any cards and strategies, though not meta-focused or tournament driven.',
        'restrictions': [
            'None (other than the banned list)'
        ],
        'power_level': (7, 9)  # Power range 7-9
    },
    5: {
        'name': 'cEDH',
        'description': 'Competitive EDH decks designed for tournament play, making choices dependent on the competitive meta.',
        'restrictions': [
            'None (other than the banned list)'
        ],
        'power_level': (9, 10)  # Power range 9-10
    }
}

# Lists of cards that shouldn't appear in lower bracket decks
GAME_CHANGER_CARDS = [
    "Cyclonic Rift", "Expropriate", "Craterhoof Behemoth", "Rise of the Dark Realms",
    "Torment of Hailfire", "Omniscience", "Zacama, Primal Calamity", "Jin-Gitaxias, Core Augur",
    "Vorinclex, Voice of Hunger", "Elesh Norn, Grand Cenobite", "Ulamog, the Ceaseless Hunger",
    "Kozilek, Butcher of Truth", "Smothering Tithe", "Rhystic Study", "Mystic Remora",
    "Necropotence", "Sylvan Library", "Demonic Tutor", "Vampiric Tutor", "Worldly Tutor",
    "Time Spiral", "Force of Will", "Mana Drain", "Fierce Guardianship", "Toxic Deluge"
]

MASS_LAND_DENIAL = [
    "Armageddon", "Ravages of War", "Catastrophe", "Wildfire", "Burning of Xinye",
    "Ruination", "Death Cloud", "Jokulhaups", "Obliterate", "Decree of Annihilation",
    "Fall of the Thran", "Impending Disaster", "Natural Balance", "Winter Orb",
    "Static Orb", "Rising Waters", "Stasis", "Blood Moon", "Magus of the Moon"
]

EXTRA_TURN_SPELLS = [
    "Time Warp", "Temporal Manipulation", "Capture of Jingzhou", "Time Stretch",
    "Nexus of Fate", "Temporal Mastery", "Walk the Aeons", "Karn's Temporal Sundering",
    "Part the Waterveil", "Temporal Trespass", "Alrund's Epiphany", "Expropriate"
]

TUTOR_CARDS = [
    "Demonic Tutor", "Vampiric Tutor", "Worldly Tutor", "Mystical Tutor", "Enlightened Tutor",
    "Imperial Seal", "Diabolic Tutor", "Grim Tutor", "Personal Tutor", "Sylvan Tutor",
    "Demonic Consultation", "Tainted Pact", "Scheming Symmetry", "Cruel Tutor",
    "Diabolic Intent", "Gamble", "Chord of Calling", "Green Sun's Zenith", "Tooth and Nail",
    "Eladamri's Call", "Idyllic Tutor", "Beseech the Queen", "Final Parting"
]

COMBO_CARDS = {
    # Format: card_name: [combo_partner1, combo_partner2, ...]
    "Thassa's Oracle": ["Demonic Consultation", "Tainted Pact"],
    "Laboratory Maniac": ["Demonic Consultation", "Tainted Pact"],
    "Jace, Wielder of Mysteries": ["Demonic Consultation", "Tainted Pact"],
    "Isochron Scepter": ["Dramatic Reversal"],
    "Dramatic Reversal": ["Isochron Scepter"],
    "Splinter Twin": ["Deceiver Exarch", "Pestermite", "Zealous Conscripts"],
    "Kiki-Jiki, Mirror Breaker": ["Deceiver Exarch", "Pestermite", "Zealous Conscripts"],
    "Mikaeus, the Unhallowed": ["Triskelion", "Walking Ballista"],
    "Triskelion": ["Mikaeus, the Unhallowed"],
    "Walking Ballista": ["Mikaeus, the Unhallowed", "Heliod, Sun-Crowned"],
    "Heliod, Sun-Crowned": ["Walking Ballista"],
    "Sanguine Bond": ["Exquisite Blood"],
    "Exquisite Blood": ["Sanguine Bond"],
    "Painters Servant": ["Grindstone"],
    "Grindstone": ["Painters Servant"],
    "Doomsday": ["Thassa's Oracle", "Laboratory Maniac"],
    "Food Chain": ["Eternal Scourge", "Misthollow Griffin", "Squee, the Immortal"],
    "Demonic Consultation": ["Thassa's Oracle", "Laboratory Maniac", "Jace, Wielder of Mysteries"],
    "Tainted Pact": ["Thassa's Oracle", "Laboratory Maniac", "Jace, Wielder of Mysteries"],
    "Earthcraft": ["Squirrel Nest"],
    "Squirrel Nest": ["Earthcraft"]
}

def check_card_bracket_compatibility(card_name, bracket):
    """Check if a card is compatible with the bracket's restrictions"""
    if card_name not in card_dict:
        return True  # If we don't have card info, default to allowing it

    # Always allow the card in brackets 4 and 5 (no restrictions)
    if bracket >= 4:
        return True

    # Check for mass land denial
    if bracket <= 3 and card_name in MASS_LAND_DENIAL:
        return False

    # Check for game changers in lower brackets
    if bracket <= 2 and card_name in GAME_CHANGER_CARDS:
        return False

    # Extra turns limitations
    if bracket <= 2 and card_name in EXTRA_TURN_SPELLS:
        return False

    # Tutor limitations (stricter for lower brackets)
    if bracket <= 2 and card_name in TUTOR_CARDS:
        # For bracket 2, allow some specific tutors but limit quantity
        return False

    # Check for combo pieces
    if bracket <= 3 and card_name in COMBO_CARDS:
        return False

    return True

def filter_deck_for_bracket(deck, bracket):
    """Filter a deck to comply with the chosen bracket's restrictions"""
    filtered_deck = {}
    removed_cards = {}

    # If bracket is 4 or 5, no restrictions apply
    if bracket >= 4:
        return deck, {}

    # Count restricted card types
    game_changers_count = 0
    tutor_count = 0
    extra_turns_count = 0
    combo_pieces = {}

    # First pass: identify all restricted cards
    for card, qty in deck.items():
        # Check for game changers
        if card in GAME_CHANGER_CARDS:
            game_changers_count += qty

        # Check for tutors
        if card in TUTOR_CARDS:
            tutor_count += qty

        # Check for extra turns
        if card in EXTRA_TURN_SPELLS:
            extra_turns_count += qty

        # Check for combo pieces
        if card in COMBO_CARDS:
            combo_partners = COMBO_CARDS[card]
            for partner in combo_partners:
                if partner in deck:
                    if card not in combo_pieces:
                        combo_pieces[card] = []
                    combo_pieces[card].append(partner)

    # Second pass: apply bracket-specific filters
    for card, qty in deck.items():
        # Exhibition and Core brackets (1 & 2)
        if bracket <= 2:
            # No game changers
            if card in GAME_CHANGER_CARDS:
                removed_cards[card] = qty
                continue

            # No mass land denial
            if card in MASS_LAND_DENIAL:
                removed_cards[card] = qty
                continue

            # No extra turns
            if card in EXTRA_TURN_SPELLS:
                removed_cards[card] = qty
                continue

            # Limited tutors (allow just 1-2 specific ones for bracket 2)
            if card in TUTOR_CARDS:
                if bracket == 1 or (bracket == 2 and tutor_count > 2):
                    removed_cards[card] = qty
                    continue

            # No 2-card combos
            if card in combo_pieces:
                removed_cards[card] = qty
                continue

        # Upgraded bracket (3)
        elif bracket == 3:
            # No mass land denial
            if card in MASS_LAND_DENIAL:
                removed_cards[card] = qty
                continue

            # Limit game changers to 3
            if card in GAME_CHANGER_CARDS and game_changers_count > 3:
                removed_cards[card] = qty
                game_changers_count -= qty
                continue

        # If we made it here, the card is allowed
        filtered_deck[card] = qty

    return filtered_deck, removed_cards

def replace_removed_cards(deck, removed_cards, commander_identity, bracket):
    """Replace cards that were removed due to bracket restrictions"""
    if not removed_cards:
        return deck

    # Calculate how many cards we need to replace
    cards_to_add = sum(removed_cards.values())
    logging.info(f"Replacing {cards_to_add} cards removed due to bracket {bracket} restrictions")

    # Get list of cards already in the deck
    exclude_list = list(deck.keys()) + list(removed_cards.keys())

    # Generate replacement queries based on removed card types
    replacement_queries = []

    # Check what types of cards were removed and generate appropriate replacement queries
    has_removed_tutors = any(card in TUTOR_CARDS for card in removed_cards)
    has_removed_extra_turns = any(card in EXTRA_TURN_SPELLS for card in removed_cards)
    has_removed_land_denial = any(card in MASS_LAND_DENIAL for card in removed_cards)
    has_removed_game_changers = any(card in GAME_CHANGER_CARDS for card in removed_cards)
    has_removed_combos = any(card in COMBO_CARDS for card in removed_cards)

    # Add replacement queries based on what was removed
    if has_removed_tutors:
        replacement_queries.extend(["card draw", "card advantage"])

    if has_removed_extra_turns:
        replacement_queries.extend(["additional combat", "untap effects"])

    if has_removed_land_denial:
        replacement_queries.extend(["targeted removal", "single target land destruction"])

    if has_removed_game_changers:
        replacement_queries.extend(["value engine", "gradual advantage"])

    if has_removed_combos:
        replacement_queries.extend(["synergy pieces", "value creatures"])

    # If we don't have specific replacements, use general good stuff
    if not replacement_queries:
        replacement_queries = ["card draw", "removal", "ramp", "board wipe", "value creature"]

    # Add cards by cycling through queries
    added_count = 0
    query_index = 0

    while added_count < cards_to_add:
        query = replacement_queries[query_index % len(replacement_queries)]
        query_index += 1

        # Search for cards matching the query
        results = search_cards_by_criteria(query, commander_identity, exclude_cards=exclude_list, n=20)

        # Filter results for bracket compatibility
        bracket_compatible = [card for card in results if check_card_bracket_compatibility(card, bracket)]

        if bracket_compatible:
            # Pick one randomly
            card_to_add = random.choice(bracket_compatible)

            # Add to deck
            if card_to_add in deck:
                deck[card_to_add] += 1
            else:
                deck[card_to_add] = 1

            exclude_list.append(card_to_add)
            added_count += 1

            # If we're struggling to find cards, relax the search a bit
            if query_index > 10 * len(replacement_queries):
                # Try to add basic lands as a last resort
                for color in commander_identity:
                    if color in BASIC_LANDS:
                        land = BASIC_LANDS[color]
                        if land in deck:
                            deck[land] += 1
                        else:
                            deck[land] = 1
                        added_count += 1
                        if added_count >= cards_to_add:
                            break
                break

    return deck

def get_appropriate_bracket_for_win_condition(win_condition):
    """Determine the appropriate bracket for a win condition based on its power level"""
    power_level = win_condition.get('power_level', 5)  # Default to mid-power if not specified

    if power_level <= 3:
        return 1  # Exhibition
    elif power_level <= 5:
        return 2  # Core
    elif power_level <= 7:
        return 3  # Upgraded
    elif power_level <= 9:
        return 4  # Optimized
    else:
        return 5  # cEDH
def load_card_data():
    global card_dict, card_objects
    bulk_data = requests.get("https://api.scryfall.com/bulk-data").json()
    oracle_url = next(item for item in bulk_data["data"] if item["name"] == "Oracle Cards")["download_uri"]
    cards = requests.get(oracle_url).json()
    card_objects = []
    for card in cards:
        if 'name' not in card or 'legalities' not in card:
            continue
        if card.get('legalities', {}).get('commander') in ['legal', 'restricted']:
            card_objects.append(card)
            card_dict[card['name']] = {
                'color_identity': card.get('color_identity', []),
                'type_line': card.get('type_line', ''),
                'oracle_text': card.get('oracle_text', ''),
                'mana_value': card.get('cmc', 0),
                'types': extract_types(card.get('type_line', ''))
            }
            if 'card_faces' in card:
                for face in card['card_faces']:
                    if 'name' in face and face['name'] != card['name']:
                        card_dict[face['name']] = {
                            'color_identity': card.get('color_identity', []),
                            'type_line': face.get('type_line', ''),
                            'oracle_text': face.get('oracle_text', ''),
                            'mana_value': card.get('cmc', 0),
                            'types': extract_types(face.get('type_line', ''))
                        }
    logging.info(f"Loaded {len(card_objects)} legal Commander cards")
    return card_objects

def extract_types(type_line):
    types = []
    if "Land" in type_line:
        types.append("Land")
    if "Creature" in type_line:
        types.append("Creature")
    if "Artifact" in type_line:
        types.append("Artifact")
    if "Enchantment" in type_line:
        types.append("Enchantment")
    if "Planeswalker" in type_line:
        types.append("Planeswalker")
    if "Instant" in type_line:
        types.append("Instant")
    if "Sorcery" in type_line:
        types.append("Sorcery")
    return types

def setup_retriever(cards):
    global faiss_index, embed_model, card_texts
    embed_model = SentenceTransformer('all-MiniLM-L6-v2')

    # Create more detailed card texts for better semantic search
    card_texts = []
    for c in cards:
        if 'name' not in c or 'type_line' not in c:
            continue

        # Extract key card properties for embedding context
        card_name = c['name']
        type_line = c['type_line']
        oracle_text = c.get('oracle_text', '')
        keywords = c.get('keywords', [])
        mana_cost = c.get('mana_cost', '')

        # Create a rich text representation
        text = f"{card_name}. Cost: {mana_cost}. Types: {type_line}. "

        if keywords:
            text += f"Keywords: {', '.join(keywords)}. "

        if oracle_text:
            text += f"Text: {oracle_text}"

        # Add card faces for dual-faced cards
        if 'card_faces' in c:
            face_texts = []
            for face in c['card_faces']:
                if 'name' in face and 'type_line' in face:
                    face_name = face['name']
                    face_type = face['type_line']
                    face_text = face.get('oracle_text', '')
                    face_cost = face.get('mana_cost', '')

                    face_desc = f"{face_name}. Cost: {face_cost}. Types: {face_type}. "
                    if face_text:
                        face_desc += f"Text: {face_text}"
                    face_texts.append(face_desc)

            if face_texts:
                text += f" Faces: {' | '.join(face_texts)}"

        card_texts.append(text)

    logging.info(f"Generating embeddings for {len(card_texts)} cards...")

    # Create embeddings with batched processing for memory efficiency
    batch_size = 256
    all_embeddings = []

    for i in range(0, len(card_texts), batch_size):
        batch = card_texts[i:i + batch_size]
        batch_embeddings = embed_model.encode(batch, show_progress_bar=True)
        all_embeddings.append(batch_embeddings)

    embeddings = np.vstack(all_embeddings).astype('float32')

    # Create and populate the FAISS index
    faiss_index = faiss.IndexFlatL2(embeddings.shape[1])
    faiss_index.add(embeddings)
    logging.info("Embeddings completed and indexed")

def load_language_model(model_dir):
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForCausalLM.from_pretrained(model_dir)
    return tokenizer, model

def get_commander_identity(commander_name):
    commander_info = card_dict.get(commander_name)
    if not commander_info:
        logging.warning(f"Commander '{commander_name}' not found in database")
        # Try to find similar commander names
        similar_names = search_similar_cards(commander_name, n=5)
        logging.info(f"Did you mean one of these? {', '.join(similar_names)}")
        return []
    return commander_info.get('color_identity', [])

def validate_deck_identity(deck, commander_identity):
    valid_deck, invalid_cards = {}, []
    for card, qty in deck.items():
        # Skip validation for basic lands that match commander color identity
        if card in BASIC_LANDS.values():
            if not commander_identity or any(color in commander_identity for color, land in BASIC_LANDS.items() if land == card):
                valid_deck[card] = qty
                continue

        card_info = card_dict.get(card)
        if not card_info:
            invalid_cards.append(f"{card} (not found)")
            continue

        card_identity = card_info.get('color_identity', [])

        # Check if card's color identity is a subset of commander's identity
        if all(c in commander_identity for c in card_identity) and card not in COMMANDER_BANNED_CARDS:
            valid_deck[card] = qty
        else:
            reason = "banned" if card in COMMANDER_BANNED_CARDS else "color identity mismatch"
            invalid_cards.append(f"{card} ({reason})")

    logging.info(f"Removed {len(invalid_cards)} invalid cards: {', '.join(invalid_cards)}")
    return valid_deck, invalid_cards

def search_similar_cards(query, n=10):
    if not faiss_index or not card_objects:
        return []

    query_embedding = embed_model.encode([query])
    distances, indices = faiss_index.search(query_embedding, n)
    return [card_objects[i]['name'] for i in indices[0] if i < len(card_objects)]

def search_cards_by_criteria(query, commander_identity, exclude_cards=None, n=30):
    """Search for cards matching query with specified color identity"""
    if not exclude_cards:
        exclude_cards = set()
    else:
        exclude_cards = set(exclude_cards)

    query_embedding = embed_model.encode([query])
    distances, indices = faiss_index.search(query_embedding, n*3)  # Get more results to filter

    results = []
    for i in indices[0]:
        if i < len(card_objects):
            card = card_objects[i]
            if 'name' not in card or card['name'] in exclude_cards:
                continue

            # Check color identity
            card_identity = card.get('color_identity', [])
            if not all(c in commander_identity for c in card_identity):
                continue

            # Check legality
            if card.get('legalities', {}).get('commander') not in ['legal', 'restricted'] or card['name'] in COMMANDER_BANNED_CARDS:
                continue

            results.append(card['name'])
            if len(results) >= n:
                break

    return results

def get_commander_themes(commander_name):
    """Identify potential themes for the commander"""
    if not commander_name in card_dict:
        return []

    commander_info = card_dict[commander_name]
    oracle_text = commander_info.get('oracle_text', '')

    themes = []
    if 'draw' in oracle_text.lower() or 'card' in oracle_text.lower():
        themes.append("card draw")
    if 'damage' in oracle_text.lower():
        themes.append("damage")
    if 'counter' in oracle_text.lower():
        themes.append("counters")
    if 'token' in oracle_text.lower():
        themes.append("tokens")
    if 'graveyard' in oracle_text.lower() or 'cemetery' in oracle_text.lower():
        themes.append("graveyard")
    if 'sacrifice' in oracle_text.lower():
        themes.append("sacrifice")
    if 'discard' in oracle_text.lower():
        themes.append("discard")

    return themes

def generate_synergy_queries(commander_name):
    """Generate search queries based on commander themes"""
    themes = get_commander_themes(commander_name)
    synergy_queries = []

    # Add commander name for direct synergy
    synergy_queries.append(f"synergy with {commander_name}")

    # Add theme-based queries
    for theme in themes:
        synergy_queries.append(f"cards that work with {theme}")

    # Add general synergy queries based on commander text
    if commander_name in card_dict:
        commander_text = card_dict[commander_name].get('oracle_text', '')
        key_terms = re.findall(r'\b\w+\b', commander_text.lower())

        # Filter out common words
        stop_words = {'a', 'an', 'the', 'in', 'on', 'at', 'to', 'for', 'and', 'or', 'of', 'with', 'by'}
        key_terms = [term for term in key_terms if term not in stop_words and len(term) > 3]

        # Add key terms as synergy queries
        for term in key_terms[:3]:  # Use top 3 terms
            synergy_queries.append(f"cards with {term}")

    return synergy_queries

def generate_deck(commander_name, tokenizer, model):
    """Generate initial deck using language model"""
    prompt = f"Generate a synergistic Commander decklist for {commander_name}. Include card names and quantities, with 99 cards plus the commander."
    inputs = tokenizer(prompt, return_tensors='pt')
    outputs = model.generate(
        **inputs,
        max_length=2000,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def extract_deck(deck_text):
    """Extract card names and quantities from generated text"""
    deck = {}

    # Pattern for "NxCardName" format
    pattern1 = r"(\d+)x ([^\n,]+)"
    # Pattern for "N CardName" format
    pattern2 = r"(\d+) ([^\n,]+)"
    # Pattern for lines with card names only
    pattern3 = r"^([^0-9\n][^\n]+)$"

    # Find all matches for each pattern
    matches1 = re.findall(pattern1, deck_text)
    matches2 = re.findall(pattern2, deck_text)

    # Process matches from patterns with quantities
    for qty_str, card in matches1 + matches2:
        try:
            qty = int(qty_str)
            card_name = card.strip()
            if card_name and 0 < qty <= 99:  # Sanity check
                deck[card_name] = qty
        except ValueError:
            continue

    # Only use pattern3 if we didn't get enough cards
    if len(deck) < 20:
        lines = deck_text.split('\n')
        for line in lines:
            line = line.strip()
            if line and not any(c.isdigit() for c in line[:2]):  # Avoid lines starting with numbers
                deck[line] = 1

    return deck

def complete_deck(partial_deck, commander_name, commander_identity, win_condition=None):
    """Complete deck with appropriate cards to reach 100 cards total"""
    logging.info(f"Completing deck with {len(partial_deck)} initial cards")

    # Create a copy of the partial deck
    completed_deck = dict(partial_deck)

    # Count cards by type
    card_count = sum(completed_deck.values())

    # Generate synergy search queries
    synergy_queries = generate_synergy_queries(commander_name)
    CARD_CATEGORIES['synergy'] = synergy_queries

    # If win condition is provided, add win condition-specific queries
    if win_condition:
        win_condition_name = win_condition.get('name', '')

        # Create targeted queries based on win condition
        win_queries = []

        # Basic win condition categorization
        if 'combo' in win_condition_name.lower():
            win_queries.extend([
                "infinite combo pieces",
                "combo enablers",
                "tutor effects"
            ])
        elif any(x in win_condition_name.lower() for x in ['damage', 'burn']):
            win_queries.extend([
                "direct damage spells",
                "damage doubler",
                "burn spells"
            ])
        elif 'token' in win_condition_name.lower():
            win_queries.extend([
                "token generators",
                "token doublers",
                "anthem effects"
            ])
        elif any(x in win_condition_name.lower() for x in ['mill', 'deck out']):
            win_queries.extend([
                "mill effects",
                "exile library cards"
            ])
        elif 'storm' in win_condition_name.lower():
            win_queries.extend([
                "storm cards",
                "cast multiple spells",
                "copy spells"
            ])

        # Add win condition name itself as a query
        win_queries.append(f"cards for {win_condition_name} strategy")

        # Add key cards from win condition as semantic anchors
        for key_card in win_condition.get('key_cards', []):
            if key_card in card_dict:
                win_queries.append(f"cards that work well with {key_card}")

        # Add these to our synergy queries
        CARD_CATEGORIES['win_condition'] = win_queries

    # Calculate how many cards to add
    cards_needed = 99 - card_count  # 99 cards plus commander = 100
    if cards_needed <= 0:
        return completed_deck

    logging.info(f"Need to add {cards_needed} more cards to reach 99")

    # Track all cards we're excluding
    exclude_list = list(completed_deck.keys())

    # Count card types in current deck
    type_counts = count_card_types(completed_deck)
    logging.info(f"Current type distribution: {type_counts}")

    # Calculate mana curve to analyze deck balance
    mana_curve = calculate_mana_curve(completed_deck)
    logging.info(f"Current mana curve: {mana_curve}")

    # Add necessary basic lands if missing (most decks need them)
    added_lands = 0
    if type_counts.get('Land', 0) < 36:  # Aim for 36 lands in total
        lands_needed = min(36 - type_counts.get('Land', 0), cards_needed)
        logging.info(f"Adding {lands_needed} basic lands")

        # Determine which basic lands to add based on color identity
        land_types = []
        for color in commander_identity:
            if color in BASIC_LANDS:
                land_types.append(BASIC_LANDS[color])

        # Add equal numbers of each basic land type
        if land_types:
            per_land = lands_needed // len(land_types)
            remainder = lands_needed % len(land_types)

            for i, land in enumerate(land_types):
                qty = per_land + (1 if i < remainder else 0)
                if land in completed_deck:
                    completed_deck[land] += qty
                else:
                    completed_deck[land] = qty
                added_lands += qty

    # Recalculate cards needed after adding lands
    cards_needed -= added_lands

    # If we still need cards, add by category
    if cards_needed > 0:
        # Define target card type distribution based on win condition
        if win_condition and 'name' in win_condition:
            win_condition_name = win_condition['name'].lower()

            # Adjust distribution based on win condition type
            if 'combo' in win_condition_name:
                target_distribution = {
                    'ramp': 0.15,      # More ramp for combo decks
                    'draw': 0.15,      # More card draw to find combo pieces
                    'removal': 0.10,   # Standard removal
                    'wrath': 0.05,     # Standard board wipes
                    'win_condition': 0.40, # Heavy focus on win condition
                    'synergy': 0.15    # Some general synergy
                }
            elif any(x in win_condition_name for x in ['aggro', 'damage', 'combat']):
                target_distribution = {
                    'ramp': 0.10,      # Standard ramp
                    'draw': 0.10,      # Standard draw
                    'removal': 0.15,   # More removal for combat-focused decks
                    'wrath': 0.05,     # Standard board wipes
                    'win_condition': 0.40, # Heavy focus on win condition
                    'synergy': 0.20    # More general synergy
                }
            elif 'control' in win_condition_name:
                target_distribution = {
                    'ramp': 0.10,      # Standard ramp
                    'draw': 0.15,      # More card draw for control
                    'removal': 0.20,   # More removal for control
                    'wrath': 0.10,     # More board wipes for control
                    'win_condition': 0.30, # Focus on win condition
                    'synergy': 0.15    # Some general synergy
                }
            else:
                # Default balanced distribution
                target_distribution = {
                    'ramp': 0.10,      # 10% ramp spells
                    'draw': 0.10,      # 10% card draw
                    'removal': 0.10,   # 10% removal
                    'wrath': 0.05,     # 5% board wipes
                    'win_condition': 0.35, # 35% win condition focus
                    'synergy': 0.30    # 30% general synergy
                }
        else:
            # Standard distribution without win condition
            target_distribution = {
                'ramp': 0.10,      # 10% ramp spells
                'draw': 0.10,      # 10% card draw
                'removal': 0.10,   # 10% removal
                'wrath': 0.05,     # 5% board wipes
                'synergy': 0.65    # 65% synergy cards
            }

        # Add cards by category
        for category, percentage in target_distribution.items():
            if category not in CARD_CATEGORIES:
                continue

            category_count = int(cards_needed * percentage)
            if category_count > 0:
                logging.info(f"Adding {category_count} {category} cards")
                added = 0

                # Use different queries for the category
                queries = CARD_CATEGORIES[category]
                if not queries:
                    continue

                for query in queries:
                    if added >= category_count:
                        break

                    # Search for matching cards
                    results = search_cards_by_criteria(
                        query,
                        commander_identity,
                        exclude_cards=exclude_list
                    )

                    # Add a portion of results
                    to_add = min(len(results), category_count - added)
                    for i in range(to_add):
                        card_name = results[i]
                        completed_deck[card_name] = 1
                        exclude_list.append(card_name)
                        added += 1

    # Analyze and balance mana curve if needed
    final_curve = calculate_mana_curve(completed_deck)
    curve_issues = analyze_mana_curve(final_curve)

    if curve_issues:
        logging.info(f"Mana curve issues detected: {curve_issues}")
        fix_mana_curve(completed_deck, final_curve, commander_identity, exclude_list)

    # Check final count
    total_cards = sum(completed_deck.values())
    if total_cards < 99:
        # Still need more cards - add random cards that match color identity
        logging.info(f"Still need {99 - total_cards} more cards")

        # Get cards that match color identity
        valid_cards = []
        for card_name, info in card_dict.items():
            if (card_name not in exclude_list and
                all(c in commander_identity for c in info.get('color_identity', [])) and
                card_name not in COMMANDER_BANNED_CARDS and
                not any(land == card_name for land in BASIC_LANDS.values())):
                valid_cards.append(card_name)

        # Randomly select remaining cards
        random.shuffle(valid_cards)
        for card_name in valid_cards:
            if sum(completed_deck.values()) >= 99:
                break
            completed_deck[card_name] = 1

    # If we have too many cards, trim some
    while sum(completed_deck.values()) > 99:
        # Find a card with qty > 1 to reduce
        for card, qty in list(completed_deck.items()):
            if qty > 1 and not any(land == card for land in BASIC_LANDS.values()):
                completed_deck[card] -= 1
                break
        else:
            # If all cards have qty=1, remove a random card
            non_essential = [card for card in completed_deck.keys()
                           if card != commander_name
                           and not any(land == card for land in BASIC_LANDS.values())
                           and (not win_condition or card not in win_condition.get('key_cards', []))]
            if non_essential:
                card_to_remove = random.choice(non_essential)
                del completed_deck[card_to_remove]

    return completed_deck

def calculate_mana_curve(deck):
    """Calculate the mana curve of the deck"""
    curve = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, '7+': 0}

    for card, qty in deck.items():
        card_info = card_dict.get(card)
        if not card_info:
            continue

        # Skip lands
        if 'Land' in card_info.get('types', []):
            continue

        mana_value = card_info.get('mana_value', 0)

        # Categorize by mana value
        if mana_value >= 7:
            curve['7+'] += qty
        else:
            curve[int(mana_value)] += qty

    return curve

def analyze_mana_curve(curve):
    """Analyze the mana curve for potential issues"""
    issues = []

    # Calculate total spells
    total_spells = sum(curve.values())

    if total_spells < 10:
        return []  # Not enough spells to analyze

    # Check for imbalances in the curve
    if curve[1] + curve[2] < total_spells * 0.2:
        issues.append("low_early_drops")

    if curve['7+'] > total_spells * 0.15:
        issues.append("top_heavy")

    # Check for gaps in the curve
    for i in range(2, 5):
        if curve[i] == 0:
            issues.append(f"gap_at_{i}")

    return issues

def fix_mana_curve(deck, curve, commander_identity, exclude_list):
    """Attempt to fix issues with the mana curve"""
    issues = analyze_mana_curve(curve)

    if not issues:
        return deck

    # Fix common issues
    if "low_early_drops" in issues:
        # Add more low-cost cards
        logging.info("Fixing mana curve: Adding more low-drop cards")

        # Replace some high-cost cards with low-cost ones
        high_cost_cards = []
        for card, qty in deck.items():
            card_info = card_dict.get(card)
            if not card_info:
                continue

            if 'Land' in card_info.get('types', []):
                continue

            mana_value = card_info.get('mana_value', 0)
            if mana_value >= 5:
                high_cost_cards.extend([card] * qty)

        # Shuffle to randomize selections
        random.shuffle(high_cost_cards)

        # Replace up to 5 high-cost cards
        replacements = min(5, len(high_cost_cards))

        for i in range(replacements):
            if i < len(high_cost_cards):
                card_to_replace = high_cost_cards[i]

                # Find low-cost replacements
                low_cost_cards = search_cards_by_criteria(
                    "mana value 1 or mana value 2",
                    commander_identity,
                    exclude_cards=exclude_list
                )

                if low_cost_cards:
                    # Remove high-cost card
                    if deck[card_to_replace] > 1:
                        deck[card_to_replace] -= 1
                    else:
                        del deck[card_to_replace]

                    # Add low-cost card
                    replacement = low_cost_cards[0]
                    if replacement in deck:
                        deck[replacement] += 1
                    else:
                        deck[replacement] = 1
                    exclude_list.append(replacement)

    if "top_heavy" in issues:
        # Replace some high-cost cards with mid-range ones
        logging.info("Fixing mana curve: Reducing top-heavy cards")

        high_cost_cards = []
        for card, qty in deck.items():
            card_info = card_dict.get(card)
            if not card_info:
                continue

            if 'Land' in card_info.get('types', []):
                continue

            mana_value = card_info.get('mana_value', 0)
            if mana_value >= 7:
                high_cost_cards.extend([card] * qty)

        # Replace up to half of the high-cost cards
        replacements = min(len(high_cost_cards) // 2 + 1, len(high_cost_cards))

        for i in range(replacements):
            if i < len(high_cost_cards):
                card_to_replace = high_cost_cards[i]

                # Find mid-range replacements
                mid_cost_cards = search_cards_by_criteria(
                    "mana value 3 or mana value 4 or mana value 5",
                    commander_identity,
                    exclude_cards=exclude_list
                )

                if mid_cost_cards:
                    # Remove high-cost card
                    if deck[card_to_replace] > 1:
                        deck[card_to_replace] -= 1
                    else:
                        del deck[card_to_replace]

                    # Add mid-cost card
                    replacement = mid_cost_cards[0]
                    if replacement in deck:
                        deck[replacement] += 1
                    else:
                        deck[replacement] = 1
                    exclude_list.append(replacement)

    # Fix gaps in the curve
    for issue in issues:
        if issue.startswith("gap_at_"):
            mana_value = int(issue.split("_")[-1])
            logging.info(f"Fixing mana curve: Adding cards at mana value {mana_value}")

            # Find cards with the missing mana value
            gap_cards = search_cards_by_criteria(
                f"mana value {mana_value}",
                commander_identity,
                exclude_cards=exclude_list
            )

            if gap_cards:
                # Find a card to replace
                replaced = False
                for card, qty in list(deck.items()):
                    card_info = card_dict.get(card)
                    if not card_info:
                        continue

                    if 'Land' in card_info.get('types', []):
                        continue

                    existing_mv = card_info.get('mana_value', 0)
                    if (existing_mv >= 6 or existing_mv <= 1) and card not in exclude_list:
                        # Replace this card
                        if qty > 1:
                            deck[card] -= 1
                        else:
                            del deck[card]

                        # Add gap-filling card
                        replacement = gap_cards[0]
                        if replacement in deck:
                            deck[replacement] += 1
                        else:
                            deck[replacement] = 1
                        exclude_list.append(replacement)
                        replaced = True
                        break

                # If we couldn't find a card to replace, just add the gap card
                if not replaced and sum(deck.values()) < 99:
                    replacement = gap_cards[0]
                    if replacement in deck:
                        deck[replacement] += 1
                    else:
                        deck[replacement] = 1
                    exclude_list.append(replacement)

    return deck

def count_card_types(deck):
    """Count cards by type in the deck"""
    type_counts = {'Land': 0, 'Creature': 0, 'Artifact': 0,
                  'Enchantment': 0, 'Planeswalker': 0,
                  'Instant': 0, 'Sorcery': 0, 'Other': 0}

    for card, qty in deck.items():
        if card in BASIC_LANDS.values():
            type_counts['Land'] += qty
            continue

        card_info = card_dict.get(card)
        if not card_info:
            type_counts['Other'] += qty
            continue

        types = card_info.get('types', [])

        # Count by primary type (use first match in hierarchy)
        if 'Land' in types:
            type_counts['Land'] += qty
        elif 'Creature' in types:
            type_counts['Creature'] += qty
        elif 'Artifact' in types:
            type_counts['Artifact'] += qty
        elif 'Enchantment' in types:
            type_counts['Enchantment'] += qty
        elif 'Planeswalker' in types:
            type_counts['Planeswalker'] += qty
        elif 'Instant' in types:
            type_counts['Instant'] += qty
        elif 'Sorcery' in types:
            type_counts['Sorcery'] += qty
        else:
            type_counts['Other'] += qty

    return type_counts

def determine_win_condition(commander, tokenizer, model, commander_identity):
    """Use the language model to determine appropriate win conditions for the commander"""
    prompt = f"""Analyze {commander} for Commander format and suggest the top 3 win conditions that synergize with this commander.
For each win condition:
1. Name the strategy (e.g., "Storm", "Combo", "Aggro", etc.)
2. Describe how it works with {commander}
3. List 5-10 key cards that enable this win condition within the {', '.join(commander_identity) if commander_identity else 'colorless'} color identity
4. Rate its power level from 1-10

Format your response with clear section headers for each win condition and bullet points for key cards.
"""

    inputs = tokenizer(prompt, return_tensors='pt')
    outputs = model.generate(
        **inputs,
        max_length=1500,
        temperature=0.8,  # Slightly higher temperature for more diverse responses
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    win_condition_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Define some pre-set win conditions if we can't parse the model output
    preset_win_conditions = [
        {
            'name': 'Storm Combo',
            'description': 'Cast multiple cheap spells in a turn to generate mana with Birgi, then finish with a big payoff spell.',
            'key_cards': ['Grapeshot', 'Mana Geyser', 'Seething Song', 'Jeska\'s Will', 'Runaway Steam-Kin', 'Grinning Ignus'],
            'power_level': 8
        },
        {
            'name': 'Wheel Effects',
            'description': 'Use Harnfel\'s "discard to draw" ability with wheel effects to cycle through your deck quickly.',
            'key_cards': ['Wheel of Fortune', 'Reforge the Soul', 'Magus of the Wheel', 'Past in Flames', 'Underworld Breach'],
            'power_level': 7
        },
        {
            'name': 'Artifact Combo',
            'description': 'Use cost reducers and Birgi\'s mana generation to chain artifact casts together.',
            'key_cards': ['Grinning Ignus', 'Sensei\'s Divining Top', 'Aetherflux Reservoir', 'Helm of Awakening', 'Skullclamp'],
            'power_level': 7
        },
        {
            'name': 'Red Aggro',
            'description': 'Cast numerous cheap red creatures and use Birgi\'s mana to overwhelm opponents.',
            'key_cards': ['Runaway Steam-Kin', 'Monastery Swiftspear', 'Lightning Bolt', 'Light Up the Stage', 'Reckless Impulse'],
            'power_level': 6
        }
    ]

    # Attempt to parse the model's output
    try:
        # Parse the win condition text to extract structured data
        win_conditions = []
        sections = re.split(r'(?:\n\n|\n#+\s*|\n\d+\.?\s*)(Win Condition|Strategy|Approach)[:\s-]+', win_condition_text)

        # If we have sections after splitting
        if len(sections) > 1:
            for i in range(1, len(sections), 2):
                if i+1 < len(sections):
                    section_header = sections[i].strip()
                    section_content = sections[i+1].strip()

                    # Extract strategy name from header or first line
                    strategy_name = section_header
                    if not strategy_name:
                        first_line = section_content.split('\n')[0].strip()
                        strategy_name = first_line

                    # Extract key cards - look for bullet points, numbers, or capital letters at line start
                    key_cards = []
                    description = ""
                    power_level = 0

                    for line in section_content.split('\n'):
                        line = line.strip()

                        # Look for power level rating
                        if re.search(r'(?:power|rating|score).*?(\d+)\s*(/|out of)\s*10', line.lower()):
                            match = re.search(r'(?:power|rating|score).*?(\d+)\s*(/|out of)\s*10', line.lower())
                            if match:
                                power_level = int(match.group(1))

                        # If line starts with a bullet, number, or capital letter, it might be a card
                        elif line.startswith('-') or line.startswith('*') or re.match(r'^\d+\.', line) or re.match(r'^[A-Z]', line):
                            # Clean up formatting
                            card = re.sub(r'^[-*•\d.)\]]+\s*', '', line)

                            # Remove explanations after the card name
                            if ' - ' in card:
                                card = card.split(' - ')[0].strip()
                            elif ' – ' in card:
                                card = card.split(' – ')[0].strip()
                            elif ' — ' in card:
                                card = card.split(' — ')[0].strip()
                            elif ': ' in card:
                                card = card.split(': ')[0].strip()

                            # Remove brackets if present
                            card = re.sub(r'[\[\]]', '', card)

                            # Validate card exists in database
                            if card in card_dict:
                                key_cards.append(card)
                        else:
                            # If not a card or power level, it's part of the description
                            description += line + " "

                    # Only add if we found some key cards
                    if key_cards:
                        win_conditions.append({
                            'name': strategy_name,
                            'description': description.strip(),
                            'key_cards': key_cards,
                            'power_level': power_level
                        })

        # If we failed to extract win conditions, use preset ones
        if not win_conditions:
            # Filter preset conditions to ensure cards exist in commander's color identity
            valid_presets = []
            for condition in preset_win_conditions:
                valid_cards = [card for card in condition['key_cards']
                              if card in card_dict and
                              all(c in commander_identity for c in card_dict[card].get('color_identity', []))]

                if valid_cards:
                    valid_condition = condition.copy()
                    valid_condition['key_cards'] = valid_cards
                    valid_presets.append(valid_condition)

            win_conditions = valid_presets

        # If we still don't have any, create a generic one
        if not win_conditions:
            logging.warning("Using default generic win condition")
            win_conditions = [{
                'name': 'Generic Synergy',
                'description': f'A balanced approach utilizing {commander}\'s abilities.',
                'key_cards': [],
                'power_level': 5
            }]

        return win_conditions, win_condition_text

    except Exception as e:
        logging.warning(f"Error parsing win conditions: {str(e)}")

        # Fall back to preset win conditions
        valid_presets = []
        for condition in preset_win_conditions:
            valid_cards = [card for card in condition['key_cards']
                          if card in card_dict and
                          all(c in commander_identity for c in card_dict[card].get('color_identity', []))]

            if valid_cards:
                valid_condition = condition.copy()
                valid_condition['key_cards'] = valid_cards
                valid_presets.append(valid_condition)

        if valid_presets:
            return valid_presets, win_condition_text
        else:
            return [{
                'name': 'Generic Synergy',
                'description': f'A balanced approach utilizing {commander}\'s abilities.',
                'key_cards': [],
                'power_level': 5
            }], win_condition_text

def generate_random_deck(commander, commander_identity):
    """Generate a random but coherent deck for the given commander"""
    logging.info(f"Generating random deck for {commander}")

    # Start with an empty deck
    deck = {}

    # Add commander if not already in deck
    if commander not in deck:
        deck[commander] = 1

    # Define card categories and their target percentages
    categories = {
        'Ramp': 0.10,     # 10% ramp effects
        'Draw': 0.10,     # 10% card draw
        'Removal': 0.10,  # 10% removal
        'Wrath': 0.05,    # 5% board wipes
        'Creatures': 0.25, # 25% creatures
        'Synergy': 0.15,  # 15% synergy pieces
        'Lands': 0.38     # 38% lands (including basic lands)
    }

    # Calculate target counts
    total_cards = 99  # 99 cards plus commander = 100
    card_counts = {category: int(percentage * total_cards) for category, percentage in categories.items()}

    # Adjust to ensure we hit exactly 99 cards
    remaining = total_cards - sum(card_counts.values())
    if remaining > 0:
        card_counts['Creatures'] += remaining

    # Add cards by category
    for category, count in card_counts.items():
        if category == 'Lands':
            # Handle lands separately to ensure proper basic land distribution
            continue

        logging.info(f"Adding {count} {category} cards")

        # Generate query based on category
        if category == 'Ramp':
            queries = ['type:artifact text:add mana', 'text:"add mana"', 'text:ritual']
        elif category == 'Draw':
            queries = ['text:"draw a card"', 'text:"draw cards"']
        elif category == 'Removal':
            queries = ['text:destroy target', 'text:exile target', 'text:damage to target']
        elif category == 'Wrath':
            queries = ['text:"destroy all"', 'text:"exile all"', 'text:"damage to all"']
        elif category == 'Creatures':
            queries = ['type:creature power>2', 'type:creature toughness>2', 'type:creature text:when']
        elif category == 'Synergy':
            # Generate synergy queries based on commander text
            queries = generate_synergy_queries(commander)
            if not queries:
                queries = ['type:instant', 'type:sorcery']

        # Add cards for each query
        added = 0
        exclude_list = list(deck.keys())

        for query in queries:
            if added >= count:
                break

            results = search_cards_by_criteria(query, commander_identity, exclude_cards=exclude_list)
            random.shuffle(results)  # Randomize the results

            # Add a portion of results
            to_add = min(len(results), count - added)
            for i in range(to_add):
                if i < len(results):
                    card_name = results[i]
                    deck[card_name] = 1
                    exclude_list.append(card_name)
                    added += 1

    # Add lands to reach target
    lands_to_add = card_counts['Lands']

    # First add some non-basic lands if available
    nonbasic_query = 'type:land -type:basic'
    nonbasic_lands = search_cards_by_criteria(nonbasic_query, commander_identity, exclude_cards=list(deck.keys()), n=10)

    for land in nonbasic_lands:
        if lands_to_add <= 0:
            break
        deck[land] = 1
        lands_to_add -= 1

    # Fill the rest with basic lands
    if lands_to_add > 0:
        # Determine which basic lands to add based on color identity
        land_types = []
        for color in commander_identity:
            if color in BASIC_LANDS:
                land_types.append(BASIC_LANDS[color])

        # If no colors or colorless, add Wastes
        if not land_types:
            land_types = ['Wastes']

        # Add equal numbers of each basic land type
        per_land = lands_to_add // len(land_types)
        remainder = lands_to_add % len(land_types)

        for i, land in enumerate(land_types):
            qty = per_land + (1 if i < remainder else 0)
            if land in deck:
                deck[land] += qty
            else:
                deck[land] = qty

    return deck

def main():
    cards = load_card_data()
    setup_retriever(cards)

    model_dir = '/content/drive/MyDrive/MTGModel/real_deck_training/final-model'
    tokenizer, model = load_language_model(model_dir)

    commander = input("Enter commander name: ")
    commander_identity = get_commander_identity(commander)

    if not commander_identity:
        logging.error(f"Could not determine color identity for {commander}")
        similar_commanders = search_similar_cards(commander, n=5)
        print(f"Did you mean one of these? {', '.join(similar_commanders)}")
        commander = input("Try entering commander name again: ")
        commander_identity = get_commander_identity(commander)
        if not commander_identity:
            print("Still couldn't find commander. Using default color identity (colorless).")
            commander_identity = []

    print(f"\nAnalyzing {commander} (Color Identity: {', '.join(commander_identity)})")

    # First, determine possible win conditions
    win_conditions, win_condition_text = determine_win_condition(commander, tokenizer, model, commander_identity)

    # Display win conditions to user
    print("\n=== Possible Win Conditions ===")
    print(win_condition_text)
    print("\n===========================")

    # Let user select a win condition or choose random
    if len(win_conditions) > 1:
        print("\nSelect a win condition to build around:")
        for i, wc in enumerate(win_conditions):
            key_cards_str = ", ".join(wc.get('key_cards', [])[:3])
            power_level = wc.get('power_level', 0)
            power_str = f" (Power: {power_level}/10)" if power_level > 0 else ""
            suggested_bracket = get_appropriate_bracket_for_win_condition(wc)
            bracket_str = f" - Bracket: {suggested_bracket} ({COMMANDER_BRACKETS[suggested_bracket]['name']})" if suggested_bracket > 0 else ""
            print(f"{i+1}. {wc['name']}{power_str}{bracket_str} - Key cards: {key_cards_str}...")

        print(f"{len(win_conditions)+1}. Random - Let me pick a random win condition")
        print(f"{len(win_conditions)+2}. Fully Random Deck - Generate a completely random deck")

        selection = input(f"Enter 1-{len(win_conditions)+2} [1]: ").strip()
        if not selection:
            selected_idx = 0
        else:
            try:
                selected_idx = int(selection) - 1
                if selected_idx < 0 or selected_idx > len(win_conditions) + 1:
                    selected_idx = 0
            except ValueError:
                selected_idx = 0

        # Handle random selection
        if selected_idx == len(win_conditions):
            selected_idx = random.randint(0, len(win_conditions) - 1)
            print(f"Randomly selected: {win_conditions[selected_idx]['name']}")

        # Handle fully random deck
        elif selected_idx == len(win_conditions) + 1:
            print(f"\nGenerating random deck for {commander}")
            random_deck = generate_random_deck(commander, commander_identity)

            # Ask for bracket selection
            print("\nSelect a power level bracket for the deck:")
            for i, (bracket_num, bracket_info) in enumerate(COMMANDER_BRACKETS.items()):
                print(f"{bracket_num}. {bracket_info['name']} - {bracket_info['description']}")

            bracket_selection = input(f"Enter bracket (1-5) [2]: ").strip()
            try:
                selected_bracket = int(bracket_selection)
                if selected_bracket < 1 or selected_bracket > 5:
                    selected_bracket = 2  # Default to Core
            except ValueError:
                selected_bracket = 2  # Default to Core

            # Filter deck for bracket compatibility
            filtered_deck, removed_cards = filter_deck_for_bracket(random_deck, selected_bracket)

            # Replace removed cards
            if removed_cards:
                print(f"Removed {len(removed_cards)} cards that don't comply with bracket {selected_bracket} restrictions.")
                filtered_deck = replace_removed_cards(filtered_deck, removed_cards, commander_identity, selected_bracket)

            # Print final deck
            type_counts = count_card_types(filtered_deck)
            print(f"\nRandom Deck for {commander} (Color Identity: {', '.join(commander_identity)}):")
            print(f"Bracket: {selected_bracket} - {COMMANDER_BRACKETS[selected_bracket]['name']}")
            print(f"Total cards: {sum(filtered_deck.values())}")
            print(f"Card type distribution: {type_counts}")

            print_formatted_deck(filtered_deck, commander)
            return
    else:
        selected_idx = 0

    # Use selected win condition
    selected_win_condition = win_conditions[selected_idx]

    # Determine appropriate bracket based on win condition
    suggested_bracket = get_appropriate_bracket_for_win_condition(selected_win_condition)

    # Ask for bracket selection
    print("\nSelect a power level bracket for the deck:")
    for i, (bracket_num, bracket_info) in enumerate(COMMANDER_BRACKETS.items()):
        if bracket_num == suggested_bracket:
            print(f"{bracket_num}. {bracket_info['name']} (RECOMMENDED) - {bracket_info['description']}")
        else:
            print(f"{bracket_num}. {bracket_info['name']} - {bracket_info['description']}")

    bracket_selection = input(f"Enter bracket (1-5) [{suggested_bracket}]: ").strip()
    try:
        selected_bracket = int(bracket_selection)
        if selected_bracket < 1 or selected_bracket > 5:
            selected_bracket = suggested_bracket
    except ValueError:
        selected_bracket = suggested_bracket

    print(f"\nGenerating deck for {commander} with '{selected_win_condition['name']}' win condition")
    print(f"Using bracket {selected_bracket} - {COMMANDER_BRACKETS[selected_bracket]['name']}")

    # Check if we should use a temperature boost for more variation
    use_variation = input("Would you like more variety in the generated deck? (y/n) [n]: ").strip().lower()
    temp_boost = 0.2 if use_variation == 'y' else 0.0

    # Generate deck with chosen win condition
    deck_text = generate_conditioned_deck(commander, tokenizer, model, selected_win_condition, commander_identity, temp_boost)
    initial_deck = extract_deck(deck_text)

    # Add key cards from win condition if not already in deck
    for card in selected_win_condition.get('key_cards', []):
        if card not in initial_deck and card != commander:
            card_info = card_dict.get(card)
            if card_info and all(c in commander_identity for c in card_info.get('color_identity', [])):
                initial_deck[card] = 1

    # Validate and filter cards
    valid_deck, invalid_cards = validate_deck_identity(initial_deck, commander_identity)

    # Complete deck to 100 cards
    final_deck = complete_deck(valid_deck, commander, commander_identity, selected_win_condition)

    # Filter deck for bracket compatibility
    filtered_deck, removed_cards = filter_deck_for_bracket(final_deck.copy(), selected_bracket)

    # Replace removed cards
    if removed_cards:
        print(f"Removed {len(removed_cards)} cards that don't comply with bracket {selected_bracket} restrictions:")
        for card, qty in removed_cards.items():
            print(f"  {qty}x {card}")
        filtered_deck = replace_removed_cards(filtered_deck, removed_cards, commander_identity, selected_bracket)

    # Add commander as 1x if not already in deck
    if commander not in filtered_deck:
        filtered_deck[commander] = 1

    # Print final deck
    type_counts = count_card_types(filtered_deck)
    print(f"\nFinal Deck for {commander} (Color Identity: {', '.join(commander_identity)}):")
    print(f"Strategy: {selected_win_condition['name']}")
    print(f"Bracket: {selected_bracket} - {COMMANDER_BRACKETS[selected_bracket]['name']}")
    print(f"Total cards: {sum(filtered_deck.values())}")
    print(f"Card type distribution: {type_counts}")

    print_formatted_deck(filtered_deck, commander)

def print_formatted_deck(deck, commander):
    """Print the deck in a nicely formatted way"""
    print("\n--- Commander ---")
    print(f"1x {commander}")

    print("\n--- Lands ---")
    for card, qty in sorted(deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if 'Land' in types or card in BASIC_LANDS.values():
            print(f"{qty}x {card}")

    print("\n--- Creatures ---")
    for card, qty in sorted(deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if 'Creature' in types and 'Land' not in types:
            print(f"{qty}x {card}")

    print("\n--- Artifacts ---")
    for card, qty in sorted(deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if 'Artifact' in types and 'Creature' not in types and 'Land' not in types:
            print(f"{qty}x {card}")

    print("\n--- Enchantments ---")
    for card, qty in sorted(deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if 'Enchantment' in types and 'Creature' not in types and 'Land' not in types:
            print(f"{qty}x {card}")

    print("\n--- Planeswalkers ---")
    for card, qty in sorted(deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if 'Planeswalker' in types:
            print(f"{qty}x {card}")

    print("\n--- Instants and Sorceries ---")
    for card, qty in sorted(deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if ('Instant' in types or 'Sorcery' in types) and 'Creature' not in types and 'Land' not in types:
            print(f"{qty}x {card}")

def generate_conditioned_deck(commander, tokenizer, model, win_condition, commander_identity, temp_boost=0):
    """Generate a deck with a specific win condition in mind"""
    # Prepare a prompt that specifies the win condition
    key_cards_str = ", ".join(win_condition.get('key_cards', [])[:5])  # Use top 5 key cards for prompt
    description = win_condition.get('description', f"A {win_condition['name']} strategy")

    prompt = f"""Generate a Commander deck for {commander} focusing on the "{win_condition['name']}" win condition.
Description: {description}
Include the following key cards if within color identity: {key_cards_str}.
Make sure the deck is cohesive and all cards work toward the win condition.
Format as a list with quantities (e.g., "1x Card Name").
Include exactly 99 cards (excluding the commander)."""

    inputs = tokenizer(prompt, return_tensors='pt')
    outputs = model.generate(
        **inputs,
        max_length=2000,
        temperature=0.65 + temp_boost,  # Adjust temperature based on desired variation
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    conditioned_deck_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return conditioned_deck_text

    # Print final deck with type categorization
    type_counts = count_card_types(final_deck)
    print(f"\nFinal Deck for {commander} (Color Identity: {', '.join(commander_identity)}):")
    print(f"Total cards: {sum(final_deck.values())}")
    print(f"Card type distribution: {type_counts}")
    print("\n--- Commander ---")
    print(f"1x {commander}")
    print("\n--- Lands ---")
    for card, qty in sorted(final_deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if 'Land' in types or card in BASIC_LANDS.values():
            print(f"{qty}x {card}")

    print("\n--- Creatures ---")
    for card, qty in sorted(final_deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if 'Creature' in types and 'Land' not in types:
            print(f"{qty}x {card}")

    print("\n--- Spells ---")
    for card, qty in sorted(final_deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if not ('Land' in types or 'Creature' in types or card in BASIC_LANDS.values()):
            print(f"{qty}x {card}")

if __name__ == '__main__':
    main()

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Enter commander name: Kykar, Wind's Fury

Analyzing Kykar, Wind's Fury (Color Identity: R, U, W)

=== Possible Win Conditions ===
Analyze Kykar, Wind's Fury for Commander format and suggest the top 3 win conditions that synergize with this commander.
For each win condition:
1. Name the strategy (e.g., "Storm", "Combo", "Aggro", etc.)
2. Describe how it works with Kykar, Wind's Fury
3. List 5-10 key cards that enable this win condition within the R, U, W color identity
4. Rate its power level from 1-10

Format your response with clear section headers for each win condition and bullet points for key cards.
1x Fortifying Draught
1x Frantic Search
1x Gandalf's Sanction
1x Gift of Immortality
1x Goblin Test Pilot
1x Havengul Laboratory
1x Hop to It
1x Hurl into History
1x Ignite the Future
1x Ishai, Ojutai Dragonspeaker
35x Island
1x Jace's Sanctum
1x Jin-Gitaxias, Progress Tyrant
1x Karametra's Blessing
1x Kykar, Wind's Fury
1x Leveler
1x Magus of the Wheel
1x Memory Deluge
1x Minn, Wily I

In [ ]:
import os
import json
import logging
import requests
import torch
import faiss
import numpy as np
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
import random

logging.basicConfig(level=logging.INFO)

card_dict = {}
faiss_index, embed_model, card_texts, card_objects = None, None, [], []


# Define Commander brackets for power level targeting
COMMANDER_BRACKETS = {
    1: {
        'name': 'Exhibition',
        'description': 'Heavily themed decks where winning is not the primary goal. Focus on showing off a concept or project.',
        'restrictions': [
            'No Mass Land Denial',
            'No Game Changer Cards',
            'No 2-card Combos',
            'Extra Turns spells should be thematic and not intended to be chained or looped',
            'Tutors should be sparse and specific'
        ],
        'power_level': (1, 3)  # Power range 1-3
    },
    2: {
        'name': 'Core',
        'description': 'The bulk of casual decks and modern precons. Draw power from synergy rather than card quality.',
        'restrictions': [
            'No Mass Land Denial',
            'No Game Changer Cards',
            'No 2-card Combos',
            'Extra Turns spells should be minimal and not intended to be chained or looped',
            'Tutors should be sparse and specific'
        ],
        'power_level': (3, 5)  # Power range 3-5
    },
    3: {
        'name': 'Upgraded',
        'description': 'Intentionally tuned and upgraded decks with improved power level. Theme and flavor take a back seat.',
        'restrictions': [
            'No Mass Land Denial',
            'Up to 3 Game Changer Cards',
            'No 2-card Combos before turn 6',
            'Some extra turns and tutors expected (but not looped)'
        ],
        'power_level': (5, 7)  # Power range 5-7
    },
    4: {
        'name': 'Optimized',
        'description': 'High-powered decks using any cards and strategies, though not meta-focused or tournament driven.',
        'restrictions': [
            'None (other than the banned list)'
        ],
        'power_level': (7, 9)  # Power range 7-9
    },
    5: {
        'name': 'cEDH',
        'description': 'Competitive EDH decks designed for tournament play, making choices dependent on the competitive meta.',
        'restrictions': [
            'None (other than the banned list)'
        ],
        'power_level': (9, 10)  # Power range 9-10
    }
}

# Lists of cards that shouldn't appear in lower bracket decks
GAME_CHANGER_CARDS = [
    "Cyclonic Rift", "Expropriate", "Craterhoof Behemoth", "Rise of the Dark Realms",
    "Torment of Hailfire", "Omniscience", "Zacama, Primal Calamity", "Jin-Gitaxias, Core Augur",
    "Vorinclex, Voice of Hunger", "Elesh Norn, Grand Cenobite", "Ulamog, the Ceaseless Hunger",
    "Kozilek, Butcher of Truth", "Smothering Tithe", "Rhystic Study", "Mystic Remora",
    "Necropotence", "Sylvan Library", "Demonic Tutor", "Vampiric Tutor", "Worldly Tutor",
    "Time Spiral", "Force of Will", "Mana Drain", "Fierce Guardianship", "Toxic Deluge"
]

MASS_LAND_DENIAL = [
    "Armageddon", "Ravages of War", "Catastrophe", "Wildfire", "Burning of Xinye",
    "Ruination", "Death Cloud", "Jokulhaups", "Obliterate", "Decree of Annihilation",
    "Fall of the Thran", "Impending Disaster", "Natural Balance", "Winter Orb",
    "Static Orb", "Rising Waters", "Stasis", "Blood Moon", "Magus of the Moon"
]

EXTRA_TURN_SPELLS = [
    "Time Warp", "Temporal Manipulation", "Capture of Jingzhou", "Time Stretch",
    "Nexus of Fate", "Temporal Mastery", "Walk the Aeons", "Karn's Temporal Sundering",
    "Part the Waterveil", "Temporal Trespass", "Alrund's Epiphany", "Expropriate"
]

TUTOR_CARDS = [
    "Demonic Tutor", "Vampiric Tutor", "Worldly Tutor", "Mystical Tutor", "Enlightened Tutor",
    "Imperial Seal", "Diabolic Tutor", "Grim Tutor", "Personal Tutor", "Sylvan Tutor",
    "Demonic Consultation", "Tainted Pact", "Scheming Symmetry", "Cruel Tutor",
    "Diabolic Intent", "Gamble", "Chord of Calling", "Green Sun's Zenith", "Tooth and Nail",
    "Eladamri's Call", "Idyllic Tutor", "Beseech the Queen", "Final Parting"
]

COMBO_CARDS = {
    # Format: card_name: [combo_partner1, combo_partner2, ...]
    "Thassa's Oracle": ["Demonic Consultation", "Tainted Pact"],
    "Laboratory Maniac": ["Demonic Consultation", "Tainted Pact"],
    "Jace, Wielder of Mysteries": ["Demonic Consultation", "Tainted Pact"],
    "Isochron Scepter": ["Dramatic Reversal"],
    "Dramatic Reversal": ["Isochron Scepter"],
    "Splinter Twin": ["Deceiver Exarch", "Pestermite", "Zealous Conscripts"],
    "Kiki-Jiki, Mirror Breaker": ["Deceiver Exarch", "Pestermite", "Zealous Conscripts"],
    "Mikaeus, the Unhallowed": ["Triskelion", "Walking Ballista"],
    "Triskelion": ["Mikaeus, the Unhallowed"],
    "Walking Ballista": ["Mikaeus, the Unhallowed", "Heliod, Sun-Crowned"],
    "Heliod, Sun-Crowned": ["Walking Ballista"],
    "Sanguine Bond": ["Exquisite Blood"],
    "Exquisite Blood": ["Sanguine Bond"],
    "Painters Servant": ["Grindstone"],
    "Grindstone": ["Painters Servant"],
    "Doomsday": ["Thassa's Oracle", "Laboratory Maniac"],
    "Food Chain": ["Eternal Scourge", "Misthollow Griffin", "Squee, the Immortal"],
    "Demonic Consultation": ["Thassa's Oracle", "Laboratory Maniac", "Jace, Wielder of Mysteries"],
    "Tainted Pact": ["Thassa's Oracle", "Laboratory Maniac", "Jace, Wielder of Mysteries"],
    "Earthcraft": ["Squirrel Nest"],
    "Squirrel Nest": ["Earthcraft"]
}

def check_card_bracket_compatibility(card_name, bracket):
    """Check if a card is compatible with the bracket's restrictions"""
    if card_name not in card_dict:
        return True  # If we don't have card info, default to allowing it

    # Always allow the card in brackets 4 and 5 (no restrictions)
    if bracket >= 4:
        return True

    # Check for mass land denial
    if bracket <= 3 and card_name in MASS_LAND_DENIAL:
        return False

    # Check for game changers in lower brackets
    if bracket <= 2 and card_name in GAME_CHANGER_CARDS:
        return False

    # Extra turns limitations
    if bracket <= 2 and card_name in EXTRA_TURN_SPELLS:
        return False

    # Tutor limitations (stricter for lower brackets)
    if bracket <= 2 and card_name in TUTOR_CARDS:
        # For bracket 2, allow some specific tutors but limit quantity
        return False

    # Check for combo pieces
    if bracket <= 3 and card_name in COMBO_CARDS:
        return False

    return True

def filter_deck_for_bracket(deck, bracket):
    """Filter a deck to comply with the chosen bracket's restrictions"""
    filtered_deck = {}
    removed_cards = {}

    # If bracket is 4 or 5, no restrictions apply
    if bracket >= 4:
        return deck, {}

    # Count restricted card types
    game_changers_count = 0
    tutor_count = 0
    extra_turns_count = 0
    combo_pieces = {}

    # First pass: identify all restricted cards
    for card, qty in deck.items():
        # Check for game changers
        if card in GAME_CHANGER_CARDS:
            game_changers_count += qty

        # Check for tutors
        if card in TUTOR_CARDS:
            tutor_count += qty

        # Check for extra turns
        if card in EXTRA_TURN_SPELLS:
            extra_turns_count += qty

        # Check for combo pieces
        if card in COMBO_CARDS:
            combo_partners = COMBO_CARDS[card]
            for partner in combo_partners:
                if partner in deck:
                    if card not in combo_pieces:
                        combo_pieces[card] = []
                    combo_pieces[card].append(partner)

    # Second pass: apply bracket-specific filters
    for card, qty in deck.items():
        # Exhibition and Core brackets (1 & 2)
        if bracket <= 2:
            # No game changers
            if card in GAME_CHANGER_CARDS:
                removed_cards[card] = qty
                continue

            # No mass land denial
            if card in MASS_LAND_DENIAL:
                removed_cards[card] = qty
                continue

            # No extra turns
            if card in EXTRA_TURN_SPELLS:
                removed_cards[card] = qty
                continue

            # Limited tutors (allow just 1-2 specific ones for bracket 2)
            if card in TUTOR_CARDS:
                if bracket == 1 or (bracket == 2 and tutor_count > 2):
                    removed_cards[card] = qty
                    continue

            # No 2-card combos
            if card in combo_pieces:
                removed_cards[card] = qty
                continue

        # Upgraded bracket (3)
        elif bracket == 3:
            # No mass land denial
            if card in MASS_LAND_DENIAL:
                removed_cards[card] = qty
                continue

            # Limit game changers to 3
            if card in GAME_CHANGER_CARDS and game_changers_count > 3:
                removed_cards[card] = qty
                game_changers_count -= qty
                continue

        # If we made it here, the card is allowed
        filtered_deck[card] = qty

    return filtered_deck, removed_cards

def replace_removed_cards(deck, removed_cards, commander_identity, bracket):
    """Replace cards that were removed due to bracket restrictions"""
    if not removed_cards:
        return deck

    # Calculate how many cards we need to replace
    cards_to_add = sum(removed_cards.values())
    logging.info(f"Replacing {cards_to_add} cards removed due to bracket {bracket} restrictions")

    # Get list of cards already in the deck
    exclude_list = list(deck.keys()) + list(removed_cards.keys())

    # Generate replacement queries based on removed card types
    replacement_queries = []

    # Check what types of cards were removed and generate appropriate replacement queries
    has_removed_tutors = any(card in TUTOR_CARDS for card in removed_cards)
    has_removed_extra_turns = any(card in EXTRA_TURN_SPELLS for card in removed_cards)
    has_removed_land_denial = any(card in MASS_LAND_DENIAL for card in removed_cards)
    has_removed_game_changers = any(card in GAME_CHANGER_CARDS for card in removed_cards)
    has_removed_combos = any(card in COMBO_CARDS for card in removed_cards)

    # Add replacement queries based on what was removed
    if has_removed_tutors:
        replacement_queries.extend(["card draw", "card advantage"])

    if has_removed_extra_turns:
        replacement_queries.extend(["additional combat", "untap effects"])

    if has_removed_land_denial:
        replacement_queries.extend(["targeted removal", "single target land destruction"])

    if has_removed_game_changers:
        replacement_queries.extend(["value engine", "gradual advantage"])

    if has_removed_combos:
        replacement_queries.extend(["synergy pieces", "value creatures"])

    # If we don't have specific replacements, use general good stuff
    if not replacement_queries:
        replacement_queries = ["card draw", "removal", "ramp", "board wipe", "value creature"]

    # Add cards by cycling through queries
    added_count = 0
    query_index = 0

    while added_count < cards_to_add:
        query = replacement_queries[query_index % len(replacement_queries)]
        query_index += 1

        # Search for cards matching the query
        results = search_cards_by_criteria(query, commander_identity, exclude_cards=exclude_list, n=20)

        # Filter results for bracket compatibility
        bracket_compatible = [card for card in results if check_card_bracket_compatibility(card, bracket)]

        if bracket_compatible:
            # Pick one randomly
            card_to_add = random.choice(bracket_compatible)

            # Add to deck
            if card_to_add in deck:
                deck[card_to_add] += 1
            else:
                deck[card_to_add] = 1

            exclude_list.append(card_to_add)
            added_count += 1

            # If we're struggling to find cards, relax the search a bit
            if query_index > 10 * len(replacement_queries):
                # Try to add basic lands as a last resort
                for color in commander_identity:
                    if color in BASIC_LANDS:
                        land = BASIC_LANDS[color]
                        if land in deck:
                            deck[land] += 1
                        else:
                            deck[land] = 1
                        added_count += 1
                        if added_count >= cards_to_add:
                            break
                break

    return deck

def get_appropriate_bracket_for_win_condition(win_condition):
    """Determine the appropriate bracket for a win condition based on its power level"""
    power_level = win_condition.get('power_level', 5)  # Default to mid-power if not specified

    if power_level <= 3:
        return 1  # Exhibition
    elif power_level <= 5:
        return 2  # Core
    elif power_level <= 7:
        return 3  # Upgraded
    elif power_level <= 9:
        return 4  # Optimized
    else:
        return 5  # cEDH
# MTG Deck Generator with Comprehensive Rules, Dual-Faced Card Support, and Validation
# Enhanced for Commander Deck Building Guidelines
# Run this in Google Colab


COMMANDER_BANNED_CARDS = ["Ancestral Recall", "Balance", "Biorhythm", "Black Lotus", "Braids, Cabal Minion", "Channel", "Chaos Orb",
"Coalition Victory", "Dockside Extortionist", "Emrakul, the Aeons Torn", "Erayo, Soratami Ascendant", "Falling Star",
"Fastbond", "Flash", "Gifts Ungiven", "Golos, Tireless Pilgrim", "Griselbrand", "Hullbreacher", "Iona, Shield of Emeria",
"Jeweled Lotus", "Karakas", "Leovold, Emissary of Trest", "Library of Alexandria", "Limited Resources", "Lutri, the Spellchaser",
"Mana Crypt", "Mox Emerald", "Mox Jet", "Mox Pearl", "Mox Ruby", "Mox Sapphire", "Nadu, Winged Wisdom", "Panoptic Mirror",
"Paradox Engine", "Primeval Titan", "Prophet of Kruphix", "Recurring Nightmare", "Rofellos, Llanowar Emissary", "Shahrazad",
"Sundering Titan", "Sway of the Stars", "Sylvan Primordial", "Time Vault", "Time Walk", "Tinker", "Tolarian Academy",
"Trade Secrets", "Upheaval", "Yawgmoth's Bargain"]

BASIC_LANDS = {
    'W': 'Plains',
    'U': 'Island',
    'B': 'Swamp',
    'R': 'Mountain',
    'G': 'Forest'
}

CARD_CATEGORIES = {
    'ramp': ['type:artifact text:add mana', 'type:land text:add mana', 'type:creature text:"add mana"'],
    'draw': ['text:"draw a card"', 'text:"draw cards"'],
    'removal': ['text:destroy', 'text:exile', 'text:"damage to"'],
    'wrath': ['text:"destroy all"', 'text:"exile all"', 'text:"damage to all"'],
    'synergy': []  # Will be populated based on commander
}

def load_card_data():
    global card_dict, card_objects
    bulk_data = requests.get("https://api.scryfall.com/bulk-data").json()
    oracle_url = next(item for item in bulk_data["data"] if item["name"] == "Oracle Cards")["download_uri"]
    cards = requests.get(oracle_url).json()
    card_objects = []
    for card in cards:
        if 'name' not in card or 'legalities' not in card:
            continue
        if card.get('legalities', {}).get('commander') in ['legal', 'restricted']:
            card_objects.append(card)
            card_dict[card['name']] = {
                'color_identity': card.get('color_identity', []),
                'type_line': card.get('type_line', ''),
                'oracle_text': card.get('oracle_text', ''),
                'mana_value': card.get('cmc', 0),
                'types': extract_types(card.get('type_line', ''))
            }
            if 'card_faces' in card:
                for face in card['card_faces']:
                    if 'name' in face and face['name'] != card['name']:
                        card_dict[face['name']] = {
                            'color_identity': card.get('color_identity', []),
                            'type_line': face.get('type_line', ''),
                            'oracle_text': face.get('oracle_text', ''),
                            'mana_value': card.get('cmc', 0),
                            'types': extract_types(face.get('type_line', ''))
                        }
    logging.info(f"Loaded {len(card_objects)} legal Commander cards")
    return card_objects

def extract_types(type_line):
    types = []
    if "Land" in type_line:
        types.append("Land")
    if "Creature" in type_line:
        types.append("Creature")
    if "Artifact" in type_line:
        types.append("Artifact")
    if "Enchantment" in type_line:
        types.append("Enchantment")
    if "Planeswalker" in type_line:
        types.append("Planeswalker")
    if "Instant" in type_line:
        types.append("Instant")
    if "Sorcery" in type_line:
        types.append("Sorcery")
    return types

def setup_retriever(cards):
    global faiss_index, embed_model, card_texts
    embed_model = SentenceTransformer('all-MiniLM-L6-v2')

    # Create more detailed card texts for better semantic search
    card_texts = []
    for c in cards:
        if 'name' not in c or 'type_line' not in c:
            continue

        # Extract key card properties for embedding context
        card_name = c['name']
        type_line = c['type_line']
        oracle_text = c.get('oracle_text', '')
        keywords = c.get('keywords', [])
        mana_cost = c.get('mana_cost', '')

        # Create a rich text representation
        text = f"{card_name}. Cost: {mana_cost}. Types: {type_line}. "

        if keywords:
            text += f"Keywords: {', '.join(keywords)}. "

        if oracle_text:
            text += f"Text: {oracle_text}"

        # Add card faces for dual-faced cards
        if 'card_faces' in c:
            face_texts = []
            for face in c['card_faces']:
                if 'name' in face and 'type_line' in face:
                    face_name = face['name']
                    face_type = face['type_line']
                    face_text = face.get('oracle_text', '')
                    face_cost = face.get('mana_cost', '')

                    face_desc = f"{face_name}. Cost: {face_cost}. Types: {face_type}. "
                    if face_text:
                        face_desc += f"Text: {face_text}"
                    face_texts.append(face_desc)

            if face_texts:
                text += f" Faces: {' | '.join(face_texts)}"

        card_texts.append(text)

    logging.info(f"Generating embeddings for {len(card_texts)} cards...")

    # Create embeddings with batched processing for memory efficiency
    batch_size = 256
    all_embeddings = []

    for i in range(0, len(card_texts), batch_size):
        batch = card_texts[i:i + batch_size]
        batch_embeddings = embed_model.encode(batch, show_progress_bar=True)
        all_embeddings.append(batch_embeddings)

    embeddings = np.vstack(all_embeddings).astype('float32')

    # Create and populate the FAISS index
    faiss_index = faiss.IndexFlatL2(embeddings.shape[1])
    faiss_index.add(embeddings)
    logging.info("Embeddings completed and indexed")

def load_language_model(model_dir):
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForCausalLM.from_pretrained(model_dir)
    return tokenizer, model

def get_commander_identity(commander_name):
    commander_info = card_dict.get(commander_name)
    if not commander_info:
        logging.warning(f"Commander '{commander_name}' not found in database")
        # Try to find similar commander names
        similar_names = search_similar_cards(commander_name, n=5)
        logging.info(f"Did you mean one of these? {', '.join(similar_names)}")
        return []
    return commander_info.get('color_identity', [])

def validate_deck_identity(deck, commander_identity):
    valid_deck, invalid_cards = {}, []
    for card, qty in deck.items():
        # Skip validation for basic lands that match commander color identity
        if card in BASIC_LANDS.values():
            if not commander_identity or any(color in commander_identity for color, land in BASIC_LANDS.items() if land == card):
                valid_deck[card] = qty
                continue

        card_info = card_dict.get(card)
        if not card_info:
            invalid_cards.append(f"{card} (not found)")
            continue

        card_identity = card_info.get('color_identity', [])

        # Check if card's color identity is a subset of commander's identity
        if all(c in commander_identity for c in card_identity) and card not in COMMANDER_BANNED_CARDS:
            valid_deck[card] = qty
        else:
            reason = "banned" if card in COMMANDER_BANNED_CARDS else "color identity mismatch"
            invalid_cards.append(f"{card} ({reason})")

    logging.info(f"Removed {len(invalid_cards)} invalid cards: {', '.join(invalid_cards)}")
    return valid_deck, invalid_cards

def search_similar_cards(query, n=10):
    if not faiss_index or not card_objects:
        return []

    query_embedding = embed_model.encode([query])
    distances, indices = faiss_index.search(query_embedding, n)
    return [card_objects[i]['name'] for i in indices[0] if i < len(card_objects)]

def search_cards_by_criteria(query, commander_identity, exclude_cards=None, n=30):
    """Search for cards matching query with specified color identity"""
    if not exclude_cards:
        exclude_cards = set()
    else:
        exclude_cards = set(exclude_cards)

    query_embedding = embed_model.encode([query])
    distances, indices = faiss_index.search(query_embedding, n*3)  # Get more results to filter

    results = []
    for i in indices[0]:
        if i < len(card_objects):
            card = card_objects[i]
            if 'name' not in card or card['name'] in exclude_cards:
                continue

            # Check color identity
            card_identity = card.get('color_identity', [])
            if not all(c in commander_identity for c in card_identity):
                continue

            # Check legality
            if card.get('legalities', {}).get('commander') not in ['legal', 'restricted'] or card['name'] in COMMANDER_BANNED_CARDS:
                continue

            results.append(card['name'])
            if len(results) >= n:
                break

    return results

def get_commander_themes(commander_name):
    """Identify potential themes for the commander"""
    if not commander_name in card_dict:
        return []

    commander_info = card_dict[commander_name]
    oracle_text = commander_info.get('oracle_text', '')

    themes = []
    if 'draw' in oracle_text.lower() or 'card' in oracle_text.lower():
        themes.append("card draw")
    if 'damage' in oracle_text.lower():
        themes.append("damage")
    if 'counter' in oracle_text.lower():
        themes.append("counters")
    if 'token' in oracle_text.lower():
        themes.append("tokens")
    if 'graveyard' in oracle_text.lower() or 'cemetery' in oracle_text.lower():
        themes.append("graveyard")
    if 'sacrifice' in oracle_text.lower():
        themes.append("sacrifice")
    if 'discard' in oracle_text.lower():
        themes.append("discard")

    return themes

def generate_synergy_queries(commander_name):
    """Generate search queries based on commander themes"""
    themes = get_commander_themes(commander_name)
    synergy_queries = []

    # Add commander name for direct synergy
    synergy_queries.append(f"synergy with {commander_name}")

    # Add theme-based queries
    for theme in themes:
        synergy_queries.append(f"cards that work with {theme}")

    # Add general synergy queries based on commander text
    if commander_name in card_dict:
        commander_text = card_dict[commander_name].get('oracle_text', '')
        key_terms = re.findall(r'\b\w+\b', commander_text.lower())

        # Filter out common words
        stop_words = {'a', 'an', 'the', 'in', 'on', 'at', 'to', 'for', 'and', 'or', 'of', 'with', 'by'}
        key_terms = [term for term in key_terms if term not in stop_words and len(term) > 3]

        # Add key terms as synergy queries
        for term in key_terms[:3]:  # Use top 3 terms
            synergy_queries.append(f"cards with {term}")

    return synergy_queries

def generate_deck(commander_name, tokenizer, model):
    """Generate initial deck using language model"""
    prompt = f"Generate a synergistic Commander decklist for {commander_name}. Include card names and quantities, with 99 cards plus the commander."
    inputs = tokenizer(prompt, return_tensors='pt')
    outputs = model.generate(
        **inputs,
        max_length=2000,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def extract_deck(deck_text):
    """Extract card names and quantities from generated text"""
    deck = {}

    # Pattern for "NxCardName" format
    pattern1 = r"(\d+)x ([^\n,]+)"
    # Pattern for "N CardName" format
    pattern2 = r"(\d+) ([^\n,]+)"
    # Pattern for lines with card names only
    pattern3 = r"^([^0-9\n][^\n]+)$"

    # Find all matches for each pattern
    matches1 = re.findall(pattern1, deck_text)
    matches2 = re.findall(pattern2, deck_text)

    # Process matches from patterns with quantities
    for qty_str, card in matches1 + matches2:
        try:
            qty = int(qty_str)
            card_name = card.strip()
            if card_name and 0 < qty <= 99:  # Sanity check
                deck[card_name] = qty
        except ValueError:
            continue

    # Only use pattern3 if we didn't get enough cards
    if len(deck) < 20:
        lines = deck_text.split('\n')
        for line in lines:
            line = line.strip()
            if line and not any(c.isdigit() for c in line[:2]):  # Avoid lines starting with numbers
                deck[line] = 1

    return deck

# Land quality settings
LAND_QUALITY_SETTINGS = {
    'competitive': {
        'name': 'Competitive Lands',
        'description': 'Prioritize lands that enter untapped, even at higher financial cost (shocklands, fetchlands, etc.)',
        'allowed_tapped': False,
        'budget': 'high'
    },
    'balanced': {
        'name': 'Balanced Lands',
        'description': 'Mix of tapped and untapped lands with good utility',
        'allowed_tapped': True,
        'budget': 'medium'
    },
    'budget': {
        'name': 'Budget Lands',
        'description': 'Primarily basic lands with some budget tapped lands for utility',
        'allowed_tapped': True,
        'budget': 'low'
    }
}

# Lists to identify enter-tapped lands
ENTERS_TAPPED_LANDS = [
    # Common tap lands
    "Akoum Refuge", "Bloodfell Caves", "Blossoming Sands", "Bojuka Bog", "Cinder Glade",
    "Coastal Tower", "Dismal Backwater", "Evolving Wilds", "Exotic Orchard", "Frontier Bivouac",
    "Frostboil Snarl", "Furycalm Snarl", "Glacial Fortress", "Highland Lake", "Irrigated Farmland",
    "Jungle Hollow", "Meandering River", "Memorial to Genius", "Mystic Monastery", "Nomad Outpost",
    "Path of Ancestry", "Prairie Stream", "Rugged Highlands", "Sandsteppe Citadel", "Scoured Barrens",
    "Sejiri Refuge", "Stone Quarry", "Sunpetal Grove", "Swiftwater Cliffs", "Temple of Abandon",
    "Temple of Deceit", "Temple of Enlightenment", "Temple of Epiphany", "Temple of Malady",
    "Temple of Malice", "Temple of Mystery", "Temple of Plenty", "Temple of Silence", "Temple of Triumph",
    "Terramorphic Expanse", "Thornwood Falls", "Tranquil Cove", "Wind-Scarred Crag", "Woodland Stream",

    # Cycling lands
    "Barren Moor", "Desert of the Fervent", "Desert of the Glorified", "Desert of the Indomitable",
    "Desert of the Mindful", "Desert of the True", "Drifting Meadow", "Forgotten Cave", "Lonely Sandbar",
    "Remote Isle", "Secluded Steppe", "Slippery Karst", "Smoldering Crater", "Tranquil Thicket",

    # Common tap tri-lands
    "Arcane Sanctum", "Crumbling Necropolis", "Frontier Bivouac", "Jungle Shrine", "Mystic Monastery",
    "Nomad Outpost", "Opulent Palace", "Sandsteppe Citadel", "Savage Lands", "Seaside Citadel",

    # Bounce lands
    "Azorius Chancery", "Boros Garrison", "Dimir Aqueduct", "Golgari Rot Farm", "Gruul Turf",
    "Izzet Boilerworks", "Orzhov Basilica", "Rakdos Carnarium", "Selesnya Sanctuary", "Simic Growth Chamber",

    # Gain lands
    "Akoum Refuge", "Bloodfell Caves", "Blossoming Sands", "Dismal Backwater", "Graypelt Refuge",
    "Jungle Hollow", "Jwar Isle Refuge", "Kazandu Refuge", "Rugged Highlands", "Scoured Barrens",
    "Sejiri Refuge", "Swiftwater Cliffs", "Thornwood Falls", "Tranquil Cove", "Wind-Scarred Crag"
]

# List of premium lands that typically enter untapped
PREMIUM_UNTAPPED_LANDS = [
    # Shock lands
    "Breeding Pool", "Blood Crypt", "Godless Shrine", "Hallowed Fountain", "Overgrown Tomb",
    "Sacred Foundry", "Steam Vents", "Stomping Ground", "Temple Garden", "Watery Grave",

    # Fetch lands
    "Arid Mesa", "Bloodstained Mire", "Flooded Strand", "Marsh Flats", "Misty Rainforest",
    "Polluted Delta", "Scalding Tarn", "Verdant Catacombs", "Windswept Heath", "Wooded Foothills",

    # Check lands
    "Clifftop Retreat", "Dragonskull Summit", "Drowned Catacomb", "Glacial Fortress", "Hinterland Harbor",
    "Isolated Chapel", "Rootbound Crag", "Sulfur Falls", "Sunpetal Grove", "Woodland Cemetery",

    # Fast lands
    "Blackcleave Cliffs", "Botanical Sanctum", "Concealed Courtyard", "Copperline Gorge", "Darkslick Shores",
    "Inspiring Vantage", "Razorverge Thicket", "Seachrome Coast", "Spirebluff Canal", "Blooming Marsh",

    # Filter lands
    "Cascade Bluffs", "Fetid Heath", "Fire-Lit Thicket", "Flooded Grove", "Graven Cairns",
    "Mystic Gate", "Rugged Prairie", "Sunken Ruins", "Twilight Mire", "Wooded Bastion",

    # Pain lands
    "Adarkar Wastes", "Battlefield Forge", "Brushland", "Caves of Koilos", "Karplusan Forest",
    "Llanowar Wastes", "Shivan Reef", "Sulfurous Springs", "Underground River", "Yavimaya Coast",

    # Original dual lands
    "Badlands", "Bayou", "Plateau", "Savannah", "Scrubland",
    "Taiga", "Tropical Island", "Tundra", "Underground Sea", "Volcanic Island",

    # Other premium lands
    "City of Brass", "Mana Confluence", "Reflecting Pool", "Cavern of Souls", "Ancient Tomb"
]

def filter_lands_by_quality(lands, land_quality):
    """Filter lands based on the selected land quality setting"""
    setting = LAND_QUALITY_SETTINGS.get(land_quality, LAND_QUALITY_SETTINGS['balanced'])

    filtered_lands = []
    for land in lands:
        # Always include basic lands
        if any(land == basic for basic in BASIC_LANDS.values()):
            filtered_lands.append(land)
            continue

        # Check if land enters tapped
        enters_tapped = land in ENTERS_TAPPED_LANDS

        # For competitive setting, exclude tapped lands
        if not setting['allowed_tapped'] and enters_tapped:
            continue

        # For budget setting, exclude expensive premium lands
        if setting['budget'] == 'low' and land in PREMIUM_UNTAPPED_LANDS:
            continue

        # Land passed filters
        filtered_lands.append(land)

    return filtered_lands

def search_lands_by_quality(commander_identity, land_quality, exclude_cards=None, count=10):
    """Search for lands that match the commander's color identity and land quality setting"""
    setting = LAND_QUALITY_SETTINGS.get(land_quality, LAND_QUALITY_SETTINGS['balanced'])

    # Base land query for the color identity
    land_query = "type:land"

    # Refine search based on land quality
    if setting['budget'] == 'high':
        # Prioritize premium untapped lands
        land_results = [land for land in PREMIUM_UNTAPPED_LANDS
                        if land in card_dict and
                        all(c in commander_identity for c in card_dict[land].get('color_identity', []))]

        # If we need more lands, search for other untapped options
        if len(land_results) < count:
            additional_lands = search_cards_by_criteria(
                f"{land_query} -text:\"enters the battlefield tapped\"",
                commander_identity,
                exclude_cards=(exclude_cards or []) + land_results,
                n=count-len(land_results)
            )
            land_results.extend(additional_lands)

    elif setting['budget'] == 'low':
        # Start with some tap lands for budget setting
        tap_lands = [land for land in ENTERS_TAPPED_LANDS
                    if land in card_dict and
                    all(c in commander_identity for c in card_dict[land].get('color_identity', []))]

        # Get a subset of tap lands
        land_results = tap_lands[:min(count // 2, len(tap_lands))]

        # Add budget untapped lands if needed
        if len(land_results) < count:
            additional_lands = search_cards_by_criteria(
                f"{land_query} -text:\"enters the battlefield tapped\"",
                commander_identity,
                exclude_cards=(exclude_cards or []) + PREMIUM_UNTAPPED_LANDS + land_results,
                n=count-len(land_results)
            )
            land_results.extend(additional_lands)

    else:  # balanced approach
        # Mix of tapped and untapped lands
        tap_lands = [land for land in ENTERS_TAPPED_LANDS
                    if land in card_dict and
                    all(c in commander_identity for c in card_dict[land].get('color_identity', []))]

        # Take some tap lands
        land_results = tap_lands[:min(count // 3, len(tap_lands))]

        # Add some premium lands
        premium_lands = [land for land in PREMIUM_UNTAPPED_LANDS
                        if land in card_dict and
                        all(c in commander_identity for c in card_dict[land].get('color_identity', []))]

        land_results.extend(premium_lands[:min(count // 3, len(premium_lands))])

        # Fill remaining slots with other lands
        if len(land_results) < count:
            additional_lands = search_cards_by_criteria(
                land_query,
                commander_identity,
                exclude_cards=(exclude_cards or []) + land_results,
                n=count-len(land_results)
            )
            land_results.extend(additional_lands)

    # Filter results to ensure they match commander identity
    filtered_results = [land for land in land_results
                       if land in card_dict and
                       all(c in commander_identity for c in card_dict[land].get('color_identity', []))]

    return filtered_results[:count]

def complete_deck(partial_deck, commander_name, commander_identity, win_condition=None, land_quality='balanced'):
    """Complete deck with appropriate cards to reach 100 cards total"""
    logging.info(f"Completing deck with {len(partial_deck)} initial cards")

    # Create a copy of the partial deck
    completed_deck = dict(partial_deck)

    # Count cards by type
    card_count = sum(completed_deck.values())

    # Generate synergy search queries
    synergy_queries = generate_synergy_queries(commander_name)
    CARD_CATEGORIES['synergy'] = synergy_queries

    # If win condition is provided, add win condition-specific queries
    if win_condition:
        win_condition_name = win_condition.get('name', '')

        # Create targeted queries based on win condition
        win_queries = []

        # Basic win condition categorization
        if 'combo' in win_condition_name.lower():
            win_queries.extend([
                "infinite combo pieces",
                "combo enablers",
                "tutor effects"
            ])
        elif any(x in win_condition_name.lower() for x in ['damage', 'burn']):
            win_queries.extend([
                "direct damage spells",
                "damage doubler",
                "burn spells"
            ])
        elif 'token' in win_condition_name.lower():
            win_queries.extend([
                "token generators",
                "token doublers",
                "anthem effects"
            ])
        elif any(x in win_condition_name.lower() for x in ['mill', 'deck out']):
            win_queries.extend([
                "mill effects",
                "exile library cards"
            ])
        elif 'storm' in win_condition_name.lower():
            win_queries.extend([
                "storm cards",
                "cast multiple spells",
                "copy spells"
            ])

        # Add win condition name itself as a query
        win_queries.append(f"cards for {win_condition_name} strategy")

        # Add key cards from win condition as semantic anchors
        for key_card in win_condition.get('key_cards', []):
            if key_card in card_dict:
                win_queries.append(f"cards that work well with {key_card}")

        # Add these to our synergy queries
        CARD_CATEGORIES['win_condition'] = win_queries

    # Calculate how many cards to add
    cards_needed = 99 - card_count  # 99 cards plus commander = 100
    if cards_needed <= 0:
        return completed_deck

    logging.info(f"Need to add {cards_needed} more cards to reach 99")

    # Track all cards we're excluding
    exclude_list = list(completed_deck.keys())

    # Count card types in current deck
    type_counts = count_card_types(completed_deck)
    logging.info(f"Current type distribution: {type_counts}")

    # Calculate mana curve to analyze deck balance
    mana_curve = calculate_mana_curve(completed_deck)
    logging.info(f"Current mana curve: {mana_curve}")

    # Add necessary lands if missing (most decks need them)
    added_lands = 0
    if type_counts.get('Land', 0) < 36:  # Aim for 36 lands in total
        lands_needed = min(36 - type_counts.get('Land', 0), cards_needed)
        logging.info(f"Adding {lands_needed} lands with quality setting: {land_quality}")

        # First, try to add non-basic lands based on land quality
        non_basic_lands_to_add = min(lands_needed // 2, 10)  # Add up to 10 non-basic lands
        if non_basic_lands_to_add > 0:
            non_basic_lands = search_lands_by_quality(
                commander_identity,
                land_quality,
                exclude_cards=exclude_list,
                count=non_basic_lands_to_add
            )

            for land in non_basic_lands:
                completed_deck[land] = 1
                exclude_list.append(land)
                added_lands += 1

        # Calculate remaining lands needed
        remaining_lands = lands_needed - added_lands

        # Add basic lands for the remaining slots
        if remaining_lands > 0:
            # Determine which basic lands to add based on color identity
            land_types = []
            for color in commander_identity:
                if color in BASIC_LANDS:
                    land_types.append(BASIC_LANDS[color])

            # Add equal numbers of each basic land type
            if land_types:
                per_land = remaining_lands // len(land_types)
                remainder = remaining_lands % len(land_types)

                for i, land in enumerate(land_types):
                    qty = per_land + (1 if i < remainder else 0)
                    if land in completed_deck:
                        completed_deck[land] += qty
                    else:
                        completed_deck[land] = qty
                    added_lands += qty

    # Recalculate cards needed after adding lands
    cards_needed -= added_lands

    # If we still need cards, add by category
    if cards_needed > 0:
        # Define target card type distribution based on win condition
        if win_condition and 'name' in win_condition:
            win_condition_name = win_condition['name'].lower()

            # Adjust distribution based on win condition type
            if 'combo' in win_condition_name:
                target_distribution = {
                    'ramp': 0.15,      # More ramp for combo decks
                    'draw': 0.15,      # More card draw to find combo pieces
                    'removal': 0.10,   # Standard removal
                    'wrath': 0.05,     # Standard board wipes
                    'win_condition': 0.40, # Heavy focus on win condition
                    'synergy': 0.15    # Some general synergy
                }
            elif any(x in win_condition_name for x in ['aggro', 'damage', 'combat']):
                target_distribution = {
                    'ramp': 0.10,      # Standard ramp
                    'draw': 0.10,      # Standard draw
                    'removal': 0.15,   # More removal for combat-focused decks
                    'wrath': 0.05,     # Standard board wipes
                    'win_condition': 0.40, # Heavy focus on win condition
                    'synergy': 0.20    # More general synergy
                }
            elif 'control' in win_condition_name:
                target_distribution = {
                    'ramp': 0.10,      # Standard ramp
                    'draw': 0.15,      # More card draw for control
                    'removal': 0.20,   # More removal for control
                    'wrath': 0.10,     # More board wipes for control
                    'win_condition': 0.30, # Focus on win condition
                    'synergy': 0.15    # Some general synergy
                }
            else:
                # Default balanced distribution
                target_distribution = {
                    'ramp': 0.10,      # 10% ramp spells
                    'draw': 0.10,      # 10% card draw
                    'removal': 0.10,   # 10% removal
                    'wrath': 0.05,     # 5% board wipes
                    'win_condition': 0.35, # 35% win condition focus
                    'synergy': 0.30    # 30% general synergy
                }
        else:
            # Standard distribution without win condition
            target_distribution = {
                'ramp': 0.10,      # 10% ramp spells
                'draw': 0.10,      # 10% card draw
                'removal': 0.10,   # 10% removal
                'wrath': 0.05,     # 5% board wipes
                'synergy': 0.65    # 65% synergy cards
            }

        # Add cards by category
        for category, percentage in target_distribution.items():
            if category not in CARD_CATEGORIES:
                continue

            category_count = int(cards_needed * percentage)
            if category_count > 0:
                logging.info(f"Adding {category_count} {category} cards")
                added = 0

                # Use different queries for the category
                queries = CARD_CATEGORIES[category]
                if not queries:
                    continue

                for query in queries:
                    if added >= category_count:
                        break

                    # Search for matching cards
                    results = search_cards_by_criteria(
                        query,
                        commander_identity,
                        exclude_cards=exclude_list
                    )

                    # Add a portion of results
                    to_add = min(len(results), category_count - added)
                    for i in range(to_add):
                        card_name = results[i]
                        completed_deck[card_name] = 1
                        exclude_list.append(card_name)
                        added += 1

    # Analyze and balance mana curve if needed
    final_curve = calculate_mana_curve(completed_deck)
    curve_issues = analyze_mana_curve(final_curve)

    if curve_issues:
        logging.info(f"Mana curve issues detected: {curve_issues}")
        fix_mana_curve(completed_deck, final_curve, commander_identity, exclude_list)

    # Check final count
    total_cards = sum(completed_deck.values())
    if total_cards < 99:
        # Still need more cards - add random cards that match color identity
        logging.info(f"Still need {99 - total_cards} more cards")

        # Get cards that match color identity
        valid_cards = []
        for card_name, info in card_dict.items():
            if (card_name not in exclude_list and
                all(c in commander_identity for c in info.get('color_identity', [])) and
                card_name not in COMMANDER_BANNED_CARDS and
                not any(land == card_name for land in BASIC_LANDS.values())):
                valid_cards.append(card_name)

        # Randomly select remaining cards
        random.shuffle(valid_cards)
        for card_name in valid_cards:
            if sum(completed_deck.values()) >= 99:
                break
            completed_deck[card_name] = 1

    # If we have too many cards, trim some
    while sum(completed_deck.values()) > 99:
        # Find a card with qty > 1 to reduce
        for card, qty in list(completed_deck.items()):
            if qty > 1 and not any(land == card for land in BASIC_LANDS.values()):
                completed_deck[card] -= 1
                break
        else:
            # If all cards have qty=1, remove a random card
            non_essential = [card for card in completed_deck.keys()
                           if card != commander_name
                           and not any(land == card for land in BASIC_LANDS.values())
                           and (not win_condition or card not in win_condition.get('key_cards', []))]
            if non_essential:
                card_to_remove = random.choice(non_essential)
                del completed_deck[card_to_remove]

    return completed_deck

def calculate_mana_curve(deck):
    """Calculate the mana curve of the deck"""
    curve = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, '7+': 0}

    for card, qty in deck.items():
        card_info = card_dict.get(card)
        if not card_info:
            continue

        # Skip lands
        if 'Land' in card_info.get('types', []):
            continue

        mana_value = card_info.get('mana_value', 0)

        # Categorize by mana value
        if mana_value >= 7:
            curve['7+'] += qty
        else:
            curve[int(mana_value)] += qty

    return curve

def analyze_commander_for_strategies(commander_name, card_dict):
    """Generate custom win conditions based on commander attributes"""
    commander_info = card_dict.get(commander_name, {})
    oracle_text = commander_info.get('oracle_text', '')
    color_identity = commander_info.get('color_identity', [])
    mana_value = commander_info.get('mana_value', 0)

    # Extract key mechanics from text
    strategies = []
    if 'spirit' in oracle_text.lower() or 'token' in oracle_text.lower():
        strategies.append('Token/Spirit Tribal')
    if 'cast' in oracle_text.lower() and 'noncreature' in oracle_text.lower():
        strategies.append('Spellslinger')
    # Additional strategy identifications...

    return strategies

def analyze_mana_curve(curve):
    """Analyze the mana curve for potential issues"""
    issues = []

    # Calculate total spells
    total_spells = sum(curve.values())

    if total_spells < 10:
        return []  # Not enough spells to analyze

    # Check for imbalances in the curve
    if curve[1] + curve[2] < total_spells * 0.2:
        issues.append("low_early_drops")

    if curve['7+'] > total_spells * 0.15:
        issues.append("top_heavy")

    # Check for gaps in the curve
    for i in range(2, 5):
        if curve[i] == 0:
            issues.append(f"gap_at_{i}")

    return issues

def fix_mana_curve(deck, curve, commander_identity, exclude_list):
    """Attempt to fix issues with the mana curve"""
    issues = analyze_mana_curve(curve)

    if not issues:
        return deck

    # Fix common issues
    if "low_early_drops" in issues:
        # Add more low-cost cards
        logging.info("Fixing mana curve: Adding more low-drop cards")

        # Replace some high-cost cards with low-cost ones
        high_cost_cards = []
        for card, qty in deck.items():
            card_info = card_dict.get(card)
            if not card_info:
                continue

            if 'Land' in card_info.get('types', []):
                continue

            mana_value = card_info.get('mana_value', 0)
            if mana_value >= 5:
                high_cost_cards.extend([card] * qty)

        # Shuffle to randomize selections
        random.shuffle(high_cost_cards)

        # Replace up to 5 high-cost cards
        replacements = min(5, len(high_cost_cards))

        for i in range(replacements):
            if i < len(high_cost_cards):
                card_to_replace = high_cost_cards[i]

                # Find low-cost replacements
                low_cost_cards = search_cards_by_criteria(
                    "mana value 1 or mana value 2",
                    commander_identity,
                    exclude_cards=exclude_list
                )

                if low_cost_cards:
                    # Remove high-cost card
                    if deck[card_to_replace] > 1:
                        deck[card_to_replace] -= 1
                    else:
                        del deck[card_to_replace]

                    # Add low-cost card
                    replacement = low_cost_cards[0]
                    if replacement in deck:
                        deck[replacement] += 1
                    else:
                        deck[replacement] = 1
                    exclude_list.append(replacement)

    if "top_heavy" in issues:
        # Replace some high-cost cards with mid-range ones
        logging.info("Fixing mana curve: Reducing top-heavy cards")

        high_cost_cards = []
        for card, qty in deck.items():
            card_info = card_dict.get(card)
            if not card_info:
                continue

            if 'Land' in card_info.get('types', []):
                continue

            mana_value = card_info.get('mana_value', 0)
            if mana_value >= 7:
                high_cost_cards.extend([card] * qty)

        # Replace up to half of the high-cost cards
        replacements = min(len(high_cost_cards) // 2 + 1, len(high_cost_cards))

        for i in range(replacements):
            if i < len(high_cost_cards):
                card_to_replace = high_cost_cards[i]

                # Find mid-range replacements
                mid_cost_cards = search_cards_by_criteria(
                    "mana value 3 or mana value 4 or mana value 5",
                    commander_identity,
                    exclude_cards=exclude_list
                )

                if mid_cost_cards:
                    # Remove high-cost card
                    if deck[card_to_replace] > 1:
                        deck[card_to_replace] -= 1
                    else:
                        del deck[card_to_replace]

                    # Add mid-cost card
                    replacement = mid_cost_cards[0]
                    if replacement in deck:
                        deck[replacement] += 1
                    else:
                        deck[replacement] = 1
                    exclude_list.append(replacement)

    # Fix gaps in the curve
    for issue in issues:
        if issue.startswith("gap_at_"):
            mana_value = int(issue.split("_")[-1])
            logging.info(f"Fixing mana curve: Adding cards at mana value {mana_value}")

            # Find cards with the missing mana value
            gap_cards = search_cards_by_criteria(
                f"mana value {mana_value}",
                commander_identity,
                exclude_cards=exclude_list
            )

            if gap_cards:
                # Find a card to replace
                replaced = False
                for card, qty in list(deck.items()):
                    card_info = card_dict.get(card)
                    if not card_info:
                        continue

                    if 'Land' in card_info.get('types', []):
                        continue

                    existing_mv = card_info.get('mana_value', 0)
                    if (existing_mv >= 6 or existing_mv <= 1) and card not in exclude_list:
                        # Replace this card
                        if qty > 1:
                            deck[card] -= 1
                        else:
                            del deck[card]

                        # Add gap-filling card
                        replacement = gap_cards[0]
                        if replacement in deck:
                            deck[replacement] += 1
                        else:
                            deck[replacement] = 1
                        exclude_list.append(replacement)
                        replaced = True
                        break

                # If we couldn't find a card to replace, just add the gap card
                if not replaced and sum(deck.values()) < 99:
                    replacement = gap_cards[0]
                    if replacement in deck:
                        deck[replacement] += 1
                    else:
                        deck[replacement] = 1
                    exclude_list.append(replacement)

    return deck

def count_card_types(deck):
    """Count cards by type in the deck"""
    type_counts = {'Land': 0, 'Creature': 0, 'Artifact': 0,
                  'Enchantment': 0, 'Planeswalker': 0,
                  'Instant': 0, 'Sorcery': 0, 'Other': 0}

    for card, qty in deck.items():
        if card in BASIC_LANDS.values():
            type_counts['Land'] += qty
            continue

        card_info = card_dict.get(card)
        if not card_info:
            type_counts['Other'] += qty
            continue

        types = card_info.get('types', [])

        # Count by primary type (use first match in hierarchy)
        if 'Land' in types:
            type_counts['Land'] += qty
        elif 'Creature' in types:
            type_counts['Creature'] += qty
        elif 'Artifact' in types:
            type_counts['Artifact'] += qty
        elif 'Enchantment' in types:
            type_counts['Enchantment'] += qty
        elif 'Planeswalker' in types:
            type_counts['Planeswalker'] += qty
        elif 'Instant' in types:
            type_counts['Instant'] += qty
        elif 'Sorcery' in types:
            type_counts['Sorcery'] += qty
        else:
            type_counts['Other'] += qty

    return type_counts

def determine_win_condition(commander, tokenizer, model, commander_identity):
    """Use the language model to determine appropriate win conditions for the commander"""
    prompt = f"""Analyze {commander} for Commander format and suggest the top 3 win conditions that synergize with this commander.
For each win condition:
1. Name the strategy (e.g., "Storm", "Combo", "Aggro", etc.)
2. Describe how it works with {commander}
3. List 5-10 key cards that enable this win condition within the {', '.join(commander_identity) if commander_identity else 'colorless'} color identity
4. Rate its power level from 1-10

Format your response with clear section headers for each win condition and bullet points for key cards.
"""

    inputs = tokenizer(prompt, return_tensors='pt')
    outputs = model.generate(
        **inputs,
        max_length=1500,
        temperature=0.8,  # Slightly higher temperature for more diverse responses
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    win_condition_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Define some pre-set win conditions if we can't parse the model output
    preset_win_conditions = [
        {
            'name': 'Storm Combo',
            'description': 'Cast multiple cheap spells in a turn to generate mana with Birgi, then finish with a big payoff spell.',
            'key_cards': ['Grapeshot', 'Mana Geyser', 'Seething Song', 'Jeska\'s Will', 'Runaway Steam-Kin', 'Grinning Ignus'],
            'power_level': 8
        },
        {
            'name': 'Wheel Effects',
            'description': 'Use Harnfel\'s "discard to draw" ability with wheel effects to cycle through your deck quickly.',
            'key_cards': ['Wheel of Fortune', 'Reforge the Soul', 'Magus of the Wheel', 'Past in Flames', 'Underworld Breach'],
            'power_level': 7
        },
        {
            'name': 'Artifact Combo',
            'description': 'Use cost reducers and Birgi\'s mana generation to chain artifact casts together.',
            'key_cards': ['Grinning Ignus', 'Sensei\'s Divining Top', 'Aetherflux Reservoir', 'Helm of Awakening', 'Skullclamp'],
            'power_level': 7
        },
        {
            'name': 'Red Aggro',
            'description': 'Cast numerous cheap red creatures and use Birgi\'s mana to overwhelm opponents.',
            'key_cards': ['Runaway Steam-Kin', 'Monastery Swiftspear', 'Lightning Bolt', 'Light Up the Stage', 'Reckless Impulse'],
            'power_level': 6
        }
    ]

    # Attempt to parse the model's output
    try:
        # Parse the win condition text to extract structured data
        win_conditions = []
        sections = re.split(r'(?:\n\n|\n#+\s*|\n\d+\.?\s*)(Win Condition|Strategy|Approach)[:\s-]+', win_condition_text)

        # If we have sections after splitting
        if len(sections) > 1:
            for i in range(1, len(sections), 2):
                if i+1 < len(sections):
                    section_header = sections[i].strip()
                    section_content = sections[i+1].strip()

                    # Extract strategy name from header or first line
                    strategy_name = section_header
                    if not strategy_name:
                        first_line = section_content.split('\n')[0].strip()
                        strategy_name = first_line

                    # Extract key cards - look for bullet points, numbers, or capital letters at line start
                    key_cards = []
                    description = ""
                    power_level = 0

                    for line in section_content.split('\n'):
                        line = line.strip()

                        # Look for power level rating
                        if re.search(r'(?:power|rating|score).*?(\d+)\s*(/|out of)\s*10', line.lower()):
                            match = re.search(r'(?:power|rating|score).*?(\d+)\s*(/|out of)\s*10', line.lower())
                            if match:
                                power_level = int(match.group(1))

                        # If line starts with a bullet, number, or capital letter, it might be a card
                        elif line.startswith('-') or line.startswith('*') or re.match(r'^\d+\.', line) or re.match(r'^[A-Z]', line):
                            # Clean up formatting
                            card = re.sub(r'^[-*•\d.)\]]+\s*', '', line)

                            # Remove explanations after the card name
                            if ' - ' in card:
                                card = card.split(' - ')[0].strip()
                            elif ' – ' in card:
                                card = card.split(' – ')[0].strip()
                            elif ' — ' in card:
                                card = card.split(' — ')[0].strip()
                            elif ': ' in card:
                                card = card.split(': ')[0].strip()

                            # Remove brackets if present
                            card = re.sub(r'[\[\]]', '', card)

                            # Validate card exists in database
                            if card in card_dict:
                                key_cards.append(card)
                        else:
                            # If not a card or power level, it's part of the description
                            description += line + " "

                    # Only add if we found some key cards
                    if key_cards:
                        win_conditions.append({
                            'name': strategy_name,
                            'description': description.strip(),
                            'key_cards': key_cards,
                            'power_level': power_level
                        })

        # If we failed to extract win conditions, use preset ones
        if not win_conditions:
            # Filter preset conditions to ensure cards exist in commander's color identity
            valid_presets = []
            for condition in preset_win_conditions:
                valid_cards = [card for card in condition['key_cards']
                              if card in card_dict and
                              all(c in commander_identity for c in card_dict[card].get('color_identity', []))]

                if valid_cards:
                    valid_condition = condition.copy()
                    valid_condition['key_cards'] = valid_cards
                    valid_presets.append(valid_condition)

            win_conditions = valid_presets

        # If we still don't have any, create a generic one
        if not win_conditions:
            logging.warning("Using default generic win condition")
            win_conditions = [{
                'name': 'Generic Synergy',
                'description': f'A balanced approach utilizing {commander}\'s abilities.',
                'key_cards': [],
                'power_level': 5
            }]

        return win_conditions, win_condition_text

    except Exception as e:
        logging.warning(f"Error parsing win conditions: {str(e)}")

        # Fall back to preset win conditions
        valid_presets = []
        for condition in preset_win_conditions:
            valid_cards = [card for card in condition['key_cards']
                          if card in card_dict and
                          all(c in commander_identity for c in card_dict[card].get('color_identity', []))]

            if valid_cards:
                valid_condition = condition.copy()
                valid_condition['key_cards'] = valid_cards
                valid_presets.append(valid_condition)

        if valid_presets:
            return valid_presets, win_condition_text
        else:
            return [{
                'name': 'Generic Synergy',
                'description': f'A balanced approach utilizing {commander}\'s abilities.',
                'key_cards': [],
                'power_level': 5
            }], win_condition_text

def generate_random_deck(commander, commander_identity):
    """Generate a random but coherent deck for the given commander"""
    logging.info(f"Generating random deck for {commander}")

    # Start with an empty deck
    deck = {}

    # Add commander if not already in deck
    if commander not in deck:
        deck[commander] = 1

    # Define card categories and their target percentages
    categories = {
        'Ramp': 0.10,     # 10% ramp effects
        'Draw': 0.10,     # 10% card draw
        'Removal': 0.10,  # 10% removal
        'Wrath': 0.05,    # 5% board wipes
        'Creatures': 0.25, # 25% creatures
        'Synergy': 0.15,  # 15% synergy pieces
        'Lands': 0.38     # 38% lands (including basic lands)
    }

    # Calculate target counts
    total_cards = 99  # 99 cards plus commander = 100
    card_counts = {category: int(percentage * total_cards) for category, percentage in categories.items()}

    # Adjust to ensure we hit exactly 99 cards
    remaining = total_cards - sum(card_counts.values())
    if remaining > 0:
        card_counts['Creatures'] += remaining

    # Add cards by category
    for category, count in card_counts.items():
        if category == 'Lands':
            # Handle lands separately to ensure proper basic land distribution
            continue

        logging.info(f"Adding {count} {category} cards")

        # Generate query based on category
        if category == 'Ramp':
            queries = ['type:artifact text:add mana', 'text:"add mana"', 'text:ritual']
        elif category == 'Draw':
            queries = ['text:"draw a card"', 'text:"draw cards"']
        elif category == 'Removal':
            queries = ['text:destroy target', 'text:exile target', 'text:damage to target']
        elif category == 'Wrath':
            queries = ['text:"destroy all"', 'text:"exile all"', 'text:"damage to all"']
        elif category == 'Creatures':
            queries = ['type:creature power>2', 'type:creature toughness>2', 'type:creature text:when']
        elif category == 'Synergy':
            # Generate synergy queries based on commander text
            queries = generate_synergy_queries(commander)
            if not queries:
                queries = ['type:instant', 'type:sorcery']

        # Add cards for each query
        added = 0
        exclude_list = list(deck.keys())

        for query in queries:
            if added >= count:
                break

            results = search_cards_by_criteria(query, commander_identity, exclude_cards=exclude_list)
            random.shuffle(results)  # Randomize the results

            # Add a portion of results
            to_add = min(len(results), count - added)
            for i in range(to_add):
                if i < len(results):
                    card_name = results[i]
                    deck[card_name] = 1
                    exclude_list.append(card_name)
                    added += 1

    # Add lands to reach target
    lands_to_add = card_counts['Lands']

    # First add some non-basic lands if available
    nonbasic_query = 'type:land -type:basic'
    nonbasic_lands = search_cards_by_criteria(nonbasic_query, commander_identity, exclude_cards=list(deck.keys()), n=10)

    for land in nonbasic_lands:
        if lands_to_add <= 0:
            break
        deck[land] = 1
        lands_to_add -= 1

    # Fill the rest with basic lands
    if lands_to_add > 0:
        # Determine which basic lands to add based on color identity
        land_types = []
        for color in commander_identity:
            if color in BASIC_LANDS:
                land_types.append(BASIC_LANDS[color])

        # If no colors or colorless, add Wastes
        if not land_types:
            land_types = ['Wastes']

        # Add equal numbers of each basic land type
        per_land = lands_to_add // len(land_types)
        remainder = lands_to_add % len(land_types)

        for i, land in enumerate(land_types):
            qty = per_land + (1 if i < remainder else 0)
            if land in deck:
                deck[land] += qty
            else:
                deck[land] = qty

    return deck

def main():
    cards = load_card_data()
    setup_retriever(cards)

    model_dir = '/content/drive/MyDrive/MTGModel/real_deck_training/final-model'
    tokenizer, model = load_language_model(model_dir)

    commander = input("Enter commander name: ")
    commander_identity = get_commander_identity(commander)

    if not commander_identity:
        logging.error(f"Could not determine color identity for {commander}")
        similar_commanders = search_similar_cards(commander, n=5)
        print(f"Did you mean one of these? {', '.join(similar_commanders)}")
        commander = input("Try entering commander name again: ")
        commander_identity = get_commander_identity(commander)
        if not commander_identity:
            print("Still couldn't find commander. Using default color identity (colorless).")
            commander_identity = []

    print(f"\nAnalyzing {commander} (Color Identity: {', '.join(commander_identity)})")

    # First, determine possible win conditions
    win_conditions, win_condition_text = determine_win_condition(commander, tokenizer, model, commander_identity)

    # Display win conditions to user
    print("\n=== Possible Win Conditions ===")
    print(win_condition_text)
    print("\n===========================")

    # Let user select a win condition or choose random
    if len(win_conditions) > 1:
        print("\nSelect a win condition to build around:")
        for i, wc in enumerate(win_conditions):
            key_cards_str = ", ".join(wc.get('key_cards', [])[:3])
            power_level = wc.get('power_level', 0)
            power_str = f" (Power: {power_level}/10)" if power_level > 0 else ""
            suggested_bracket = get_appropriate_bracket_for_win_condition(wc)
            bracket_str = f" - Bracket: {suggested_bracket} ({COMMANDER_BRACKETS[suggested_bracket]['name']})" if suggested_bracket > 0 else ""
            print(f"{i+1}. {wc['name']}{power_str}{bracket_str} - Key cards: {key_cards_str}...")

        print(f"{len(win_conditions)+1}. Random - Let me pick a random win condition")
        print(f"{len(win_conditions)+2}. Fully Random Deck - Generate a completely random deck")

        selection = input(f"Enter 1-{len(win_conditions)+2} [1]: ").strip()
        if not selection:
            selected_idx = 0
        else:
            try:
                selected_idx = int(selection) - 1
                if selected_idx < 0 or selected_idx > len(win_conditions) + 1:
                    selected_idx = 0
            except ValueError:
                selected_idx = 0

        # Handle random selection
        if selected_idx == len(win_conditions):
            selected_idx = random.randint(0, len(win_conditions) - 1)
            print(f"Randomly selected: {win_conditions[selected_idx]['name']}")

            # Use selected win condition
            selected_win_condition = win_conditions[selected_idx]

            # Determine appropriate bracket based on win condition
            suggested_bracket = get_appropriate_bracket_for_win_condition(selected_win_condition)

            # Ask for bracket selection
            print("\nSelect a power level bracket for the deck:")
            for i, (bracket_num, bracket_info) in enumerate(COMMANDER_BRACKETS.items()):
                if bracket_num == suggested_bracket:
                    print(f"{bracket_num}. {bracket_info['name']} (RECOMMENDED) - {bracket_info['description']}")
                else:
                    print(f"{bracket_num}. {bracket_info['name']} - {bracket_info['description']}")

            bracket_selection = input(f"Enter bracket (1-5) [{suggested_bracket}]: ").strip()
            try:
                selected_bracket = int(bracket_selection)
                if selected_bracket < 1 or selected_bracket > 5:
                    selected_bracket = suggested_bracket
            except ValueError:
                selected_bracket = suggested_bracket

            # Select land quality
            print("\nSelect a land quality for the mana base:")
            for i, (key, land_setting) in enumerate(LAND_QUALITY_SETTINGS.items()):
                print(f"{i+1}. {land_setting['name']} - {land_setting['description']}")

            land_quality_selection = input("Enter choice (1-3) [2]: ").strip()
            try:
                land_choice = int(land_quality_selection) - 1
                if land_choice < 0 or land_choice >= len(LAND_QUALITY_SETTINGS):
                    land_choice = 1  # Default to balanced
            except ValueError:
                land_choice = 1  # Default to balanced

            land_quality = list(LAND_QUALITY_SETTINGS.keys())[land_choice]
            print(f"Selected land quality: {LAND_QUALITY_SETTINGS[land_quality]['name']}")

            print(f"\nGenerating deck for {commander} with '{selected_win_condition['name']}' win condition")
            print(f"Using bracket {selected_bracket} - {COMMANDER_BRACKETS[selected_bracket]['name']}")

            # Check if we should use a temperature boost for more variation
            use_variation = input("Would you like more variety in the generated deck? (y/n) [n]: ").strip().lower()
            temp_boost = 0.2 if use_variation == 'y' else 0.0

            # Generate deck with chosen win condition
            deck_text = generate_conditioned_deck(commander, tokenizer, model, selected_win_condition, commander_identity, temp_boost)
            initial_deck = extract_deck(deck_text)

            # Add key cards from win condition if not already in deck
            for card in selected_win_condition.get('key_cards', []):
                if card not in initial_deck and card != commander:
                    card_info = card_dict.get(card)
                    if card_info and all(c in commander_identity for c in card_info.get('color_identity', [])):
                        initial_deck[card] = 1

            # Validate and filter cards
            valid_deck, invalid_cards = validate_deck_identity(initial_deck, commander_identity)

            # Complete deck to 100 cards
            final_deck = complete_deck(valid_deck, commander, commander_identity, selected_win_condition, land_quality)

            # Filter deck for bracket compatibility
            filtered_deck, removed_cards = filter_deck_for_bracket(final_deck.copy(), selected_bracket)

            # Replace removed cards
            if removed_cards:
                print(f"Removed {len(removed_cards)} cards that don't comply with bracket {selected_bracket} restrictions:")
                for card, qty in removed_cards.items():
                    print(f"  {qty}x {card}")
                filtered_deck = replace_removed_cards(filtered_deck, removed_cards, commander_identity, selected_bracket)

            # Add commander as 1x if not already in deck
            if commander not in filtered_deck:
                filtered_deck[commander] = 1

            # Print final deck
            type_counts = count_card_types(filtered_deck)
            print(f"\nFinal Deck for {commander} (Color Identity: {', '.join(commander_identity)}):")
            print(f"Strategy: {selected_win_condition['name']}")
            print(f"Bracket: {selected_bracket} - {COMMANDER_BRACKETS[selected_bracket]['name']}")
            print(f"Land Quality: {LAND_QUALITY_SETTINGS[land_quality]['name']}")
            print(f"Total cards: {sum(filtered_deck.values())}")
            print(f"Card type distribution: {type_counts}")

            print_formatted_deck(filtered_deck, commander)
            return

        # Handle fully random deck
        elif selected_idx == len(win_conditions) + 1:
            print(f"\nGenerating random deck for {commander}")

            # Select land quality before generating the random deck
            print("\nSelect a land quality for the mana base:")
            for i, (key, land_setting) in enumerate(LAND_QUALITY_SETTINGS.items()):
                print(f"{i+1}. {land_setting['name']} - {land_setting['description']}")

            land_quality_selection = input("Enter choice (1-3) [2]: ").strip()
            try:
                land_choice = int(land_quality_selection) - 1
                if land_choice < 0 or land_choice >= len(LAND_QUALITY_SETTINGS):
                    land_choice = 1  # Default to balanced
            except ValueError:
                land_choice = 1  # Default to balanced

            land_quality = list(LAND_QUALITY_SETTINGS.keys())[land_choice]
            print(f"Selected land quality: {LAND_QUALITY_SETTINGS[land_quality]['name']}")

            random_deck = generate_random_deck(commander, commander_identity, land_quality)

            # Ask for bracket selection
            print("\nSelect a power level bracket for the deck:")
            for i, (bracket_num, bracket_info) in enumerate(COMMANDER_BRACKETS.items()):
                print(f"{bracket_num}. {bracket_info['name']} - {bracket_info['description']}")

            bracket_selection = input(f"Enter bracket (1-5) [2]: ").strip()
            try:
                selected_bracket = int(bracket_selection)
                if selected_bracket < 1 or selected_bracket > 5:
                    selected_bracket = 2  # Default to Core
            except ValueError:
                selected_bracket = 2  # Default to Core

            # Filter deck for bracket compatibility
            filtered_deck, removed_cards = filter_deck_for_bracket(random_deck, selected_bracket)

            # Replace removed cards
            if removed_cards:
                print(f"Removed {len(removed_cards)} cards that don't comply with bracket {selected_bracket} restrictions.")
                filtered_deck = replace_removed_cards(filtered_deck, removed_cards, commander_identity, selected_bracket)

            # Print final deck
            type_counts = count_card_types(filtered_deck)
            print(f"\nRandom Deck for {commander} (Color Identity: {', '.join(commander_identity)}):")
            print(f"Bracket: {selected_bracket} - {COMMANDER_BRACKETS[selected_bracket]['name']}")
            print(f"Land Quality: {LAND_QUALITY_SETTINGS[land_quality]['name']}")
            print(f"Total cards: {sum(filtered_deck.values())}")
            print(f"Card type distribution: {type_counts}")

            print_formatted_deck(filtered_deck, commander)
            return
    else:
        selected_idx = 0

    # Use selected win condition
    selected_win_condition = win_conditions[selected_idx]

    # Determine appropriate bracket based on win condition
    suggested_bracket = get_appropriate_bracket_for_win_condition(selected_win_condition)

    # Ask for bracket selection
    print("\nSelect a power level bracket for the deck:")
    for i, (bracket_num, bracket_info) in enumerate(COMMANDER_BRACKETS.items()):
        if bracket_num == suggested_bracket:
            print(f"{bracket_num}. {bracket_info['name']} (RECOMMENDED) - {bracket_info['description']}")
        else:
            print(f"{bracket_num}. {bracket_info['name']} - {bracket_info['description']}")

    bracket_selection = input(f"Enter bracket (1-5) [{suggested_bracket}]: ").strip()
    try:
        selected_bracket = int(bracket_selection)
        if selected_bracket < 1 or selected_bracket > 5:
            selected_bracket = suggested_bracket
    except ValueError:
        selected_bracket = suggested_bracket

    # Select land quality
    print("\nSelect a land quality for the mana base:")
    for i, (key, land_setting) in enumerate(LAND_QUALITY_SETTINGS.items()):
        print(f"{i+1}. {land_setting['name']} - {land_setting['description']}")

    land_quality_selection = input("Enter choice (1-3) [2]: ").strip()
    try:
        land_choice = int(land_quality_selection) - 1
        if land_choice < 0 or land_choice >= len(LAND_QUALITY_SETTINGS):
            land_choice = 1  # Default to balanced
    except ValueError:
        land_choice = 1  # Default to balanced

    land_quality = list(LAND_QUALITY_SETTINGS.keys())[land_choice]
    print(f"Selected land quality: {LAND_QUALITY_SETTINGS[land_quality]['name']}")

    print(f"\nGenerating deck for {commander} with '{selected_win_condition['name']}' win condition")
    print(f"Using bracket {selected_bracket} - {COMMANDER_BRACKETS[selected_bracket]['name']}")

    # Check if we should use a temperature boost for more variation
    use_variation = input("Would you like more variety in the generated deck? (y/n) [n]: ").strip().lower()
    temp_boost = 0.2 if use_variation == 'y' else 0.0

    # Generate deck with chosen win condition
    deck_text = generate_conditioned_deck(commander, tokenizer, model, selected_win_condition, commander_identity, temp_boost)
    initial_deck = extract_deck(deck_text)

    # Add key cards from win condition if not already in deck
    for card in selected_win_condition.get('key_cards', []):
        if card not in initial_deck and card != commander:
            card_info = card_dict.get(card)
            if card_info and all(c in commander_identity for c in card_info.get('color_identity', [])):
                initial_deck[card] = 1

    # Validate and filter cards
    valid_deck, invalid_cards = validate_deck_identity(initial_deck, commander_identity)

    # Complete deck to 100 cards
    final_deck = complete_deck(valid_deck, commander, commander_identity, selected_win_condition, land_quality)

    # Filter deck for bracket compatibility
    filtered_deck, removed_cards = filter_deck_for_bracket(final_deck.copy(), selected_bracket)

    # Replace removed cards
    if removed_cards:
        print(f"Removed {len(removed_cards)} cards that don't comply with bracket {selected_bracket} restrictions:")
        for card, qty in removed_cards.items():
            print(f"  {qty}x {card}")
        filtered_deck = replace_removed_cards(filtered_deck, removed_cards, commander_identity, selected_bracket)

    # Add commander as 1x if not already in deck
    if commander not in filtered_deck:
        filtered_deck[commander] = 1

    # Print final deck
    type_counts = count_card_types(filtered_deck)
    print(f"\nFinal Deck for {commander} (Color Identity: {', '.join(commander_identity)}):")
    print(f"Strategy: {selected_win_condition['name']}")
    print(f"Bracket: {selected_bracket} - {COMMANDER_BRACKETS[selected_bracket]['name']}")
    print(f"Land Quality: {LAND_QUALITY_SETTINGS[land_quality]['name']}")
    print(f"Total cards: {sum(filtered_deck.values())}")
    print(f"Card type distribution: {type_counts}")

    print_formatted_deck(filtered_deck, commander)

def print_formatted_deck(deck, commander):
    """Print the deck in a nicely formatted way"""
    print("\n--- Commander ---")
    print(f"1x {commander}")

    print("\n--- Lands ---")
    for card, qty in sorted(deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if 'Land' in types or card in BASIC_LANDS.values():
            print(f"{qty}x {card}")

    print("\n--- Creatures ---")
    for card, qty in sorted(deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if 'Creature' in types and 'Land' not in types:
            print(f"{qty}x {card}")

    print("\n--- Artifacts ---")
    for card, qty in sorted(deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if 'Artifact' in types and 'Creature' not in types and 'Land' not in types:
            print(f"{qty}x {card}")

    print("\n--- Enchantments ---")
    for card, qty in sorted(deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if 'Enchantment' in types and 'Creature' not in types and 'Land' not in types:
            print(f"{qty}x {card}")

    print("\n--- Planeswalkers ---")
    for card, qty in sorted(deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if 'Planeswalker' in types:
            print(f"{qty}x {card}")

    print("\n--- Instants and Sorceries ---")
    for card, qty in sorted(deck.items()):
        if card == commander:
            continue
        card_info = card_dict.get(card, {})
        types = card_info.get('types', [])
        if ('Instant' in types or 'Sorcery' in types) and 'Creature' not in types and 'Land' not in types:
            print(f"{qty}x {card}")

def generate_conditioned_deck(commander, tokenizer, model, win_condition, commander_identity, temp_boost=0):
    """Generate a deck with a specific win condition in mind"""
    # Prepare a prompt that specifies the win condition
    key_cards_str = ", ".join(win_condition.get('key_cards', [])[:5])  # Use top 5 key cards for prompt
    description = win_condition.get('description', f"A {win_condition['name']} strategy")

    prompt = f"""Generate a Commander deck for {commander} focusing on the "{win_condition['name']}" win condition.
Description: {description}
Include the following key cards if within color identity: {key_cards_str}.
Make sure the deck is cohesive and all cards work toward the win condition.
Format as a list with quantities (e.g., "1x Card Name").
Include exactly 99 cards (excluding the commander)."""

    inputs = tokenizer(prompt, return_tensors='pt')
    outputs = model.generate(
        **inputs,
        max_length=2000,
        temperature=0.65 + temp_boost,  # Adjust temperature based on desired variation
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    conditioned_deck_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return conditioned_deck_text

if __name__ == '__main__':
    main()

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Enter commander name: Kykar, Wind's Fury

Analyzing Kykar, Wind's Fury (Color Identity: R, U, W)

=== Possible Win Conditions ===
Analyze Kykar, Wind's Fury for Commander format and suggest the top 3 win conditions that synergize with this commander.
For each win condition:
1. Name the strategy (e.g., "Storm", "Combo", "Aggro", etc.)
2. Describe how it works with Kykar, Wind's Fury
3. List 5-10 key cards that enable this win condition within the R, U, W color identity
4. Rate its power level from 1-10

Format your response with clear section headers for each win condition and bullet points for key cards.
1x Fortify
1x Frantic Search
1x Gandalf the Grey
1x Glorious End
1x High Tide
1x Hullbreaker Horror
1x Hylda of the Icy Crown
1x Inconsistent Denial
1x Inspiring Refrain
1x Invoke the Divine
1x Ishai, Ojutai Dragonspeaker
35x Island
1x Jace's Ingenuity
1x Jin-Gitaxias, Progress Tyrant
1x Kasmina, Enigmatic Mentor
1x Kiora, the Crashing Wave
1x Kher Keep
1x Kykar, Wind's Fury
1x Kwain, 